In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import duckdb
from pathlib import Path
import re
import seaborn as sns

#### Define directories

In [ ]:
ROOT = Path.cwd().resolve()

DATA = (ROOT / "data").resolve()

eda_df = pd.read_parquet(DATA/"eda_dataset.parquet")

#### Read the data in

In [ ]:
eda_df.columns

In [ ]:
eda_df.Year.value_counts()

#### Modeling target decision

In [ ]:
target_decision = pd.DataFrame({
    "Decision": [
        "Outcome for reporting",
        "Modeling target",
        "Transformation method",
        "Justification"
    ],
    "Value": [
        "stdzd_amt_per_service (raw, $/service)",
        "log1p(stdzd_amt_per_service) (stored as log_stdzd_amt_per_service)",
        "TTR with log1p and expm1.",
        "Heavy-tailed raw outcome. Log reduces tail dominance and improves stability."
    ]
})
target_decision

#### Training inclusion filter

In [ ]:
# Define training inclusion mask explicitly (even if eda_df is already filtered)
train_mask = (
    (eda_df["services"] >= 11) &
    (eda_df["stdzd_amt_per_service"].notna()) &
    (eda_df["stdzd_amt_per_service"] >= 0)
)

train_filter_summary = pd.DataFrame({
    "Metric": [
        "Rows in eda_df",
        "Rows meeting training filter",
        "Share kept",
        "Unique NPIs (all)",
        "Unique NPIs (kept)",
        "Years present (all)",
        "Years present (kept)",
    ],
    "Value": [
        len(eda_df),
        int(train_mask.sum()),
        float(train_mask.mean()),
        eda_df["Rndrng_NPI"].nunique(),
        eda_df.loc[train_mask, "Rndrng_NPI"].nunique(),
        sorted(eda_df["Year"].unique().tolist()),
        sorted(eda_df.loc[train_mask, "Year"].unique().tolist()),
    ]
})

train_filter_summary

#### Define feature list:

In [ ]:
# Categorical features (to encode)
cat_features = [
    "rbcs_family_desc",    # or use "RBCS_FamNumb" instead, but pick one consistently
    "Place_Of_Srvc",
    "provider_type",
    "state",
    "ruca_bucket",
]

# Numeric features
num_features = [
    "bene_avg_risk_score",
    "years_since_enumeration",
    "log_services",
    "log_benes",
    "p_cancer6", "p_diabetes", "p_ckd", "p_copd", "p_htn",
]

# Final target
target_col = "stdzd_amt_per_service"

# Exclusions (documented)
excluded = [
    # raw cost outcomes besides target
    "log_stdzd_amt_per_service", "allowed_amt_per_service", "payment_amt_per_service", "submitted_charge_per_service",
    # totals derived from outcomes (spend columns)
    "stdzd_spend", "allowed_spend", "payment_spend", "submitted_spend",
    # flags/buckets used for slicing, not for training features
    "is_top_1pct_stdzd_amt_per_service", "svc_bucket", "services_bins", "services_custom", "services_custom2",
    # provider-year totals (often avoided to prevent scale leakage; can revisit intentionally later)
    "tot_mdcr_stdzd_amt",
]

features_table = pd.DataFrame({
    "Type": (["categorical"] * len(cat_features)) + (["numeric"] * len(num_features)) + (["target"] * 1),
    "Column": cat_features + num_features + [target_col]
})

features_table

This is our intended modeling feature set.

Categorical features
- `rbcs_family_desc`
- `Place_Of_Srvc`
- `provider_type`
- `state`
- `ruca_bucket`

Interpretation:
- These explain systematic price differences due to:
    - what service is being delivered,
    - where it is delivered,
    - what specialty is delivering it,
    - geography and rurality.

Numeric features
- risk and experience:
    - `bene_avg_risk_score`
    - `years_since_enumeration`
- exposure/intensity controls (log):
    - `log_services`
    - `log_benes`
- case mix proportions:
    - `p_cancer6`, `p_diabetes`, `p_ckd`, `p_copd`, `p_htn`

Interpretation:
- This is a classic “risk adjustment + context” set.
- We are not leaking the target because you excluded spend totals and other cost measures.

Modeling implication
- We picked `rbcs_family_desc` over `RBCS_FamNumb`. Although ID is often cleaner, desc is often fine too.

#### Quick availability check:

In [ ]:
missing_cols = [c for c in (cat_features + num_features + [target_col]) if c not in eda_df.columns]
missing_cols

`missing_cols` gives empty list `[]`

Interpretation
- Everything we plan to model exists in our dataframe.
- This prevents “modeling notebook surprises.”

#### Create split masks

In [ ]:
train_years = [2020, 2021, 2022]
test_years = [2023]

mask_train = eda_df["Year"].isin(train_years)
mask_test = eda_df["Year"].isin(test_years)

split_counts = pd.DataFrame({
    "Split": ["train", "test"],
    "Years": [train_years, test_years],
    "Rows": [int(mask_train.sum()), int(mask_test.sum())],
    "Unique NPIs": [eda_df.loc[mask_train, "Rndrng_NPI"].nunique(), eda_df.loc[mask_test, "Rndrng_NPI"].nunique()],
    "Spend share": [
        float(eda_df.loc[mask_train, "stdzd_spend"].sum() / eda_df["stdzd_spend"].sum()),
        float(eda_df.loc[mask_test, "stdzd_spend"].sum() / eda_df["stdzd_spend"].sum()),
    ],
})
split_counts

The output
- Train:
    - 213,296 rows
    - 19,838 NPIs
    - 66.6% spend
- Test:
    - 105,026 rows
    - 19,226 NPIs
    - 33.4% spend

Interpretation
- YoWeu have a healthy split. About one-third of dollars are in the test year.
- The train and test have similar provider counts, which is good for generalization tests.

Modeling implication
- This is a realistic production-like test: learn patterns from earlier years, apply to the next year.


#### Provider overlap (leakage / generalization check):

In [ ]:
npi_train = set(eda_df.loc[mask_train, "Rndrng_NPI"].unique().tolist())
npi_test = set(eda_df.loc[mask_test, "Rndrng_NPI"].unique().tolist())

overlap = npi_train.intersection(npi_test)
test_only = npi_test - npi_train

provider_overlap_tbl = pd.DataFrame({
    "Metric": ["Train NPIs", "Test NPIs", "Overlap NPIs", "Test-only NPIs"],
    "Value": [len(npi_train), len(npi_test), len(overlap), len(test_only)],
    "Share of test NPIs": [
        np.nan,
        1.0,
        len(overlap)/len(npi_test) if len(npi_test) else np.nan,
        len(test_only)/len(npi_test) if len(npi_test) else np.nan,
    ]
})
provider_overlap_tbl

The `provider_overlap_tbl` output table:
- `Test NPIs`: 19,226
- `Overlap with train`: 18,145 (94.38%)
- `Test-only`: 1,081 (5.62%)

Interpretation
- Most test providers were seen in train. That means your evaluation is mostly:
    - “new year for known providers” (easier)
- But you still have a meaningful cold-start set:
    - 1,081 providers

Modeling implication
- You should report performance separately for:
    - seen providers (in train)
    - unseen providers (test-only)

Because those are different deployment realities.

#### Planned slice keys table:

In [ ]:
eval_plan = pd.DataFrame({
    "Category": [
        "Primary metrics",
        "Target scale",
        "Core slices (report)",
        "Stability slices (flagging)",
        "Tail diagnostic"
    ],
    "Plan": [
        "MAE, RMSE",
        "log of stdzd_amt_per_service via TTR (log1p)",
        "provider_type, Place_Of_Srvc, ruca_bucket, state",
        "svc_bucket and services>=50 / >=100 subsets",
        "Compare tail vs non-tail behavior without changing labels"
    ]
})
eval_plan

#### Modeling readiness summary table (final contract):

In [ ]:
readiness_contract = pd.DataFrame({
    "Decision Area": [
        "Dataset grain",
        "Final modeling target",
        "Training inclusion",
        "Tail handling",
        "Visualization-only clipping",
        "Post-model flagging threshold",
        "Features (categorical)",
        "Features (numeric)",
        "Split plan",
        "Evaluation slices"
    ],
    "Final Choice": [
        "provider-year-RBCS family-place of service",
        "log of stdzd_amt_per_service via TTR = log1p(stdzd_amt_per_service)",
        "services >= 11; stdzd_amt_per_service not null and >= 0",
        "Keep tail; do not winsorize labels; use tail flag for diagnostics",
        "Allow p99 clipping for readability in plots only",
        "Flagging candidates evaluated on services >= 50 (and >= 100 sensitivity)",
        ", ".join(cat_features),
        ", ".join(num_features),
        "Train 2020–2022, Test 2023",
        "provider_type, Place_Of_Srvc, ruca_bucket, svc_bucket, tail flag (diagnostic)"
    ],
    "Why defensible": [
        "Matches EDA grain and planned use case",
        "Controls heavy tail while preserving signal",
        "Reliability threshold reduces denominator noise",
        "Tail appears real and interpretable (not pure error). Log target handles skew",
        "Prevents misleading plots without altering training distribution",
        "Reduces false positives from low-volume instability",
        "Captures key context (service, POS, specialty, geography, rurality)",
        "Captures case-mix + intensity + experience + comorbidity composition",
        "Temporal generalization is the real-world test",
        "Ensures interpretability and stability across key segments"
    ]
})

readiness_contract

#### Create data frames for modeling

In [ ]:
model_cols = cat_features + num_features + [target_col, "Year", "Rndrng_NPI", "services", "log_stdzd_amt_per_service"]

model_df = eda_df.loc[train_mask, model_cols].copy()

train_df = model_df[model_df["Year"].isin(train_years)].copy()
test_df = model_df[model_df["Year"].isin(test_years)].copy()

train_df.shape, test_df.shape

#### Missingness in features used in modeling

In [ ]:
feature_missing = (
    model_df[cat_features + num_features + [target_col]]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("pct_missing")
    .reset_index()
    .rename(columns={"index": "column"})
)
feature_missing

The `feature_missing` is the one we should pay attention to before training.

We have missingness in:
- `p_copd`: 4.59%
- `p_ckd`: 2.05%
- `p_diabetes`: 1.15%
- `p_cancer6`: 0.86%
- `years_since_enumeration`: 0.45%
- `p_htn`: 0.05%
Everything else: 0%

Interpret each variable’s missingness and what it implies

- `p_copd` (4.59% missing)
    - This is the highest missingness among our features.
    - Likely causes:
        - certain provider-year records lack the COPD percentage due to suppression, reporting rules, or merge gaps.
    - Modeling risk:
        - dropping rows would throw away ~4.6% of our data for one feature.
    - Recommended handling:
        - impute missing with a neutral value (commonly 0) plus a missingness indicator, or
        - impute with median and add indicator.
    - Why indicator matters:
        - missingness might correlate with provider type or geography, so the “missing” itself can carry signal.

- `p_ckd` (2.05% missing)
    - Similar logic, lower magnitude.
    - We should handle it the same way as p_copd for consistency.

- `p_diabetes` (1.15% missing)
    - Small but non-trivial.
    - Same treatment.

- `p_cancer6` (0.86% missing)
    - Small. Same treatment.
    - This one is especially sensitive conceptually in oncology, so do not silently drop rows.

- `years_since_enumeration` (0.45% missing)
    - This is “provider experience proxy.”
    - Missingness likely means:
        - NPI enumeration date missing upstream, or mapping failed.
    - Modeling risk:
        - leaving it missing can break some models.
    - Handling:
        - impute median and add missingness flag is the safest.

- `p_htn` (0.05% missing)
    - Very small. Still handle systematically (same imputation pattern).
    - Consistency matters. We do not want special-case logic for one feature.

- All categoricals have 0% missing
    - That is excellent. It means our slicing features are complete.
    - Especially important for:
        - `provider_type`, `Place_Of_Srvc`, `ruca_bucket`.

- `bene_avg_risk_score` has 0% missing
    - This is great because it is typically our primary adjustment feature.

Modeling implication
- We need a missingness strategy as part of Notebook 10 or the first modeling notebook.
- The best practice approach here is:

1.	For each numeric feature with missing:

- create `is_missing_<feature>` indicator

2.	Impute missing values:

- either 0 (for percentage fields) or median

3.	Keep the indicator in the model

This preserves rows, avoids bias from dropping, and allows missingness patterns to be learned.


#### Build X/y

In [ ]:
# Target is log target (your modeling target)
y_train = train_df[target_col].copy()
y_test  = test_df[target_col].copy()

# Keep these for analysis, but not as model predictors
id_cols = ["Rndrng_NPI", "Year"]

# Also exclude raw cost outcome (you do not want leakage)
exclude_from_X = [target_col, "log_stdzd_amt_per_service"] + id_cols

X_train = train_df.drop(columns=exclude_from_X).copy()
X_test  = test_df.drop(columns=exclude_from_X).copy()

Now X_train contains exactly `cat_features + num_features`. 

#### Preprocess with `ColumnTransformer`

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None"))
])

cat_pipe_ohe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])

preprocess = ColumnTransformer(
    transformers=[
        ("cat", cat_pipe, cat_features),
        ("num", num_pipe, num_features),
    ],
    remainder="drop"
)

preprocess_ohe = ColumnTransformer(
    transformers=[
        ("cat_ohe", cat_pipe_ohe, cat_features),
        ("num", num_pipe, num_features),
    ],
    remainder="drop"
)

#### Fit-transform train, transform test

In [ ]:
X_train_proc      = preprocess.fit_transform(X_train)
X_test_proc       = preprocess.transform(X_test)

X_train_proc_ohe  = preprocess_ohe.fit_transform(X_train)
X_test_proc_ohe   = preprocess_ohe.transform(X_test)

feat_names_ohe = preprocess_ohe.get_feature_names_out()
feat_names_ohe[:10]

In [ ]:
print(X_train_proc.shape)
print(X_test_proc.shape)
print(X_train_proc_ohe.shape)
print(X_test_proc_ohe.shape)

Note: We used `get_feature_names_out()` for debugging and SHAP later.

#### Inspect the reference categories for each categorical variable, i.e., `cat_features`:

Here, we need to 

1. reach into the `preprocess_ohe`, which is the our `ColumnTransformer` and 
2. grab the `cat_ohe` step, the `cat_pipe_ohe`, which is our `Pipeline`, 
3. then we reach into the ``cat_pipe_ohe` `Pipeline`, and 
4. grab the `ohe` step, which is our `OneHotEncoder(...)`

In [ ]:
# A list to store our reference categories
refs = []

# Get the fitted OHE object of the cat_pipe_ohe Pipeline
ohe = preprocess_ohe.named_transformers_["cat_ohe"].named_steps["ohe"]

# categories_ is a list alighed to cat_features
# each entry is an array of the learned categories for that feature 
for col, cats in zip(cat_features, ohe.categories_):
    refs.append({
        "feature": col,
        "reference_dropped":cats[0], # dropped because drop="first"
        "all_categories": list(cats), #optional, can be long for rbcs_family_desc
        "n_categories": len(cats)
    })

ref_table = pd.DataFrame(refs)[["feature","reference_dropped", "n_categories"]]
ref_table

Now we have:
- `X_train_proc`: imputed features
- `X_test_proc`: same columns, same encoder mapping, no leakage

- `X_train_proc_ohe`: one-hot encoded and imputed features
- `X_test_proc_ohe`: same columns, same encoder mapping, no leakage

#### Explain `UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros warnings.warn(msg, UserWarning)`

- During `fit_transform(X_train)`, the encoder learns the set of categories seen in **train** for each categorical feature.
- During `transform(X_test)`, it encountered at least one category in **column [0]** (that is your first categorical feature, `rbcs_family_desc`) that was **not present in train**.
- Because we set `handle_unknown="ignore"`, sklearn does not crash. It encodes those unseen categories as **all zeros across the OHE columns for that feature**.

So the model effectively treats “unseen category” as “none of the known categories”.

Let's add a small check so we know how often it happens (this tells us what fraction of test rows have unseen `rbcs_family_desc`):

In [ ]:
ohe = preprocess_ohe.named_transformers_["cat_ohe"].named_steps["ohe"]
cats = ohe.categories_[0]  # categories for first categorical feature
n_unknown = (~X_test[cat_features[0]].astype(str).isin(cats)).sum()
share_unknown = n_unknown / len(X_test)
n_unknown, share_unknown

The result basically says:

- **Only 2 rows in your entire 2023 test set** have an `rbcs_family_desc` value that never appeared in 2020–2022 training.
- That is **0.0019% of test rows** (about 1 in 52,500 rows).

So the warning is totally benign here.

In [ ]:
from preprocessing import CorrelationThreshold

# ==========================================
# RE-SYNC & PLOT (RUN THIS WHOLE BLOCK)
# ==========================================

# 0. Build a dense DataFrame with correct column names, then compute correlation
import scipy.sparse as sp

# X_train_proc_ohe is csr_matrix, feat_names_ohe length = 193
X_train_proc_ohe_dense = X_train_proc_ohe.toarray() if sp.issparse(X_train_proc_ohe) else np.asarray(X_train_proc_ohe)
X_train_proc_ohe_df = pd.DataFrame(X_train_proc_ohe_dense, columns=feat_names_ohe)

# 1. RE-CALCULATE DROPS (Ensure the list is fresh!)
corr_selector = CorrelationThreshold(threshold=0.9)
corr_selector.fit(X_train_proc_ohe_df)
dropped_cols = corr_selector.to_drop_

# 2. RE-CALCULATE SUBSET DATA
# We need to rebuild the subset_corr to match the fresh dropped_cols list
if len(dropped_cols) > 0:
    # Find partners again
    corr_matrix = X_train_proc_ohe_df.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    partners = []
    for dropped in dropped_cols:
        # Find the feature kept
        partner_match = upper.index[upper[dropped] > 0.9].tolist()
        if partner_match:
            partners.append(partner_match[0])
            
    # Combine lists
    features_to_plot = list(set(dropped_cols + partners))
    subset_corr = X_train_proc_ohe_df[features_to_plot].corr()

    # 3. PLOT
    plt.figure(figsize=(16, 14))
    ax = plt.gca()

    # Heatmap
    mask = np.triu(np.ones_like(subset_corr, dtype=bool))
    sns.heatmap(
        subset_corr,
        mask=mask,
        cmap='coolwarm',
        center=0,
        square=True,
        linewidths=.5,
        cbar_kws={"shrink": .5},
        annot=False, 
        ax=ax
    )

    # 4. HIGHLIGHTING LOOP
    # Fix X-axis labels
    new_x_labels = []
    for label in ax.get_xticklabels():
        text = label.get_text()
        if text in dropped_cols:
            label.set_color('red')
            label.set_weight('bold')
            label.set_text(f"[DROP] {text}") 
        new_x_labels.append(label)
    ax.set_xticklabels(new_x_labels)

    # Fix Y-axis labels
    new_y_labels = []
    for label in ax.get_yticklabels():
        text = label.get_text()
        if text in dropped_cols:
            label.set_color('red')
            label.set_weight('bold')
            label.set_text(f"[DROP] {text}")
        new_y_labels.append(label)
    ax.set_yticklabels(new_y_labels)

    plt.title(f"Redundancy Audit: Features marked with [DROP] will be removed ({len(dropped_cols)} total)")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

else:
    print("Zero redundancies found. Nothing to plot!")

In [ ]:
X_train_proc_ohe.shape, len(feat_names_ohe), [c for c in feat_names_ohe if "ruca_bucket" in c]

In [ ]:
# What categories are actually present in train for ruca_bucket?
X_train["ruca_bucket"].astype(str).value_counts(dropna=False).head(10)

In [ ]:
ohe = preprocess_ohe.named_transformers_["cat_ohe"].named_steps["ohe"]
# index of ruca_bucket within cat_features
ruca_idx = cat_features.index("ruca_bucket")
ohe.categories_[ruca_idx]

In [ ]:
dropped_cols

#### Extract the feature names for the `X_train_proc`

Remember that `X_train_proc` is a matrix, so it does not have "column names". 

We need to extract the column names from the `Pipeline` called `cat_pipe`:

In [ ]:
feat_names = preprocess.get_feature_names_out()
feat_names[:10]

In [ ]:
len(feat_names)

#### Create a pandas dataframe from the `X_train_proc` which is a sparse `csr_matrix` 

First, we need to turn it into a dense matrix

Second, we turn the dense matrix into a Pandas DataFrame

In [ ]:
# X_train_proc_ohe is csr_matrix, feat_names length = 193
X_train_proc_dense = X_train_proc.toarray() if sp.issparse(X_train_proc) else np.asarray(X_train_proc)
X_train_proc_df = pd.DataFrame(X_train_proc_dense, columns=feat_names)

In [ ]:
X_train_proc_df.head()

> Now we can inspect the `X_train_proc_df` just like the `X_train_proc_ohe_df`, if desired. We could apply the same logic to the `X_test_proc` and `X_test_proc_ohe` also. 

### Audit correlations on the numeric block only

#### Extract the transformed numeric matrix (with missingness indicators)

In [ ]:
num_features

In [ ]:
# Grab the fitted numeric pipeline (the num_pipe)
num_pipe_fitted = preprocess_ohe.named_transformers_["num"]

# Transform only the numeric columns 
X_train_num_proc = num_pipe_fitted.transform(X_train[num_features]) # numpy array, small width

#### Get the numeric feature names (including indicators)

In [ ]:
num_feat_names = num_pipe_fitted.get_feature_names_out(num_features)
num_feat_names

#### Run your CorrelationThreshold on the numeric DataFrame

In [ ]:
X_train_num_df = pd.DataFrame(X_train_num_proc, columns=num_feat_names, index=X_train.index)

corr_selector_num = CorrelationThreshold(threshold=0.9)
corr_selector_num.fit(X_train_num_df)

dropped_num_cols = corr_selector_num.to_drop_
dropped_num_cols

### Feature Extractor

Here I define a feature extraction function that systematically checks a model's attributes to identify the object, peel off any wrappers, grab the actual model, extract the coefficients or feature importances, construct a dataframe ready to for the next function to plot. 

In [ ]:
import numpy as np
import pandas as pd

def extract_model_features(model_object, feat_names):
    """
    Extract feature names and weights (coefficients or importances).

    Parameters
    ----------
    model_object : fitted estimator
        Can be Pipeline, GridSearchCV, TransformedTargetRegressor, or a plain estimator.
    feat_names : array-like
        Feature names that align with the model's final input space (post-preprocessing).
        Example: preprocess_ohe.get_feature_names_out()

    Returns
    -------
    pd.DataFrame with columns:
      Feature, Coefficient/Importance, Abs_Weight
    """

    # 1) unwrap GridSearchCV
    obj = model_object
    if hasattr(obj, "best_estimator_"):
        obj = obj.best_estimator_

    # 2) unwrap TransformedTargetRegressor
    if hasattr(obj, "regressor_"):
        obj = obj.regressor_

    # 3) identify final fitted estimator
    # If Pipeline, final step is last named step
    if hasattr(obj, "named_steps"):
        final_step_name = list(obj.named_steps.keys())[-1]
        final_model = obj.named_steps[final_step_name]
    else:
        final_model = obj

    current_features = np.array(feat_names)

    # 4) extract weights
    metric_name = None
    weights = None

    if hasattr(final_model, "coef_"):
        weights = final_model.coef_
        metric_name = "Coefficient"

        # Handle shape (1, n_features) or (n_targets, n_features)
        weights = np.asarray(weights)
        if weights.ndim == 2:
            # common single-target 2D shapes
            if 1 in weights.shape:
                weights = weights.ravel()
            else:
                raise ValueError(
                    f"coef_ is 2D with shape {weights.shape}. "
                    "This looks like multioutput. Decide which target to plot."
                )

        # Optional: drop exact/near zeros (useful for Lasso/ElasticNet)
        mask = np.abs(weights) > 1e-5
        current_features = current_features[mask]
        weights = weights[mask]

    elif hasattr(final_model, "feature_importances_"):
        weights = np.asarray(final_model.feature_importances_)
        metric_name = "Importance"

    elif hasattr(final_model, "get_feature_importance"):
        weights = np.asarray(final_model.get_feature_importance())
        metric_name = "Importance"

    elif hasattr(final_model, "estimators_"):
        print("VotingRegressor has no single weight vector. Plot base estimators individually.")
        return pd.DataFrame()

    else:
        raise TypeError(f"Unsupported model type for extraction: {type(final_model)}")

    # 5) sanity check alignment
    if len(current_features) != len(weights):
        raise ValueError(
            f"Feature name mismatch. len(features)={len(current_features)} "
            f"but len(weights)={len(weights)}. "
            "Make sure feat_names matches the model's final input space."
        )

    df = pd.DataFrame({
        "Feature": current_features,
        metric_name: weights
    })
    df["Abs_Weight"] = df[metric_name].abs()
    return df.sort_values("Abs_Weight", ascending=False)

### Feature Impact Visuzlization Function

Here I defined a plotting function that takes in the dataframe containing a model's coefficients or feature importance, sorts the top 20 features by the absolute values of their coefficients or importances (i.e., weights), then plots them as horizontal bar plot.

In [ ]:
# ==========================================
# FEATURE IMPACT VISUALIZATION FUNCTION
# ==========================================

def plot_feature_impact(df, title="Feature Impact", top_n=20):
    if df is None or df.empty:
        print("No data to plot.")
        return

    # Pick metric column
    if "Coefficient" in df.columns:
        metric_col = "Coefficient"
        is_coef = True
    elif "Importance" in df.columns:
        metric_col = "Importance"
        is_coef = False
    else:
        raise ValueError("df must contain either 'Coefficient' or 'Importance' column.")

    # Ensure sorted by absolute weight, then take top_n
    if "Abs_Weight" not in df.columns:
        df = df.copy()
        df["Abs_Weight"] = df[metric_col].abs()

    plot_df = (
        df.sort_values("Abs_Weight", ascending=False)
          .head(top_n)
          .sort_values("Abs_Weight", ascending=True)  # for horizontal bar readability
    )

    plt.figure(figsize=(10, 8))

    if is_coef:
        colors = ["green" if x > 0 else "red" for x in plot_df[metric_col]]
        xlabel = "Impact on log1p(stdzd_amt_per_service) (TTR space)"
    else:
        colors = "skyblue"
        xlabel = "Feature importance"

    plt.barh(plot_df["Feature"], plot_df[metric_col], color=colors)

    if is_coef:
        plt.axvline(x=0, color="black", linestyle="--", linewidth=0.8)

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Features")
    plt.tight_layout()
    plt.show()

# Start modeling

## OLS

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_squared_error

- Define the inner OLS pipe

In [ ]:
ols_inner_pipe = Pipeline(steps=[
    ("preprocess", preprocess_ohe),
    ("scaler", StandardScaler(with_mean=False)),  # sparse safe
    ("model", LinearRegression())
])

- wrap the inner pipe inside a `TransformedTargetRegressor()` (TTR)

In [ ]:
ols_full_model = TransformedTargetRegressor(
    regressor=ols_inner_pipe,
    func=np.log1p,
    inverse_func=np.expm1
)

- Fit the model

In [ ]:
ols_full_model.fit(X_train, y_train)

- Diagnostics

In [ ]:
pred_test = ols_full_model.predict(X_test)

mae = mean_absolute_error(y_test, pred_test)
rmse = root_mean_squared_error(y_test, pred_test)

print("OLS (Raw y + TTR(log1p))")
print("Train R2:", ols_full_model.score(X_train, y_train))
print("Test  R2:", ols_full_model.score(X_test, y_test))
print("MAE ($):", mae)
print("RMSE ($):", rmse)

1. It's likely he relationship is not well captured by a single global linear surface in this feature space (even after the log transform)
2. The RMSE being much larger than the MAE is a classic sign that a small fraction of predictions are very wrong (heavy tail, hard categories, or rare combinations). That is consistent with our EDA.

#### Investigate the reason for poor performance

- Let's make sure we did not accidentally score on the wrong target scale.

In [ ]:
from sklearn.metrics import r2_score

pred_test = ols_full_model.predict(X_test)

r2_dollars = r2_score(y_test, pred_test)

r2_log = r2_score(np.log1p(y_test), np.log1p(pred_test.clip(min=0)))

r2_dollars, r2_log

A) The OLS model is much better at ranking and relative cost than at matching dollars

An **R² of ~0.77 in log space** says the linear model is capturing a lot of the systematic structure in **log1p(cost)**.

But **R² of ~0.22 in dollars** says that once we convert back to dollars, the remaining errors (especially for high-cost cases) explode in magnitude and dominate the variance.

This is exactly what heavy-tailed outcomes do: a small number of expensive rows contribute a huge fraction of dollar variance.

B) The log transform changes the loss geometry

In log space, being off by (say) 30% and being off by 2x are “closer” than they look in dollars.

In dollars, those same misses can be hundreds or thousands of dollars and they dominate SSE, so dollar-scale R² drops.

C) The MAE and RMSE already hinted at this

MAE ~$32 but RMSE ~$152 means “most points are okay, but a few are very wrong in dollars”. Those few are usually the tail.

In [ ]:
pred_test = ols_full_model.predict(X_test)

test_eval = test_df.copy()
test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

# Bring back tail flag (and optionally svc_bucket) from eda_df using the shared index
test_eval = test_eval.join(
    eda_df.loc[:, ["is_top_1pct_stdzd_amt_per_service", "svc_bucket"]],
    how="left"
)

tail_summary = (
    test_eval.groupby("is_top_1pct_stdzd_amt_per_service", dropna=False)
    .agg(
        n=("pred", "size"),
        mae=("abs_err", "mean"),
        rmse=("sq_err", lambda s: np.sqrt(s.mean())),
        y_mean=(target_col, "mean"),
        pred_mean=("pred", "mean"),
    )
)

tail_summary

**Non-tail (False)**

- n = 104,107 test rows
- MAE ≈ $27.93
- RMSE ≈ $50.95
- Mean actual y (y_mean) ≈ $87.04
- Mean prediction (pred_mean) ≈ $85.14

Interpretation:

- On the bulk of the data, the model is roughly centered correctly (mean prediction close to mean actual).
- Error levels are moderate relative to the mean, and consistent with our earlier “R2 in dollars is low but R2 in log space is high” observation.

**Tail (True, top 1% cost per service)**

- n = 919 test rows
- MAE ≈ $514.68
- RMSE ≈ $1532.28
- Mean actual y (y_mean) ≈ $885.80
- Mean prediction (pred_mean) ≈ $372.09

Interpretation:

- The model **massively underpredicts** the tail on average.
- The mean prediction is **about 42%** of the mean actual.

A quick calculation we can do mentally:

- Bias in tail mean ≈ $885.8 − $372.1 ≈ **$513.7**, which is basically the MAE.
- That’s a strong sign the dominant error mode in the tail is “systematic underprediction,” not just noisy scatter.

#### Two small follow-up diagnostics that will help us confirm the story

**1. Tail mean ratio and bias**

In [ ]:
tail = tail_summary.loc[True]
ratio = tail["pred_mean"] / tail["y_mean"]
bias = tail["pred_mean"] - tail["y_mean"]
ratio, bias

- **Ratio = 0.4201**
    - On average, in the tail the OLS model predicts only **42%** of the true cost per service.
- **Bias = −$513.71**
    - On average, it’s under by about **$514** per row in the tail.

This aligns almost exactly with the tail MAE we saw (**~$514.68**). That’s not a coincidence. It means the dominant error mode in the tail is a consistent downward bias, not random noise.

**2. Tail share of total squared error (how much tail dominates RMSE)**

In [ ]:
err_share = (
    test_eval.groupby("is_top_1pct_stdzd_amt_per_service")["sq_err"]
    .sum()
    .pipe(lambda s: s / s.sum())
)
err_share

We found:

- **Tail rows (True) account for 88.87% of total squared error**
- **Non-tail rows (False) account for 11.13%**

This is the key takeaway:

Even though the tail is a tiny slice of rows (919 out of 105,026, under 1%), it contributes almost **9 out of every 10 “RMSE dollars”** because squared error explodes when we miss large values.

That means:

- Our **overall RMSE in dollars is basically a tail metric**.
- Improvements that help the bulk (non-tail) may barely move RMSE if the tail remains underpredicted.
- A model can look “fine” on non-tail MAE and still look “terrible” overall due to tail.

**3. Quick sanity check (very informative): This will tell us whether the tail errors are mostly “underpredict” vs “overpredict”:**

In [ ]:
tail_rows = test_eval["is_top_1pct_stdzd_amt_per_service"]
signed_err = (test_eval.loc[tail_rows, "pred"] - test_eval.loc[tail_rows, target_col])

signed_err.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99])

The tail errors are overwhelmingly **systematic underprediction**, with a few extreme misses that dominate RMSE.

**What each line tells us**

A) Median and percentiles confirm “almost always under”

- **50% (median) = −308.69**
- **75% = −145.17**
- **90% = −76.42**
- Even at the 90th percentile, the error is still negative. That means **at least 90% of tail rows are underpredicted**.

A quick inference we can safely state: **Underprediction is the norm, not an occasional issue.**

B) Only a tiny fraction overpredict

- **max = +125.48**
    
    So the worst overprediction in the tail is only +$125, while the worst underprediction is enormous (see below). That asymmetry is telling.

C) The mean matches the bias

- **mean = −513.71**, exactly what we computed before.
    
    So the “tail bias” number is not a fluke. It’s literally the average signed error.

D) RMSE is being crushed by a few catastrophic misses

- **min = −41,766.47**
- **std = 1,444.38**

That one line explains why tail RMSE is so huge. Squared error makes a single −$41k miss count like thousands of “normal” misses.

**4. What share of tail rows are underpredicted?**

In [ ]:
tail = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"]]
under_rate = (tail["pred"] < tail[target_col]).mean()
under_rate

**Underprediction rate = 99.13% (tail)**

0.9913 means **911 out of 919** tail rows are underpredicted (roughly). So the tail problem is not “high variance”. It is a **systematic downward bias** in the tail regime.

This matches everything we saw earlier:

- tail mean ratio ≈ 0.42
- tail mean bias ≈ −$514
- tail error percentiles mostly negative

**5. How many “catastrophic” misses are driving tail SSE?**

In [ ]:
tail = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"]].copy()
tail["sq_err"] = (tail["pred"] - tail[target_col])**2

# fraction of tail SSE explained by top k worst rows
for k in [1, 5, 10, 25, 50]:
    share = tail["sq_err"].nlargest(k).sum() / tail["sq_err"].sum()
    print(k, float(share))

**Tail SSE is dominated by a single catastrophic miss**

The SSE concentration is extreme:

- **Top 1 tail row explains 80.85% of tail SSE**
- Top 5 explains 83.64%
- Top 10 explains 84.98%
- Top 50 explains 90.53%

So when we report RMSE in dollars, we are mostly measuring “how bad is the single worst tail miss,” not the typical performance.

That also explains why:

- Dollar R² is low (because SSE is huge from a few points).
- Log-space R² looks strong (because the log compresses the effect of that outlier).

**6. Identify the single worst tail row and inspect it**

In [ ]:
tail = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"]].copy()
tail["err"] = tail["pred"] - tail[target_col]
tail["abs_err"] = tail["err"].abs()
tail["sq_err"] = tail["err"]**2

worst = tail.sort_values("sq_err", ascending=False).head(1)
worst[["Rndrng_NPI","Year","rbcs_family_desc","Place_Of_Srvc","provider_type","state","ruca_bucket","services",target_col,"pred","err","abs_err"]]

**Worst tail row**

- `rbcs_family_desc` = `Chemotherapeutic Agent`
- `provider_type` = `Radiation Oncology`
- `Place_Of_Srvc` = `O`
- `services` = `53`
- **actual** `stdzd_amt_per_service` = `41,967`
- **predicted** ~`201`
- `err`or `-41,766`

>So the model is behaving like “Chemotherapeutic Agent in this context should cost a few hundred per service,” but the data says “it is forty thousand per service.”

That combination is either:

1. **A real but extremely rare regime** our linear model cannot express from the current feature set (interactions, nonlinearities), or
2. **A coding / mapping / aggregation artifact** (less common, but worth ruling out because the magnitude is so extreme).

Either way, this single point dominating SSE is why our dollar-RMSE looks disastrous while log-space metrics look decent.

Let's check whether it is:

- a weird combination (rare RBCS family + unusual POS + tiny services just above threshold), or
- a data quality oddity (e.g., denominator effect, miscoding), or
- a genuinely extreme but real provider-year outlier.

**7. Quantify “typical tail error” with a robust metric**

In [ ]:
tail = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"]].copy()
tail["abs_err"] = (tail["pred"] - tail[target_col]).abs()

tail_abs_summary = tail["abs_err"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
tail_abs_summary

**Our tail abs error summary says**

From our tail `abs_err` distribution:

- Median tail miss: **~$309**
- 90th percentile: **~$962**
- 99th percentile: **~$2,370**
- Max: **~$41,766** (the monster)

So for 99 percent of tail rows, our error is in the hundreds to low thousands. Then one row is off by forty thousand and it blows up RMSE and SSE.

**8. Pull the underlying spend totals and other per-service fields for that exact row**

In [ ]:
row_idx = worst.index[0]

eda_df.loc[row_idx, [
    "Rndrng_NPI","Year","rbcs_family_desc","Place_Of_Srvc","provider_type","state","ruca_bucket",
    "services","benes",
    "stdzd_amt_per_service","stdzd_spend",
    "allowed_amt_per_service","allowed_spend",
    "payment_amt_per_service","payment_spend",
    "submitted_charge_per_service","submitted_spend"
]]

**9. Is this NPI consistently extreme, or is 2023 a one-off spike?**

In [ ]:
npi = worst["Rndrng_NPI"].iloc[0]

eda_df.loc[
    (eda_df["Rndrng_NPI"] == npi) & (eda_df["rbcs_family_desc"] == "Chemotherapeutic Agent"),
    ["Year","services","stdzd_amt_per_service","stdzd_spend","Place_Of_Srvc","provider_type","state"]
].sort_values("Year")

In [ ]:
npi

The “catastrophic miss” is a **real, stable, learnable pattern in the data**, not a one-off glitch.

A) It is not a construction artifact

Our per-service and total fields line up:

- `stdzd_spend` ≈ `services * stdzd_amt_per_service`
    
    `53` * `41,967.405094` ≈ `2,224,272.47` (matches our `stdzd_spend`)
    
- `payment_amt_per_service` == `stdzd_amt_per_service` and `payment_spend` == `stdzd_spend`
    
    So the standardized and payment views are consistent.
    
- `allowed_amt_per_service` is even higher (~52.7k), and `submitted_charge_per_service` is higher still (98k).
    
    That “submitted > allowed > paid/standardized” ordering is typical and also internally consistent.
    

So this is a genuine extremely high-cost provider-year-service bucket.

B) It is not a 2023 spike. It is stable across years for this NPI

Same NPI, same service family, same POS, same provider type, same state:

- 2021: ~41,862 per service
- 2022: ~41,705 per service
- 2023: ~41,967 per service

That stability is exactly what we want to see if this is “real behavior” rather than noise.

**10. How extreme is this row relative to its peer group?**

In [ ]:
peer = train_df.loc[
    (train_df["rbcs_family_desc"] == "Chemotherapeutic Agent") &
    (train_df["provider_type"] == "Radiation Oncology") &
    (train_df["Place_Of_Srvc"] == "O"),
    ["stdzd_amt_per_service","services","state","ruca_bucket"]
]

peer["stdzd_amt_per_service"].describe(percentiles=[0.5,0.9,0.95,0.99])

In [ ]:
peer

The peer group is bimodal. Most rows are cheap, a few are ultra-expensive. 

Our peer group is only 18 rows, and the distribution screams “two regimes”:

- median: **~$39.78 per service**
- 90th percentile: **~$12,540**
- 95th percentile: **~$41,729**
- max: **~$41,863**

So in the exact same coarse slice (Chemotherapeutic Agent + Radiation Oncology + POS=O), there are rows clustered around tens of dollars, and a small number clustered around ~42k.

That also explains why our OLS prediction is ~200. With the current feature set, the model mostly learns the dominant “low-cost mode,” and it has no reliable signal to identify which rows belong to the “ultra-expensive mode.”

The next most informative thing to compute is this, using the `test_eval`:
- For that `peer` slice, compare feature values (the numeric covariates) between the ultra-high rows and the low-cost rows. If they are indistinguishable, then we have strong evidence we need a more granular categorical feature (for example `rbcs_cat_subcat`) or a provider-history feature to capture the regime.

1. Let's create a new dataset called `peer_test_eval` from the `test_eval` where we get `rbcs_family_desc` = `"Chemotherapeutic Agent"`, `provider_type` = `"Radiation Oncology"`, `Place_Of_Srvc` = `"O Agent"`:

**1.A. Let's first make a copy of the sliced `test_eval` dataset:**

In [ ]:
peer_test_eval = test_eval.loc[
    (test_eval["rbcs_family_desc"] == "Chemotherapeutic Agent") &
    (test_eval["provider_type"] == "Radiation Oncology") &
    (test_eval["Place_Of_Srvc"] == "O")
].copy()

**1.B. Let's add a a new column `is_worst` that indicates the rows that belong to `npi` from `worst`.**

In [ ]:
peer_test_eval["is_worst"] = peer_test_eval["Rndrng_NPI"].eq(npi)

2. Let's define “ultra-high” vs “low-cost” within the peer slice

This is usually better than using the global top 1% flag, because our peer slice is already narrow.

In [ ]:
num_cols = num_features  # our list

peer = peer_test_eval.copy()

# Define ultra-high and low-cost within this peer slice
hi_cut = peer[target_col].quantile(0.90)   # top 10% within peer
lo_cut = peer[target_col].quantile(0.50)   # bottom 50% within peer

peer["cost_group"] = np.select(
    [peer[target_col] >= hi_cut, peer[target_col] <= lo_cut],
    ["ultra_high", "low_cost"],
    default="middle"
)

peer["cost_group"].value_counts(dropna=False)

3. Let's compare numeric covariates between groups

This produces a compact “are they distinguishable?” table for numeric features:

3.A. Mean/median comparison table

In [ ]:
compare_groups = peer.loc[peer["cost_group"].isin(["ultra_high", "low_cost"])].copy()

summary = (
    compare_groups
    .groupby("cost_group")[num_cols]
    .agg(["mean", "median", "std"])
)

summary

3.B. Add standardized mean difference (best quick signal)

This gives us a single “effect size” number per feature. If SMD is near 0, the groups are basically indistinguishable on that feature.

In [ ]:
def one_vs_group_z(x, group):
    x = float(x)
    g = np.asarray(group, dtype=float)
    mu = np.nanmean(g)
    sd = np.nanstd(g, ddof=1)  # ok because low_cost has n=4 here
    return (x - mu) / sd if sd > 0 else np.nan

hi = peer.loc[peer["cost_group"] == "ultra_high"]
lo = peer.loc[peer["cost_group"] == "low_cost"]

z_tbl = pd.DataFrame({
    "feature": num_cols,
    "ultra_high_value": [hi[c].iloc[0] for c in num_cols],
    "low_cost_mean":    [lo[c].mean() for c in num_cols],
    "low_cost_sd":      [lo[c].std(ddof=1) for c in num_cols],
    "z_vs_low_cost":    [one_vs_group_z(hi[c].iloc[0], lo[c]) for c in num_cols],
}).sort_values("z_vs_low_cost", key=lambda s: s.abs(), ascending=False)

z_tbl

The key interpretation rule:

- **Negative z**: `ultra_high` value is **below** the `low_cost` mean.
- **Positive z**: `ultra_high` value is **above** the `low_cost` mean.
- **Magnitude**:
    - |z| ≈ 0 to 1: not very different
    - |z| ≈ 2: pretty different
    - |z| ≥ 3: extremely different (especially with only 4 `low_cost` rows, this is a strong signal that this point sits far from that group on that feature)

Now our table:

1) `log_services`: z = -7.37 (huge)

- `ultra_high_value` = 3.988984
- `low_cost_mean` = 9.559847
- `low_cost_sd` = 0.756033
- `z_vs_low_cost` = (3.99 - 9.56) / 0.756 ≈ -7.37

Interpretation:

- Within this peer slice, the `ultra_high` row has **much lower `log_services`** than the `low_cost` rows.
- Since `log_services` is log-transformed, this is a massive difference on the original services scale.
- This is a red flag that the `ultra_high` cost-per-service case might be associated with a very different volume regime (even though our `services` column for the worst row was `53`, the `low_cost` rows in this peer slice likely have much higher services if their `log_services` mean is `9.56`, which is extremely large). That suggests we should sanity-check how `log_services` was defined in this dataset.

This single row is telling us: “I’m expensive per service, but I do not have high service volume relative to these `low_cost` rows.”

2) `bene_avg_risk_score`: z = -3.57

- `ultra_high` row’s beneficiaries are **lower risk** than `low_cost` mean by ~3.6 SDs.
- If this holds up, it suggests the extreme cost-per-service is not explained by higher risk score, at least not relative to these low-cost peers.

3) `p_copd`: z = -2.22 and `p_ckd`: z = -1.93

- `ultra_high` row has **lower COPD and CKD prevalence** than `low_cost` peers, relative to the `low_cost` variation.
- Again, this pushes against “this is just sicker patients” as the explanation.

4) `p_cancer6`: z = +1.54

- `ultra_high` has somewhat higher cancer prevalence than `low_cost`, but only ~1.5 SD.
- Not nothing, but not nearly as extreme as the service-volume signal.

5) `log_benes`: z = -1.21

- `ultra_high` row has fewer beneficiaries (or whatever `log_benes` captures) than `low_cost` peers, by ~1.2 SD.

6) `p_htn`, `p_diabetes`, `years_since_enumeration`: z near 0

- These look basically similar between `ultra_high` and `low_cost` within this peer slice.

Within that very narrow peer slice, the ultra-high cost-per-service row is not “high” because the numeric covariates scream “complex population.” Instead it looks like:

- **Lower volume signals (`log_services`, `log_benes`)**
- Some comorbidity rates are lower, not higher
- Cancer prevalence is a bit higher, but not enough to explain a 40k per service situation

That supports the hypothesis we mentioned earlier: **our feature set cannot represent the regime that creates ultra-high per-service costs**, because it is likely driven by something categorical or structural we are not encoding at the right granularity (or by a special pricing/HCPCS subcategory, drug, setting nuance, etc.).

- Just for sanity check, let's look at the `services`, `log_services`, `stdzd_amt_per_service` columns of the `peer` dataframe and calculate services from `log_services` column named `services_from_log`:

In [ ]:
peer.loc[peer["cost_group"] == "low_cost", ["services", "log_services", target_col]] \
    .assign(services_from_log=lambda d: np.expm1(d["log_services"])) \
    .sort_values("services", ascending=False)

#### OLS performance and diagnostics summary:

- Dollar space (business impact): MAE, RMSE, dollar R²
- Log space (relative error, stability): MAE/RMSE on log1p(y) and log R²

In [ ]:
pred_test = ols_full_model.predict(X_test)
pred_test_clip = pred_test.clip(min=0)

mae = mean_absolute_error(y_test, pred_test)
rmse = root_mean_squared_error(y_test, pred_test)

log_pred_test = ols_full_model.regressor_.predict(X_test)  # predictions in transformed target space
r2_log_true = r2_score(np.log1p(y_test), log_pred_test)
mae_log_true = mean_absolute_error(np.log1p(y_test), log_pred_test)
rmse_log_true = root_mean_squared_error(np.log1p(y_test), log_pred_test)

print("OLS (Raw y + TTR(log1p))")
print("Train R2 ($):", ols_full_model.score(X_train, y_train))
print("Test  R2 ($):", ols_full_model.score(X_test, y_test))
print("MAE ($):", mae)
print("RMSE ($):", rmse)
print("Test  R2 (log $):", r2_log_true)
print("MAE (log $):", r2_log_true)
print("RMSE (log $):", rmse_log_true)

Let's prettify the prints:

In [ ]:
print("OLS (Raw y + TTR(log1p))")
print(f"Train R2 ($):     {ols_full_model.score(X_train, y_train):.3f}")
print(f"Test  R2 ($):     {ols_full_model.score(X_test, y_test):.3f}")
print(f"MAE ($):          {mae:.3f}")
print(f"RMSE ($):         {rmse:.3f}")
print(f"Test  R2 (log $): {r2_log_true:.3f}")
print(f"MAE (log $):      {r2_log_true:.3f}")
print(f"RMSE (log $):     {rmse_log_true:.3f}")

- Tail calibration: for `is_top_1pct_stdzd_amt_per_service` rows, track:
    - pred_mean / y_mean ratio
    - underprediction rate
    - tail share of SSE

In [ ]:
tail_summary_1 = tail_summary.copy()
tail_summary_1["pred_to_y_ratio"] = tail_summary_1["pred_mean"] / tail_summary_1["y_mean"]

In [ ]:
tail_summary_2 = (test_eval
 .assign(is_under_predicted = lambda d: d["pred"]<d[target_col])
 .groupby("is_top_1pct_stdzd_amt_per_service")
 .agg(under_prediction_rate = ("is_under_predicted", "mean"),
      sse = ("sq_err","sum"))
 .assign(share_of_total_sse = lambda d: d["sse"]/d["sse"].sum()))
tail_summary_2

In [ ]:
eval_tail = pd.concat([tail_summary_1,tail_summary_2], axis=1)
eval_tail["bias"] = eval_tail["pred_mean"] - eval_tail["y_mean"]
eval_tail

Let's prettify the output of `eval_tail`:

In [ ]:
eval_tail_pretty = eval_tail.round(3)
eval_tail_pretty

## Ridge

In [ ]:
from sklearn.linear_model import Ridge

ridge_inner_pipe = Pipeline(steps=[
    ("preprocess", preprocess_ohe),
    ("scaler", StandardScaler(with_mean=False)),
    ("model", Ridge())
])

In [ ]:
ridge_full_model = TransformedTargetRegressor(
    regressor=ridge_inner_pipe,
    func=np.log1p,
    inverse_func=np.expm1
)

In [ ]:
ridge_full_model.fit(X_train,y_train)

In [ ]:
pred_test = ridge_full_model.predict(X_test)
pred_test_clip = pred_test.clip(min=0)

mae = mean_absolute_error(y_test, pred_test)
rmse = root_mean_squared_error(y_test, pred_test)

log_pred_test = ridge_full_model.regressor_.predict(X_test)  # predictions in transformed target space
r2_log_true = r2_score(np.log1p(y_test), log_pred_test)
mae_log_true = mean_absolute_error(np.log1p(y_test), log_pred_test)
rmse_log_true = root_mean_squared_error(np.log1p(y_test), log_pred_test)

print("Ridge (Raw y + TTR(log1p))")
print("Train R2 ($):", ridge_full_model.score(X_train, y_train))
print("Test  R2 ($):", ridge_full_model.score(X_test, y_test))
print("MAE ($):", mae)
print("RMSE ($):", rmse)
print("Test  R2 (log $):", r2_log_true)
print("MAE (log $):", mae_log_true)
print("RMSE (log $):", rmse_log_true)

## XGBoost

In [ ]:
from xgboost import XGBRegressor

xgb_inner_pipe = Pipeline(steps=[
    ("preprocess", preprocess_ohe),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=1,
        random_state=0
    ))
])

In [ ]:
xgb_full_model = TransformedTargetRegressor(
    regressor=xgb_inner_pipe,
    func=np.log1p,
    inverse_func=np.expm1
)

In [ ]:
xgb_full_model.fit(X_train,y_train)

In [ ]:
pred_test = xgb_full_model.predict(X_test)
pred_test_clip = pred_test.clip(min=0)

mae = mean_absolute_error(y_test, pred_test)
rmse = root_mean_squared_error(y_test, pred_test)

log_pred_test = xgb_full_model.regressor_.predict(X_test)  # predictions in transformed target space
r2_log_true = r2_score(np.log1p(y_test), log_pred_test)
mae_log_true = mean_absolute_error(np.log1p(y_test), log_pred_test)
rmse_log_true = root_mean_squared_error(np.log1p(y_test), log_pred_test)

print("XGBoost (Raw y + TTR(log1p))")
print("Train R2 ($):", xgb_full_model.score(X_train, y_train))
print("Test  R2 ($):", xgb_full_model.score(X_test, y_test))
print("MAE ($):", mae)
print("RMSE ($):", rmse)
print("Test  R2 (log $):", r2_log_true)
print("MAE (log $):", mae_log_true)
print("RMSE (log $):", rmse_log_true)

In [ ]:
pred_test = xgb_full_model.predict(X_test)

test_eval = test_df.copy()
test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

# Bring back tail flag (and optionally svc_bucket) from eda_df using the shared index
test_eval = test_eval.join(
    eda_df.loc[:, ["is_top_1pct_stdzd_amt_per_service", "svc_bucket"]],
    how="left"
)

tail_summary = (
    test_eval.groupby("is_top_1pct_stdzd_amt_per_service", dropna=False)
    .agg(
        n=("pred", "size"),
        mae=("abs_err", "mean"),
        rmse=("sq_err", lambda s: np.sqrt(s.mean())),
        y_mean=(target_col, "mean"),
        pred_mean=("pred", "mean"),
    )
)

tail_summary

In [ ]:
tail_summary_1 = tail_summary.copy()
tail_summary_1["pred_to_y_ratio"] = tail_summary_1["pred_mean"] / tail_summary_1["y_mean"]

In [ ]:
tail_summary_2 = (test_eval
 .assign(is_under_predicted = lambda d: d["pred"]<d[target_col])
 .groupby("is_top_1pct_stdzd_amt_per_service")
 .agg(under_prediction_rate = ("is_under_predicted", "mean"),
      sse = ("sq_err","sum"))
 .assign(share_of_total_sse = lambda d: d["sse"]/d["sse"].sum()))
tail_summary_2

In [ ]:
eval_tail = pd.concat([tail_summary_1,tail_summary_2], axis=1)
eval_tail["bias"] = eval_tail["pred_mean"] - eval_tail["y_mean"]
eval_tail

## **Increase granularity of the train and test dataset**

### Redefine `cat_features`

In [ ]:
# Categorical features (to encode)
cat_features = [
    "rbcs_cat_subcat",    # replaced "rbcs_family_desc" for more granularity
    "Place_Of_Srvc",
    "provider_type",
    "state",
    "ruca_bucket",
]

# BELOW STAYS THE SAME AS BEFORE

# # Numeric features
# num_features = [
#     "bene_avg_risk_score",
#     "years_since_enumeration",
#     "log_services",
#     "log_benes",
#     "p_cancer6", "p_diabetes", "p_ckd", "p_copd", "p_htn",
# ]

# # Final target
# target_col = "stdzd_amt_per_service"

# # Exclusions (documented)
# excluded = [
#     # raw cost outcomes besides target
#     "log_stdzd_amt_per_service", "allowed_amt_per_service", "payment_amt_per_service", "submitted_charge_per_service",
#     # totals derived from outcomes (spend columns)
#     "stdzd_spend", "allowed_spend", "payment_spend", "submitted_spend",
#     # flags/buckets used for slicing, not for training features
#     "is_top_1pct_stdzd_amt_per_service", "svc_bucket", "services_bins", "services_custom", "services_custom2",
#     # provider-year totals (often avoided to prevent scale leakage; can revisit intentionally later)
#     "tot_mdcr_stdzd_amt",
# ]

# features_table = pd.DataFrame({
#     "Type": (["categorical"] * len(cat_features)) + (["numeric"] * len(num_features)) + (["target"] * 1),
#     "Column": cat_features + num_features + [target_col]
# })

# features_table

In [ ]:
# Reprint the features_table as sanity-check
features_table = pd.DataFrame({
    "Type": (["categorical"] * len(cat_features)) + (["numeric"] * len(num_features)) + (["target"] * 1),
    "Column": cat_features + num_features + [target_col]
})

features_table

#### Recreate dataframes for modeling

In [ ]:
model_cols = cat_features + num_features + [target_col, "Year", "Rndrng_NPI", "services", "log_stdzd_amt_per_service"]

model_df = eda_df.loc[train_mask, model_cols].copy()

train_df = model_df[model_df["Year"].isin(train_years)].copy()
test_df = model_df[model_df["Year"].isin(test_years)].copy()

train_df.shape, test_df.shape

#### Rebuild X/y

In [ ]:
# Target is log target (your modeling target)
y_train = train_df[target_col].copy()
y_test  = test_df[target_col].copy()

# Keep these for analysis, but not as model predictors
id_cols = ["Rndrng_NPI", "Year"]

# Also exclude raw cost outcome (you do not want leakage)
exclude_from_X = [target_col, "log_stdzd_amt_per_service"] + id_cols

X_train = train_df.drop(columns=exclude_from_X).copy()
X_test  = test_df.drop(columns=exclude_from_X).copy()

#### Redefine the `preprocess` and `preprocess_ohe` (so they start with clean slate; nothing fitted them)

In [ ]:
cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None"))
])

cat_pipe_ohe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])

preprocess = ColumnTransformer(
    transformers=[
        ("cat", cat_pipe, cat_features),
        ("num", num_pipe, num_features),
    ],
    remainder="drop"
)

preprocess_ohe = ColumnTransformer(
    transformers=[
        ("cat_ohe", cat_pipe_ohe, cat_features),
        ("num", num_pipe, num_features),
    ],
    remainder="drop"
)

#### Re fit-transform train and transform test using `preprocess` and `preprocess_ohe` (as defined before)

In [ ]:
X_train_proc      = preprocess.fit_transform(X_train)
X_test_proc       = preprocess.transform(X_test)

X_train_proc_ohe  = preprocess_ohe.fit_transform(X_train)
X_test_proc_ohe   = preprocess_ohe.transform(X_test)

feat_names_ohe = preprocess_ohe.get_feature_names_out()
print(feat_names_ohe[:10])

feat_names = preprocess.get_feature_names_out()
print(feat_names)

#### Introducing weights to the train data

> Here, we have to ensure we are giving higher weights to the actual tail rows because we want the model to care more about those data points in the tail. They are NOT OUTLIERS. If they were, we would give them lower weight to tell the model to not care about them as much. 

In [ ]:
is_tail_train = eda_df.loc[X_train.index, "is_top_1pct_stdzd_amt_per_service"].astype(int)
w_tail = 25
sample_w = 1 + (w_tail - 1) * is_tail_train  # non-tail=1, tail=w_tail

#### Rebuild the `xgb_inner_pipe`

In [ ]:
xgb_inner_pipe = Pipeline(steps=[
    ("preprocess", preprocess_ohe),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=1,
        random_state=0
    ))
])

#### Rebuild `xgb_full_model` with updated `xgb_inner_pipe`:

In [ ]:
xgb_full_model = TransformedTargetRegressor(
    regressor=xgb_inner_pipe,
    func=np.log1p,
    inverse_func=np.expm1
)

#### Fit the model with `model__sample_weight = sample_w`

In [ ]:
xgb_full_model.fit(
    X_train, y_train, model__sample_weight=sample_w
)

#### Redo model diagnostics

In [ ]:
pred_test = xgb_full_model.predict(X_test)
pred_train = xgb_full_model.predict(X_train)
pred_test_clip = pred_test.clip(min=0)

mae = mean_absolute_error(y_test, pred_test)
rmse = root_mean_squared_error(y_test, pred_test)

log_pred_train = xgb_full_model.regressor_.predict(X_train)  # predictions in transformed target space
log_pred_test = xgb_full_model.regressor_.predict(X_test)  # predictions in transformed target space

r2_log_true_train = r2_score(np.log1p(y_train), log_pred_train)
r2_log_true_test = r2_score(np.log1p(y_test), log_pred_test)

mae_log_true = mean_absolute_error(np.log1p(y_test), log_pred_test)
rmse_log_true = root_mean_squared_error(np.log1p(y_test), log_pred_test)

# Create weights for the test set (following same logic as train)
is_tail_test = eda_df.loc[X_test.index, "is_top_1pct_stdzd_amt_per_service"].astype(int)
sample_w_test = 1 + (w_tail - 1) * is_tail_test

r2_train_w = r2_score(y_train, pred_train, sample_weight=sample_w)
r2_test_w = r2_score(y_test, pred_test, sample_weight=sample_w_test)
mae_w = mean_absolute_error(y_test, pred_test, sample_weight=sample_w_test)
rmse_w = root_mean_squared_error(y_test, pred_test, sample_weight=sample_w_test)

print("XGBoost (Raw y + TTR(log1p))")
print(50*"-")
print("Train R2 ($) (Unweighted: Average Case):", xgb_full_model.score(X_train, y_train))
print("Test  R2 ($) (Unweighted: Average Case):", xgb_full_model.score(X_test, y_test))
print("MAE      ($) (Unweighted: Average Case):", mae)
print("RMSE     ($) (Unweighted: Average Case):", rmse)
print(50*"-")
print("Train R2 (log $) (Unweighted: Average Case):", r2_log_true_train)
print("Test  R2 (log $) (Unweighted: Average Case):", r2_log_true_test)
print("MAE      (log $) (Unweighted: Average Case):", mae_log_true)
print("RMSE     (log $) (Unweighted: Average Case):", rmse_log_true)
print(50*"-")
print("Train R2 ($) (Weighted: Eval Metrics):", r2_train_w)
print("Test  R2 ($) (Weighted: Eval Metrics):", r2_test_w)
print("MAE      ($) (Weighted: Eval Metrics):", mae_w)
print("RMSE     ($) (Weighted: Eval Metrics):", rmse_w)


In [ ]:
pred_test = xgb_full_model.predict(X_test)

test_eval = test_df.copy()
test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

# Bring back tail flag (and optionally svc_bucket) from eda_df using the shared index
test_eval = test_eval.join(
    eda_df.loc[:, ["is_top_1pct_stdzd_amt_per_service", "svc_bucket"]],
    how="left"
)

tail_summary = (
    test_eval.groupby("is_top_1pct_stdzd_amt_per_service", dropna=False)
    .agg(
        n=("pred", "size"),
        mae=("abs_err", "mean"),
        rmse=("sq_err", lambda s: np.sqrt(s.mean())),
        y_mean=(target_col, "mean"),
        pred_mean=("pred", "mean"),
    )
)

tail_summary_1 = tail_summary.copy()
tail_summary_1["pred_to_y_ratio"] = tail_summary_1["pred_mean"] / tail_summary_1["y_mean"]

tail_summary_2 = (test_eval
 .assign(is_under_predicted = lambda d: d["pred"]<d[target_col])
 .groupby("is_top_1pct_stdzd_amt_per_service")
 .agg(under_prediction_rate = ("is_under_predicted", "mean"),
      sse = ("sq_err","sum"))
 .assign(share_of_total_sse = lambda d: d["sse"]/d["sse"].sum()))

eval_tail = pd.concat([tail_summary_1,tail_summary_2], axis=1)
eval_tail["bias"] = eval_tail["pred_mean"] - eval_tail["y_mean"]
eval_tail

#### Fit the model WITHOUT `model__sample_weight = sample_w` (UNWEIGHTED BASELINE)

In [ ]:
xgb_inner_pipe = Pipeline(steps=[
    ("preprocess", preprocess_ohe),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=1,
        random_state=0
    ))
])

xgb_full_model = TransformedTargetRegressor(
    regressor=xgb_inner_pipe,
    func=np.log1p,
    inverse_func=np.expm1
)

xgb_full_model.fit(
    X_train, y_train
)

In [ ]:
pred_test = xgb_full_model.predict(X_test)
pred_train = xgb_full_model.predict(X_train)
pred_test_clip = pred_test.clip(min=0)

mae = mean_absolute_error(y_test, pred_test)
rmse = root_mean_squared_error(y_test, pred_test)

log_pred_train = xgb_full_model.regressor_.predict(X_train)  # predictions in transformed target space
log_pred_test = xgb_full_model.regressor_.predict(X_test)  # predictions in transformed target space

r2_log_true_train = r2_score(np.log1p(y_train), log_pred_train)
r2_log_true_test = r2_score(np.log1p(y_test), log_pred_test)

mae_log_true = mean_absolute_error(np.log1p(y_test), log_pred_test)
rmse_log_true = root_mean_squared_error(np.log1p(y_test), log_pred_test)

print("XGBoost (Raw y + TTR(log1p))")
print(50*"-")
print("Train R2 ($) (Unweighted: Average Case):", xgb_full_model.score(X_train, y_train))
print("Test  R2 ($) (Unweighted: Average Case):", xgb_full_model.score(X_test, y_test))
print("MAE      ($) (Unweighted: Average Case):", mae)
print("RMSE     ($) (Unweighted: Average Case):", rmse)
print(50*"-")
print("Train R2 (log $) (Unweighted: Average Case):", r2_log_true_train)
print("Test  R2 (log $) (Unweighted: Average Case):", r2_log_true_test)
print("MAE      (log $) (Unweighted: Average Case):", mae_log_true)
print("RMSE     (log $) (Unweighted: Average Case):", rmse_log_true)


In [ ]:
pred_test = xgb_full_model.predict(X_test)

test_eval = test_df.copy()
test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

# Bring back tail flag (and optionally svc_bucket) from eda_df using the shared index
test_eval = test_eval.join(
    eda_df.loc[:, ["is_top_1pct_stdzd_amt_per_service", "svc_bucket"]],
    how="left"
)

tail_summary = (
    test_eval.groupby("is_top_1pct_stdzd_amt_per_service", dropna=False)
    .agg(
        n=("pred", "size"),
        mae=("abs_err", "mean"),
        rmse=("sq_err", lambda s: np.sqrt(s.mean())),
        y_mean=(target_col, "mean"),
        pred_mean=("pred", "mean"),
    )
)

tail_summary_1 = tail_summary.copy()
tail_summary_1["pred_to_y_ratio"] = tail_summary_1["pred_mean"] / tail_summary_1["y_mean"]

tail_summary_2 = (test_eval
 .assign(is_under_predicted = lambda d: d["pred"]<d[target_col])
 .groupby("is_top_1pct_stdzd_amt_per_service")
 .agg(under_prediction_rate = ("is_under_predicted", "mean"),
      sse = ("sq_err","sum"))
 .assign(share_of_total_sse = lambda d: d["sse"]/d["sse"].sum()))

eval_tail = pd.concat([tail_summary_1,tail_summary_2], axis=1)
eval_tail["bias"] = eval_tail["pred_mean"] - eval_tail["y_mean"]
eval_tail

In [ ]:
weights = [1,2,5,10,15,25]
r2_train_lst = []
r2_test_lst = []
mae_lst = []
rmse_lst = []
r2_train_log_lst = []
r2_test_log_lst = []
mae_log_lst = []
rmse_log_lst = []
r2_train_w_lst = []
r2_test_w_lst = []
mae_w_lst = []
rmse_w_lst = []
non_tail_pred_to_y_ratio_lst = []
tail_pred_to_y_ratio_lst = []
non_tail_sse_lst = []
tail_sse_lst = []
non_tail_share_of_total_sse_lst = []
tail_share_of_total_sse_lst = []
non_tail_under_prediction_rate_lst = []
tail_under_prediction_rate_lst = []
non_tail_bias_lst = []
tail_bias_lst = []

for w in weights:

    # define the training weights 
    is_tail_train = eda_df.loc[X_train.index, "is_top_1pct_stdzd_amt_per_service"].astype(int)
    w_tail = w
    sample_w = 1 + (w_tail - 1) * is_tail_train  # non-tail=1, tail=w_tail

    # define the inner pipe
    xgb_inner_pipe = Pipeline(steps=[
    ("preprocess", preprocess_ohe),
    ("model", XGBRegressor(objective="reg:squarederror", tree_method = "hist", n_jobs = 1, random_state = 0))
    ])

    # define the full model 
    xgb_full_model = TransformedTargetRegressor(
    regressor=xgb_inner_pipe,
    func=np.log1p,
    inverse_func=np.expm1
    )

    # Fit the model
    xgb_full_model.fit(
    X_train, y_train, model__sample_weight=sample_w
    )

    # Run diagnostics

    # get train and test predictions and clip test predictions 
    pred_test = xgb_full_model.predict(X_test)
    pred_train = xgb_full_model.predict(X_train)
    pred_test_clip = pred_test.clip(min=0)

    # get weighted-train unweighted/average-case metrics
    r2_train = xgb_full_model.score(X_train, y_train)
    r2_test = xgb_full_model.score(X_test, y_test)
    mae = mean_absolute_error(y_test, pred_test)
    rmse = root_mean_squared_error(y_test, pred_test)

    # get weighted-train unweighted/average-case metrics in log space
    log_pred_train = xgb_full_model.regressor_.predict(X_train)  # predictions in transformed target space
    log_pred_test = xgb_full_model.regressor_.predict(X_test)  # predictions in transformed target space
    
    r2_log_true_train = r2_score(np.log1p(y_train), log_pred_train)
    r2_log_true_test = r2_score(np.log1p(y_test), log_pred_test)
    mae_log_true = mean_absolute_error(np.log1p(y_test), log_pred_test)
    rmse_log_true = root_mean_squared_error(np.log1p(y_test), log_pred_test)

    # get weighted-train weighted/eval metrics
    
    # Create weights for the test set (following same logic as train)
    is_tail_test = eda_df.loc[X_test.index, "is_top_1pct_stdzd_amt_per_service"].astype(int)
    sample_w_test = 1 + (w_tail - 1) * is_tail_test

    r2_train_w = r2_score(y_train, pred_train, sample_weight=sample_w)
    r2_test_w = r2_score(y_test, pred_test, sample_weight=sample_w_test)
    mae_w = mean_absolute_error(y_test, pred_test, sample_weight=sample_w_test)
    rmse_w = root_mean_squared_error(y_test, pred_test, sample_weight=sample_w_test)

    # Calculate tail evals
    test_eval = test_df.copy().join(
    eda_df.loc[:, ["is_top_1pct_stdzd_amt_per_service"]],
    how="left"
    )
    test_eval["pred"] = pred_test
    test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
    test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

    non_tail_pred_to_y_ratio = test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"].mean() / test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],target_col].mean()
    tail_pred_to_y_ratio = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"].mean() / test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],target_col].mean()

    non_tail_sse = test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],"sq_err"].sum()
    tail_sse = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],"sq_err"].sum()

    non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
    tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

    non_tail_under_prediction_rate = ((test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"]) < test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],target_col]).mean()
    tail_under_prediction_rate = ((test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"]) < test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],target_col]).mean()

    non_tail_bias = test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"].mean() - test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],target_col].mean()
    tail_bias = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"].mean() - test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],target_col].mean()


    r2_train_lst.append(r2_train)
    r2_test_lst.append(r2_test)
    mae_lst.append(mae)
    rmse_lst.append(rmse)
    r2_train_log_lst.append(r2_log_true_train)
    r2_test_log_lst.append(r2_log_true_test)
    mae_log_lst.append(mae_log_true)
    rmse_log_lst.append(rmse_log_true)
    r2_train_w_lst.append(r2_train_w)
    r2_test_w_lst.append(r2_test_w)
    mae_w_lst.append(mae_w)
    rmse_w_lst.append(rmse_w)
    non_tail_pred_to_y_ratio_lst.append(non_tail_pred_to_y_ratio)
    tail_pred_to_y_ratio_lst.append(tail_pred_to_y_ratio)
    non_tail_sse_lst.append(non_tail_sse)
    tail_sse_lst.append(tail_sse)
    non_tail_share_of_total_sse_lst.append(non_tail_share_of_total_sse)
    tail_share_of_total_sse_lst.append(tail_share_of_total_sse)
    non_tail_under_prediction_rate_lst.append(non_tail_under_prediction_rate)
    tail_under_prediction_rate_lst.append(tail_under_prediction_rate)
    non_tail_bias_lst.append(non_tail_bias)
    tail_bias_lst.append(tail_bias)


In [ ]:

weighted_train_performance_metrics_test_eval = pd.DataFrame({
    "train_sample_weights":[1,2,5,10,15,25],
    "r2_train": r2_train_lst,
    "r2_test": r2_test_lst,
    "mae": mae_lst,
    "rmse": rmse_lst,
    "r2_train_log": r2_train_log_lst,
    "r2_test_log": r2_test_log_lst,
    "mae_log": mae_log_lst,
    "rmse_log": rmse_log_lst,
    "r2_train_w": r2_train_w_lst,
    "r2_test_w": r2_test_w_lst,
    "mae_w": mae_w_lst,
    "rmse_w": rmse_w_lst,
    "non_tail_pred_to_y_ratio": non_tail_pred_to_y_ratio_lst,
    "tail_pred_to_y_ratio": tail_pred_to_y_ratio_lst,
    "non_tail_sse": non_tail_sse_lst,
    "tail_sse": tail_sse_lst,
    "non_tail_share_of_total_sse": non_tail_share_of_total_sse_lst,
    "tail_share_of_total_sse": tail_share_of_total_sse_lst,
    "non_tail_under_prediction_rate": non_tail_under_prediction_rate_lst,
    "tail_under_prediction_rate": tail_under_prediction_rate_lst,
    "non_tail_bias": non_tail_bias_lst,
    "tail_bias": tail_bias_lst
})

weighted_train_performance_metrics_test_eval

The `weighted_train_performance_metrics_test_eval` table shows a very clean tradeoff curve between “average-case performance” and “tail calibration”.

A. As we increase `w_tail`, the model shifts attention toward the tail

You can see this in 4 tail-specific columns that move in the “right” direction as `w_tail` increases:

Tail calibration improves
•	`tail_pred_to_y_ratio`: `0.518` → `0.776` (we go from predicting ~52% of tail mean to ~78% of tail mean)`
•	`tail_bias`: `-427` → `-199` (tail underprediction shrinks by ~$228 on average)

Tail underprediction rate improves
•	tail_under_prediction_rate: `0.904` → `0.705`
•	Still underpredicting most tail rows, but much less extreme.

Tail dominance over `sse` improves
•	`tail_share_of_total_sse`: `0.914` → `0.775`
•	Tail still dominates total squared error, but less so.

This is exactly what tail upweighting is supposed to do.

⸻

B. But average-case performance gets worse when you upweight too hard

Look at the “standard metrics”:
•	`r2_test` peaks at `w=10`:
•	`w=1`: `0.257`
•	`w=2`: `0.276`
•	`w=5`: `0.272`
•	`w=10`: `0.290` (best)
•	`w=15`: `0.254`
•	`w=25`: `0.213`
•	`mae` and `rmse` worsen for large weights:
•	RMSE: `147.97` (`w=1`) → `152.26` (`w=25`)
•	MAE: `28.33` (`w=1`) → `33.97` (`w=25`)

This is the classic tradeoff: once you force the model to chase rare, high-cost points, it starts sacrificing fit for the majority.

⸻

C. Non-tail bias flips sign as weight grows

This is a really important diagnostic: `non_tail_bias` makes it obvious:
•	`non_tail_bias`: `-7.26` → `+1.30`
•	At low weight, you slightly underpredict non-tail on average.
•	By `w=25`, you overpredict non-tail on average.

This also matches:
•	`non_tail_pred_to_y_ratio`: `0.917` → `1.015`

So the model is “lifting” predictions overall to reduce tail underprediction, and that causes slight overprediction for the bulk.

⸻

D. The “weighted evaluation” metrics are not what we should optimize here

`r2_test_w`, `mae_w`, `rmse_w` will often look ugly when tail points are huge, because once we weight them 10x or 25x, our evaluation function is basically saying:

“I mostly care about the tail, and tail errors are still enormous.”

So it is totally normal that:
•	`rmse_w` explodes as `w` increases.

Those weighted metrics are still useful, but mainly as a “tail pain index”, not as the primary model selection metric.

⸻

2) What’s the best`w_tail` from this run?

It depends on what we want to optimize.

If our goal is best overall predictive accuracy (business average case)

Pick `w=10`.
•	Best `r2_test` (`0.290`)
•	Best `rmse` among weighted runs (`144.64`)
•	Still meaningful tail improvements:
•	`tail_bias`: `-246` (vs `-427` baseline)
•	`tail_pred_to_y_ratio`: `0.723` (vs `0.518`)
•	`tail_share_of_total_sse`: `0.841` (vs `0.914`)

If our goal is “improve tail calibration as much as possible without going insane”

Pick `w=15`.
•	Tail bias improves further (`-216`)
•	Tail ratio improves (`0.756`)
•	But overall test R2 drops (`0.254`)

If our goal is “tail calibration first, accept average-case pain”

Pick `w=25`.
•	Best tail calibration in our grid:
•	tail ratio `0.776`
•	tail bias `-199`
•	tail underprediction rate `0.705`
•	But overall metrics deteriorate noticeably, and non-tail bias flips positive.

⸻

3) Why weighting alone still cannot fix the tail

Even at `w=25`, tail bias is still `-199` and tail SSE is still `77.5%` of total SSE.

That strongly suggests: the model still lacks features that separate the high-cost tail regime, so it can only “lift” predictions globally.

This matches what we discovered earlier: ultra-high cost per service lines are legitimate but extremely rare. Without regime-identifying predictors, the best the model can do is compromise.

⸻

4) Next steps that are most likely to actually move the needle

Step 2. Keep weighting (probably w=10 or w=15), but stop using default XGBoost settings

Right now we are comparing different training weight schemes, but our model is still basically “stock XGB”.

Do a small hyperparameter search with a fixed w_tail (start with 10). Focus on parameters that affect generalization and tail handling:
•	`n_estimators` (try 500–3000)
•	`learning_rate` (0.02–0.1)
•	`max_depth` (3–8)
•	`min_child_weight` (1–20)
•	`subsample`, `colsample_bytree` (0.6–1.0)
•	`reg_alpha`, `reg_lambda` (L1/L2 regularization)

If we do this, we will usually get much better log-space fit and often slightly better dollar performance, even before feature work.

⸻

Step 3. Change the loss to something better for heavy tails

Right now we are effectively doing squared error in log space (because TTR transforms y then we fit `reg:squarederror`).

Try an objective that is more robust to extreme residuals:
•	`objective="reg:pseudohubererror"` (great for “few catastrophic misses”)
•	Or try `objective="reg:gamma"` (only for strictly positive targets, which we have)
•	Or `objective:"reg:tweedie"` (often good for skewed positive outcomes)

These objectives can reduce the incentive to massively underpredict rare huge values.

⸻

Step 4. Add “regime” features that we can use at inference time

***This is the biggest lever.***

We already saw that ultra-high is not explained by comorbidity or risk score signals. That screams “missing categorical granularity or provider-history”.

Most effective additions (and all can be done without leakage if we do them carefully):

A) Provider-history features (lagged, prior-years only)
For each `Rndrng_NPI` (and optionally within `rbcs_cat_subcat`), compute on *TRAIN years only*:
•	prior-year mean of `stdzd_amt_per_service`
•	trailing mean over years
•	trailing percentile rank of provider within category
•	log of provider total services (prior year)

Then merge those into train/test by NPI-year with proper lagging.

This will likely capture “this provider is consistently in the expensive regime”.

B) Interaction features
Tree models learn interactions, but only if the split structure can find them. Sometimes explicit cross features help:
•	`rbcs_cat_subcat` × `provider_type`
•	`rbcs_cat_subcat` × `Place_Of_Srvc`
•	`provider_type` × `Place_Of_Srvc`

With OHE, this can explode dimensionality, so this is where CatBoost becomes attractive.

⸻

Step 5. Try CatBoost instead of OHE for these high-card categoricals

We have high-card categorical fields (especially `rbcs_cat_subcat`). OHE + XGB can work, but CatBoost often wins on exactly this kind of problem because it handles categorical encodings natively and learns smoother category effects.

If we try CatBoost, keep the same evaluation framework we built. Compare tail bias, tail ratio, and tail SSE share directly.

⸻

5) Should we build two models, tail vs non-tail?

Not yet.

Two-model systems create a new hard problem: ***how do we decide tail membership at inference time without using y?***

You would need a classifier that predicts “tail-like” based only on features, and with 1% prevalence it will be brittle.

A better “two-stage” system (if we go there) is:
1.	Stage 1 predicts the expected log cost (our current model).
2.	Stage 2 predicts an uplift factor or residual correction for cases likely to be in high-cost regimes, using provider-history and fine-grain categories.

That is more stable than hard gating into two separate regressors.

⸻

6) Should we remove ultra-high rows?

I would not remove them from the dataset if they are legitimate and we care about predicting them.

But we can do one of these safer alternatives:
•	Use a robust objective (pseudohuber) so one extreme row does not dominate.
•	Winsorize y during training only (cap at p99.9), then evaluate on uncapped. This stabilizes training but keeps evaluation honest.
•	Add provider-history features so the model can actually learn why those rows are extreme.

⸻

My recommended next experiment sequence
1. Pick w_tail = 10 as our default tradeoff point.
2. Tune XGB hyperparameters (small grid) with fixed w=10.
3. Try objective="reg:pseudohubererror" with the tuned-ish setup.
4. Add lagged provider-history features (done correctly with year lagging).
5. Run CatBoost with the same feature set and compare tail metrics.

Let's execute this plan! 



### Tuning the XGBoost model

In [ ]:
is_tail_train = eda_df.loc[X_train.index, "is_top_1pct_stdzd_amt_per_service"].astype(int)
w_tail = 10
sample_w = 1 + (w_tail - 1) * is_tail_train
sample_w = np.asarray(sample_w, dtype=float)
print(len(sample_w), X_train.shape[0])

#### 1. `GridSearchCV` for the weighted XGBoost model (with `objective:"reg:squarederror"`)

In [ ]:
from sklearn.model_selection import GridSearchCV, GroupKFold
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

# Point to the model file
model_path = MODELS / "old_forecast_model_wt10_without_lags.joblib"

# 2. Check/Load or Train/Save
if model_path.exists():
    print(f"Loading {model_path.name} from disk... (Skipping training)")
    old_forecast_model_wt10_without_lags = load(model_path)

else:
    print(f"File not found. Preparing data and training...")

    is_tail_train = eda_df.loc[X_train.index, "is_top_1pct_stdzd_amt_per_service"].astype(int)
    w_tail = 10
    sample_w = 1 + (w_tail - 1) * is_tail_train
    sample_w = np.asarray(sample_w, dtype=float)
    print(len(sample_w), X_train.shape[0])

    xgb_inner_pipe = Pipeline(steps=[
        ("preprocess", preprocess_ohe),
        ("model", XGBRegressor(
            objective="reg:squarederror",
            tree_method="hist",
            n_jobs=1,
            random_state=0
        ))
    ])

    xgb_full_model = TransformedTargetRegressor(
        regressor=xgb_inner_pipe,
        func=np.log1p,
        inverse_func=np.expm1
    )

    param_grid_xgb = {
        "regressor__model__n_estimators": [800, 1600],
        "regressor__model__learning_rate": [0.03, 0.07],
        "regressor__model__max_depth": [3, 5],
        "regressor__model__min_child_weight": [1, 5, 10],
        "regressor__model__subsample": [0.7, 0.9],
        "regressor__model__colsample_bytree": [0.7, 0.9],
        "regressor__model__reg_lambda": [1, 10], # L2 (Ridge) 
        "regressor__model__reg_alpha": [0, 0.1], # L1 (Lasso)
    }

    groups = train_df.loc[X_train.index, "Rndrng_NPI"].to_numpy()
    sample_w = sample_w.to_numpy() if hasattr(sample_w, "to_numpy") else sample_w
    cv = GroupKFold(n_splits=3)

    search = GridSearchCV(
        xgb_full_model,
        param_grid_xgb,
        cv=cv,
        scoring="r2",
        n_jobs=-1,
        verbose=1
    )

    # --- MISSING LINES ADDED HERE ---
    # 1. Run the grid search
    search.fit(X_train, y_train, model__sample_weight=sample_w, groups=groups)
    
    # 2. Extract best estimator
    old_forecast_model_wt10_without_lags = search.best_estimator_
    
    # --------------------------------

    # --- SAVE ---
    # 3. Save the trained model
    dump(old_forecast_model_wt10_without_lags, model_path)
    print(f"Training complete. Model saved to {model_path}")

#### Fit the best model, `old_forecast_model_wt10_without_lags` to `X_train` and `y_train` with `sample_w`

In [ ]:
old_forecast_model_wt10_without_lags.fit(X_train, y_train, model__sample_weight = sample_w)

#### Run the `tail_eval` with the `best_model` 

In [ ]:
# Run diagnostics

# get train and test predictions and clip test predictions 
pred_test = old_forecast_model_wt10_without_lags.predict(X_test)
pred_train = old_forecast_model_wt10_without_lags.predict(X_train)

# get weighted-train unweighted/average-case metrics
r2_train = old_forecast_model_wt10_without_lags.score(X_train, y_train)
r2_test = old_forecast_model_wt10_without_lags.score(X_test, y_test)
mae = mean_absolute_error(y_test, pred_test)
rmse = root_mean_squared_error(y_test, pred_test)

# get weighted-train unweighted/average-case metrics in log space
log_pred_train = old_forecast_model_wt10_without_lags.regressor_.predict(X_train)  # predictions in transformed target space
log_pred_test = old_forecast_model_wt10_without_lags.regressor_.predict(X_test)  # predictions in transformed target space

r2_log_true_train = r2_score(np.log1p(y_train), log_pred_train)
r2_log_true_test = r2_score(np.log1p(y_test), log_pred_test)
mae_log_true = mean_absolute_error(np.log1p(y_test), log_pred_test)
rmse_log_true = root_mean_squared_error(np.log1p(y_test), log_pred_test)

# get weighted-train weighted/eval metrics

# Create weights for the test set (following same logic as train)
is_tail_test = eda_df.loc[X_test.index, "is_top_1pct_stdzd_amt_per_service"].astype(int)
sample_w_test = 1 + (w_tail - 1) * is_tail_test

r2_train_w = r2_score(y_train, pred_train, sample_weight=sample_w)
r2_test_w = r2_score(y_test, pred_test, sample_weight=sample_w_test)
mae_w = mean_absolute_error(y_test, pred_test, sample_weight=sample_w_test)
rmse_w = root_mean_squared_error(y_test, pred_test, sample_weight=sample_w_test)

# Calculate tail evals
test_eval = test_df.copy().join(
eda_df.loc[:, ["is_top_1pct_stdzd_amt_per_service"]],
how="left"
)
test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

non_tail_pred_to_y_ratio = test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"].mean() / test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],target_col].mean()
tail_pred_to_y_ratio = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"].mean() / test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],target_col].mean()

non_tail_sse = test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],"sq_err"].sum()
tail_sse = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],"sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = ((test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"]) < test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],target_col]).mean()
tail_under_prediction_rate = ((test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"]) < test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],target_col]).mean()

non_tail_bias = test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"].mean() - test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],target_col].mean()
tail_bias = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"].mean() - test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],target_col].mean()

weighted_train_performance_metrics_test_eval = pd.DataFrame({
    "train_sample_weights":[10],
    "r2_train": r2_train,
    "r2_test": r2_test,
    "mae": mae,
    "rmse": rmse,
    "r2_train_log": r2_log_true_train,
    "r2_test_log": r2_log_true_test,
    "mae_log": mae_log_true,
    "rmse_log": rmse_log_true,
    "r2_train_w": r2_train_w,
    "r2_test_w": r2_test_w,
    "mae_w": mae_w,
    "rmse_w": rmse_w,
    "non_tail_pred_to_y_ratio": non_tail_pred_to_y_ratio,
    "tail_pred_to_y_ratio": tail_pred_to_y_ratio,
    "non_tail_sse": non_tail_sse,
    "tail_sse": tail_sse,
    "non_tail_share_of_total_sse": non_tail_share_of_total_sse,
    "tail_share_of_total_sse": tail_share_of_total_sse,
    "non_tail_under_prediction_rate": non_tail_under_prediction_rate,
    "tail_under_prediction_rate": tail_under_prediction_rate,
    "non_tail_bias": non_tail_bias,
    "tail_bias": tail_bias
})

weighted_train_performance_metrics_test_eval


**Compare tuned vs untuned at w_tail = 10**

Let’s compare your tuned row to the earlier default hyperparameter row for w_tail=10 (row index 3 in your for-loop table).

A. Overall unweighted test performance (average-case)

Default (w=10):
- Test R² ($): 0.2900
- MAE ($): 30.832
- RMSE ($): 144.642

Tuned (w=10):
- Test R² ($): 0.2866
- MAE ($): 29.375
- RMSE ($): 144.985

Interpretation:
- Dollar R² is basically the same (tiny worse).
- MAE improved a bit.
- RMSE basically unchanged (tiny worse).
- Net: tuning didn’t meaningfully improve global generalization, but it didn’t break it either.

B. Log-space generalization (relative-error view)

Default (w=10):
- Test R² (log): 0.7771
- MAE (log): 0.3810
- RMSE (log): 0.5847

Tuned (w=10):
- Test R² (log): 0.7923
- MAE (log): 0.3683
- RMSE (log): 0.5644

Interpretation:
- This is a meaningful improvement. In the space the model is trained in, it is fitting better and generalizing better.

C. Weighted evaluation (tail-focused objective)

Default (w=10):
- Test R² weighted ($): 0.2287
- MAE weighted ($): 49.034
- RMSE weighted ($): 407.767

Tuned (w=10):
- Test R² weighted ($): 0.2099
- MAE weighted ($): 47.426
- RMSE weighted ($): 412.703

Interpretation:
- Slightly better weighted MAE.
- Slightly worse weighted R² and RMSE.
- This usually means you reduced typical tail error a little but did not reduce the largest explosions (RMSE and R² are dominated by those).

⸻

**Tail calibration metrics. This is where the tune helped**

These are the most actionable improvements.

Tail pred_to_y_ratio (calibration)
- Default w=10: 0.7227
- Tuned w=10: 0.7217

Basically identical (good. You didn’t lose calibration).

Tail underprediction rate
- Default w=10: 0.7378
- Tuned w=10: 0.7639

This is actually worse (more underprediction), but not crazy.

Tail SSE share
- Default w=10: 0.8415
- Tuned w=10: 0.8601

Also worse (tail dominates SSE a bit more).

Tail bias (mean pred minus mean actual)
- Default w=10: -245.60
- Tuned w=10: -246.48

Essentially unchanged.

Non-tail bias
- Default w=10: -2.21
- Tuned w=10: -2.76

Slightly worse but still tiny relative to the mean.

Interpretation:
- The tuning helped log-space fit, but did not materially improve the tail failure mode in dollars. The biggest tail misses are still dominating.

That is consistent with what you already observed. A tiny number of ultra-high cost-per-service points create massive squared errors. Hyperparameter tuning with a squared-loss objective often improves the “bulk” first, and struggles to fix a tiny extreme pocket without either (a) more signal features, (b) a different loss emphasis, or (c) structural modeling changes.


## Revisiting the worst offender and its peers (`peer` dataframe) using `rbcs_cat_subcat` (instead of `rbcs_family_desc`)

In [ ]:
row_idx = worst.index[0]

eda_df.loc[row_idx, [
    "Rndrng_NPI","Year","rbcs_family_desc","Place_Of_Srvc","provider_type","state","ruca_bucket",
    "services","benes",
    "stdzd_amt_per_service","stdzd_spend",
    "allowed_amt_per_service","allowed_spend",
    "payment_amt_per_service","payment_spend",
    "submitted_charge_per_service","submitted_spend","rbcs_cat_subcat"
]]

In [ ]:
worst

In [ ]:
npi = worst["Rndrng_NPI"].iloc[0]

eda_df.loc[
    (eda_df["Rndrng_NPI"] == npi) & (eda_df["rbcs_family_desc"] == "Chemotherapeutic Agent"),
    ["Year","services","stdzd_amt_per_service","stdzd_spend","Place_Of_Srvc","provider_type","state", "rbcs_cat_subcat", "ruca_bucket"]
].sort_values("Year")

In [ ]:
eda_df["rbcs_cat_subcat"].value_counts()[:10]

In [ ]:
peer = train_df.loc[
    (train_df["rbcs_cat_subcat"] == "RH") &
    (train_df["provider_type"] == "Radiation Oncology") &
    (train_df["Place_Of_Srvc"] == "O"),
    ["stdzd_amt_per_service","services","state","ruca_bucket"]
]

peer.sort_values("stdzd_amt_per_service", ascending=False)

> We cannot separate ultra-high cost per service lines from their peers at this level of granularity. 

***We must change strategy. See the findings in the EDA notebook at the bottom of the "Data by Provider and Service" section***

> Key insight from EDA notebook: The “ultra-high” tail is not some mysterious provider behavior at the RBCS-family level. It is mostly “this provider billed a very specific HCPCS (or a NOC HCPCS) that is inherently expensive”. If you do not include HCPCS-level signal (or a proxy for it), the model is forced to average over fundamentally different things. That is why it keeps regressing those cases toward the mean.

In [ ]:
eda_hcpcs_df = pd.read_parquet(DATA/"eda_dataset_hcpcs.parquet")

In [ ]:
eda_hcpcs_df.columns

## Streamlining generation of `train_df`, `test_df`, `X_train`, `target_col`, `cat_features`, `num_features`, `feature_cols`, `y_train`, `X_test`, `y_test`, `sample_w_train`, `sample_w_test`, `groups_train`, `cv` (`PredefinedSplit()` or `GroupKFold()`), `search` (`GridSearchCV()`) based on `approach` (either `"forecast"`, or `"new_providers"`)

### 1. Create the `PreparedModelingObjects` Class

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint, uniform

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    GroupShuffleSplit,
    PredefinedSplit,
    RandomizedSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBRegressor


# -----------------------------------------------------------------------------
# Container for everything the notebook typically needs downstream
# -----------------------------------------------------------------------------
@dataclass
class PreparedModelingObjects:
    """
    Convenience bundle returned by prepare_xgb_* helpers.

    Why this exists:
    - Keeps the split dataframes, X/y matrices, sample weights, CV object, and the
      hyperparameter search object together so the notebook stays clean.
    - Makes it easy to log/print the configuration and to reuse objects later.

    Notes:
    - `cv` is either:
        - PredefinedSplit for forecast (explicit year-based validation)
        - GroupKFold for new_providers (provider-disjoint validation)
    - `search` is ready to call `.fit(...)`.
    """

    # High-level scenario
    approach: str
    target_col: str

    # Feature configuration actually used (after filtering to existing columns)
    cat_features: List[str]
    num_features: List[str]
    feature_cols: List[str]

    # The split dataframes used to build X/y
    train_df: pd.DataFrame
    test_df: pd.DataFrame

    # Matrices/vectors for modeling
    X_train: pd.DataFrame
    y_train: pd.Series
    X_test: pd.DataFrame
    y_test: pd.Series

    # Sample weights (computed within each split)
    sample_w_train: np.ndarray
    sample_w_test: np.ndarray

    # Group vector used only for group CV (new_providers). None otherwise.
    groups_train: Optional[np.ndarray]

    # CV strategy and hyperparameter search object
    cv: Any
    search: Union[GridSearchCV, RandomizedSearchCV]

### 2. Create the `prepare_xgb_gridsearch` function

In [ ]:
def prepare_xgb_gridsearch(
    df: pd.DataFrame,
    approach: str,
    target_col: str = "avg_mdcr_stdzd_amt",
    *,
    # Tail weighting (emphasize “tail” rows during training/evaluation)
    w_tail: int = 10,
    tail_flag_col: str = "is_top_1pct_avg_mdcr_stdzd_amt",
    # Forecast-only: include or exclude 2020 rows in the final fit dataset
    incl_2020_in_final_fit: bool = True,
    # Splits
    test_year: int = 2023,
    train_years: Tuple[int, ...] = (2020, 2021, 2022),
    group_col: str = "Rndrng_NPI",
    test_size: float = 0.20,
    random_state: int = 0,
    n_splits: int = 3,
    # Features
    include_lags: bool = True,
    # Model search
    param_grid: Optional[Dict[str, List[Any]]] = None,
    scoring: str = "r2",
    n_jobs: int = -1,
    verbose: int = 1,
    # if target is log_delta_cost, do not use TTR
    use_ttr: bool = True,
    # XGB defaults
    xgb_fixed_params: Optional[Dict[str, Any]] = None,
) -> PreparedModelingObjects:
    """
    Prepare XGBoost + preprocessing + TransformedTargetRegressor, and return a
    leakage-aware GridSearchCV plus the exact split data used.

    Supported approaches
    --------------------
    1) approach="forecast"
       - Final test set is `test_year` (default 2023).
       - Inner CV is a PredefinedSplit that always validates on the last training year (default 2022).
       - Training years are `train_years` (default 2020-2022).
       - If `incl_2020_in_final_fit=True`, 2020 rows are included in X_train/y_train for final fitting,
         but CV validation remains 2022 and “always-train” includes 2020 and 2021.
       - If `incl_2020_in_final_fit=False`, X_train/y_train used by search contain only 2021-2022.

       Practical intent:
       - CV checks: “learn on 2020/2021 and validate on 2022” (or “learn on 2021 and validate on 2022”)
         while keeping the final holdout year 2023 untouched.

    2) approach="new_providers"
       - Group holdout by `group_col` (default Rndrng_NPI): providers in test never appear in train.
       - Inner CV is GroupKFold on the training portion.

    Returns
    -------
    PreparedModelingObjects
        Bundle containing split dataframes, X/y, sample weights, cv, and a GridSearchCV object.

    Notes
    -----
    - Uses TransformedTargetRegressor with log1p/expm1 by default (stabilizes heavy-tailed targets).
    - Numeric features: median impute + missing indicators.
    - Lag numeric features (if present): constant=0 impute + missing indicators.
    - Categorical features: most_frequent impute + one-hot encoding.
    - Sample weights: tail rows get weight `w_tail`, others weight 1.0.
    """

    # -------------------------------------------------------------------------
    # Validation
    # -------------------------------------------------------------------------
    if approach not in {"forecast", "new_providers"}:
        raise ValueError("approach must be one of: {'forecast', 'new_providers'}")

    if target_col not in df.columns:
        raise ValueError(f"target_col '{target_col}' not found in df.columns")

    # -------------------------------------------------------------------------
    # 1) Define splits and CV strategy
    # -------------------------------------------------------------------------
    if approach == "forecast":
        # Final holdout (never used in CV or fitting inside search)
        test_df = df[df["Year"] == test_year].copy()

        # All training years available to the forecast workflow
        train_df_all = df[df["Year"].isin(train_years)].copy()

        years_sorted = sorted(set(train_years))
        if len(years_sorted) < 3:
            raise ValueError(
                "forecast expects 3 training years in train_years (e.g., (2020, 2021, 2022)) "
                "so we can do CV train=2021, val=2022 with lags available."
            )

        # The “validation year” is always the last year in train_years.
        # The year before it is the main “train year” for the CV story.
        train_year = years_sorted[-2]  # typically 2021
        val_year = years_sorted[-1]    # typically 2022

        # CV is conceptually “validate on 2022”.
        # We keep the CV design stable even if we include 2020 rows in the final fit.
        train_df_cv = train_df_all[train_df_all["Year"].isin([train_year, val_year])].copy()

        # The dataset actually used by GridSearchCV for fitting its estimator:
        # - if True: include 2020-2022 rows (more training signal)
        # - if False: use only 2021-2022 rows (closer to original design)
        train_df = train_df_all.copy() if incl_2020_in_final_fit else train_df_cv.copy()

        # PredefinedSplit expects an array aligned to train_df rows.
        # 0 denotes the validation fold (here: 2022 rows).
        # -1 denotes “always in train” (here: 2020/2021 rows, and also 2021-only mode).
        fold = train_df["Year"].map({val_year: 0}).fillna(-1).astype(int).to_numpy()
        cv = PredefinedSplit(test_fold=fold)

        groups_train = None

    else:
        # Group holdout so test providers never appear in training
        groups_all = df[group_col].to_numpy()
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        train_idx, test_idx = next(gss.split(df, groups=groups_all))

        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        groups_train = train_df[group_col].to_numpy()
        cv = GroupKFold(n_splits=n_splits)

    # -------------------------------------------------------------------------
    # 2) Define feature lists (then filter to only existing columns)
    # -------------------------------------------------------------------------
    # Categorical features to one-hot encode
    cat_features = [
        "HCPCS_Cd",
        "Place_Of_Srvc",
        "provider_type",
        "state",
        "ruca_bucket",
        "rbcs_family_desc",
        "hcpcs_drug_ind",
        "is_core_scope",
    ]

    # Numeric features (plus optional lag features)
    num_features = [
        "services",
        "benes",
        "bene_day_services",
        "years_since_enumeration",
        "bene_avg_risk_score",
        "p_cancer6",
        "p_diabetes",
        "p_ckd",
        "p_copd",
        "p_htn",
    ]

    if include_lags:
        num_features += ["lag1_avg_amt", "lag1_tot_srvcs", "lag1_spend"]

    # Safety: allow experimentation across datasets without breaking if some columns are absent
    cat_features = [c for c in cat_features if c in train_df.columns]
    num_features = [c for c in num_features if c in train_df.columns]

    # Treat lag variables differently (common choice: fill missing lag with 0)
    lag_candidates = ["lag1_avg_amt", "lag1_tot_srvcs", "lag1_spend"]
    lag_features = [c for c in lag_candidates if c in num_features]
    base_num_features = [c for c in num_features if c not in lag_features]

    feature_cols = cat_features + num_features

    # -------------------------------------------------------------------------
    # 3) Build X/y (train and test)
    # -------------------------------------------------------------------------
    X_train = train_df[feature_cols].copy()
    y_train = train_df[target_col].astype(float)

    X_test = test_df[feature_cols].copy()
    y_test = test_df[target_col].astype(float)

    # -------------------------------------------------------------------------
    # 4) Sample weights (computed strictly within each split)
    # -------------------------------------------------------------------------
    if tail_flag_col not in train_df.columns:
        raise ValueError(f"tail_flag_col '{tail_flag_col}' not found in df.columns")

    is_tail_train = train_df[tail_flag_col].astype(int).to_numpy()
    is_tail_test = test_df[tail_flag_col].astype(int).to_numpy()

    # Non-tail => 1.0, tail => w_tail
    sample_w_train = (1 + (w_tail - 1) * is_tail_train).astype(float)
    sample_w_test = (1 + (w_tail - 1) * is_tail_test).astype(float)

    # -------------------------------------------------------------------------
    # 5) Preprocessing + model + target transform
    # -------------------------------------------------------------------------
    # Categorical pipeline: impute then one-hot
    cat_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
        ]
    )

    # Numeric pipeline: median impute + missing flags
    num_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ]
    )

    # Lag pipeline: treat missing lag as 0 (plus missing flags)
    lag_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0.0, add_indicator=True)),
        ]
    )

    transformers = [
        ("cat", cat_pipe, cat_features),
        ("num", num_pipe, base_num_features),
    ]
    if lag_features:
        transformers.append(("lag", lag_pipe, lag_features))

    preprocess_ohe = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0.3,
    )

    # XGBoost defaults: keep model single-threaded because CV/search is parallelized
    if xgb_fixed_params is None:
        xgb_fixed_params = dict(
            objective="reg:squarederror",
            tree_method="hist",
            n_jobs=1,
            random_state=random_state,
        )

    xgb_inner_pipe = Pipeline(
        steps=[
            ("preprocess", preprocess_ohe),
            ("model", XGBRegressor(**xgb_fixed_params)),
        ]
    )

    if use_ttr:
        # Transform target: model learns in log-space, predictions are returned in original scale
        xgb_full_model = TransformedTargetRegressor(
            regressor=xgb_inner_pipe,
            func=np.log1p,
            inverse_func=np.expm1,
        )
    else:
        xgb_full_model = xgb_inner_pipe # not TTR

    # -------------------------------------------------------------------------
    # 6) Parameter grid
    # -------------------------------------------------------------------------
    if use_ttr:
        if param_grid is None:
            param_grid = {
                "regressor__model__n_estimators": [800, 1600],
                "regressor__model__learning_rate": [0.03, 0.07],
                "regressor__model__max_depth": [3, 5],
                "regressor__model__min_child_weight": [1, 5, 10],
                "regressor__model__subsample": [0.7, 0.9],
                "regressor__model__colsample_bytree": [0.7, 0.9],
                "regressor__model__reg_lambda": [1, 10],
                "regressor__model__reg_alpha": [0, 0.1],
            }
    else:
        if param_grid is None:
            param_grid = {
                "model__n_estimators": [800, 1600],
                "model__learning_rate": [0.03, 0.07],
                "model__max_depth": [3, 5],
                "model__min_child_weight": [1, 5, 10],
                "model__subsample": [0.7, 0.9],
                "model__colsample_bytree": [0.7, 0.9],
                "model__reg_lambda": [1, 10],
                "model__reg_alpha": [0, 0.1],
            }

    # -------------------------------------------------------------------------
    # 7) GridSearchCV (ready to fit)
    # -------------------------------------------------------------------------
    search = GridSearchCV(
        estimator=xgb_full_model,
        param_grid=param_grid,
        cv=cv,
        scoring=scoring,
        n_jobs=n_jobs,
        verbose=verbose,
    )

    return PreparedModelingObjects(
        approach=approach,
        target_col=target_col,
        cat_features=cat_features,
        num_features=num_features,
        feature_cols=feature_cols,
        train_df=train_df,
        test_df=test_df,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        sample_w_train=sample_w_train,
        sample_w_test=sample_w_test,
        groups_train=groups_train,
        cv=cv,
        search=search,
    )

### 3. Create the `prepare_xgb_randomizedsearch` function

In [ ]:
def prepare_xgb_randomizedsearch(
    df: pd.DataFrame,
    approach: str,
    target_col: str = "avg_mdcr_stdzd_amt",
    *,
    # Tail weighting (emphasize “tail” rows during training/evaluation)
    w_tail: int = 10,
    tail_flag_col: str = "is_top_1pct_avg_mdcr_stdzd_amt",
    # Forecast-only: include or exclude 2020 rows in the final fit dataset
    incl_2020_in_final_fit: bool = True,
    # Splits
    test_year: int = 2023,
    train_years: Tuple[int, ...] = (2020, 2021, 2022),
    group_col: str = "Rndrng_NPI",
    test_size: float = 0.20,
    random_state: int = 0,
    n_splits: int = 3,
    # Features
    include_lags: bool = True,
    # Model search
    param_distributions: Optional[Dict[str, Any]] = None,
    n_iter: int = 50,
    scoring: str = "r2",
    n_jobs: int = -1,
    verbose: int = 1,
    # if target is log_delta_cost, do not use TTR
    use_ttr: bool = True,
    # XGB defaults
    xgb_fixed_params: Optional[Dict[str, Any]] = None,
) -> PreparedModelingObjects:
    """
    Same as `prepare_xgb_gridsearch`, but uses RandomizedSearchCV instead of GridSearchCV.

    The key behavior you asked for is preserved:
    - Forecast:
        - CV validates on 2022 via PredefinedSplit.
        - `incl_2020_in_final_fit` controls whether 2020 is included in the dataset that the
          hyperparameter search fits on (while still validating only on 2022 rows).
        - Final test set is 2023.
    - New providers:
        - provider-disjoint holdout split
        - GroupKFold for inner CV

    Returns
    -------
    PreparedModelingObjects
        Bundle containing split dataframes, X/y, sample weights, cv, and a RandomizedSearchCV object.
    """

    # -------------------------------------------------------------------------
    # Validation
    # -------------------------------------------------------------------------
    if approach not in {"forecast", "new_providers"}:
        raise ValueError("approach must be one of: {'forecast', 'new_providers'}")

    if target_col not in df.columns:
        raise ValueError(f"target_col '{target_col}' not found in df.columns")

    # -------------------------------------------------------------------------
    # 1) Define splits and CV strategy
    # -------------------------------------------------------------------------
    if approach == "forecast":
        test_df = df[df["Year"] == test_year].copy()
        train_df_all = df[df["Year"].isin(train_years)].copy()

        years_sorted = sorted(set(train_years))
        if len(years_sorted) < 3:
            raise ValueError(
                "forecast expects 3 training years in train_years (e.g., (2020, 2021, 2022)) "
                "so we can do CV train=2021, val=2022 with lags available."
            )

        train_year = years_sorted[-2]  # typically 2021
        val_year = years_sorted[-1]    # typically 2022

        # CV always focuses on 2021-2022, independent of final-fit choice
        train_df_cv = train_df_all[train_df_all["Year"].isin([train_year, val_year])].copy()

        # Dataset used for fitting inside the search
        train_df = train_df_all.copy() if incl_2020_in_final_fit else train_df_cv.copy()

        # Validate only on 2022 rows
        fold = train_df["Year"].map({val_year: 0}).fillna(-1).astype(int).to_numpy()
        cv = PredefinedSplit(test_fold=fold)

        groups_train = None

    else:
        groups_all = df[group_col].to_numpy()
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        train_idx, test_idx = next(gss.split(df, groups=groups_all))

        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        groups_train = train_df[group_col].to_numpy()
        cv = GroupKFold(n_splits=n_splits)

    # -------------------------------------------------------------------------
    # 2) Define feature lists (then filter to only existing columns)
    # -------------------------------------------------------------------------
    cat_features = [
        "HCPCS_Cd",
        "Place_Of_Srvc",
        "provider_type",
        "state",
        "ruca_bucket",
        "rbcs_family_desc",
        "hcpcs_drug_ind",
        "is_core_scope",
    ]

    num_features = [
        "services",
        "benes",
        "bene_day_services",
        "years_since_enumeration",
        "bene_avg_risk_score",
        "p_cancer6",
        "p_diabetes",
        "p_ckd",
        "p_copd",
        "p_htn",
    ]

    if include_lags:
        num_features += ["lag1_avg_amt", "lag1_tot_srvcs", "lag1_spend"]

    cat_features = [c for c in cat_features if c in train_df.columns]
    num_features = [c for c in num_features if c in train_df.columns]

    lag_candidates = ["lag1_avg_amt", "lag1_tot_srvcs", "lag1_spend"]
    lag_features = [c for c in lag_candidates if c in num_features]
    base_num_features = [c for c in num_features if c not in lag_features]

    feature_cols = cat_features + num_features

    # -------------------------------------------------------------------------
    # 3) Build X/y (train and test)
    # -------------------------------------------------------------------------
    X_train = train_df[feature_cols].copy()
    y_train = train_df[target_col].astype(float)

    X_test = test_df[feature_cols].copy()
    y_test = test_df[target_col].astype(float)

    # -------------------------------------------------------------------------
    # 4) Sample weights
    # -------------------------------------------------------------------------
    if tail_flag_col not in train_df.columns:
        raise ValueError(f"tail_flag_col '{tail_flag_col}' not found in df.columns")

    is_tail_train = train_df[tail_flag_col].astype(int).to_numpy()
    is_tail_test = test_df[tail_flag_col].astype(int).to_numpy()

    sample_w_train = (1 + (w_tail - 1) * is_tail_train).astype(float)
    sample_w_test = (1 + (w_tail - 1) * is_tail_test).astype(float)

    # -------------------------------------------------------------------------
    # 5) Preprocessing + model + target transform
    # -------------------------------------------------------------------------
    cat_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
        ]
    )

    num_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ]
    )

    lag_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0.0, add_indicator=True)),
        ]
    )

    transformers = [
        ("cat", cat_pipe, cat_features),
        ("num", num_pipe, base_num_features),
    ]
    if lag_features:
        transformers.append(("lag", lag_pipe, lag_features))

    preprocess_ohe = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0.3,
    )

    if xgb_fixed_params is None:
        xgb_fixed_params = dict(
            objective="reg:squarederror",
            tree_method="hist",
            n_jobs=1,
            random_state=random_state,
        )

    xgb_inner_pipe = Pipeline(
        steps=[
            ("preprocess", preprocess_ohe),
            ("model", XGBRegressor(**xgb_fixed_params)),
        ]
    )
    if use_ttr:
        xgb_full_model = TransformedTargetRegressor(
            regressor=xgb_inner_pipe,
            func=np.log1p,
            inverse_func=np.expm1,
        )
    else:
        xgb_full_model = xgb_inner_pipe # no TTR

    # -------------------------------------------------------------------------
    # 6) Parameter distributions (random search space)
    # -------------------------------------------------------------------------
    if use_ttr:
        if param_distributions is None:
            param_distributions = {
                "regressor__model__n_estimators": randint(800, 1601),
                "regressor__model__max_depth": randint(2, 5),
                "regressor__model__min_child_weight": randint(10, 81),
                "regressor__model__max_delta_step": randint(0, 6),
                "regressor__model__subsample": uniform(0.6, 0.35),
                "regressor__model__colsample_bytree": uniform(0.6, 0.35),
                "regressor__model__gamma": uniform(0.0, 2.0),
                "regressor__model__learning_rate": loguniform(0.01, 0.08),
                "regressor__model__reg_lambda": loguniform(1.0, 300.0),
                "regressor__model__reg_alpha": loguniform(1e-6, 1.0),
                "regressor__model__max_bin": [256, 512],
            }
    else:
        if param_distributions is None:
            param_distributions = {
                "model__n_estimators": randint(800, 1601),
                "model__max_depth": randint(2, 5),
                "model__min_child_weight": randint(10, 81),
                "model__max_delta_step": randint(0, 6),
                "model__subsample": uniform(0.6, 0.35),
                "model__colsample_bytree": uniform(0.6, 0.35),
                "model__gamma": uniform(0.0, 2.0),
                "model__learning_rate": loguniform(0.01, 0.08),
                "model__reg_lambda": loguniform(1.0, 300.0),
                "model__reg_alpha": loguniform(1e-6, 1.0),
                "model__max_bin": [256, 512],
            }

    # -------------------------------------------------------------------------
    # 7) RandomizedSearchCV (ready to fit)
    # -------------------------------------------------------------------------
    search = RandomizedSearchCV(
        estimator=xgb_full_model,
        param_distributions=param_distributions,
        n_iter=n_iter,
        cv=cv,
        scoring=scoring,
        n_jobs=n_jobs,
        verbose=verbose,
        random_state=random_state,
    )

    return PreparedModelingObjects(
        approach=approach,
        target_col=target_col,
        cat_features=cat_features,
        num_features=num_features,
        feature_cols=feature_cols,
        train_df=train_df,
        test_df=test_df,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        sample_w_train=sample_w_train,
        sample_w_test=sample_w_test,
        groups_train=groups_train,
        cv=cv,
        search=search,
    )

## 1. Forecast approach: Training a model that predicts the expected average cost per service for 2023 using 2021 and 2022 using `GridSearchCV()`

In [ ]:
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

# We now point to a package file instead of just the model file
pkg_path = MODELS / "forecast_pkg_wt10_with_lags_no_2020.joblib"

# 2. Check/Load or Train/Save
if pkg_path.exists():
    print(f"Loading {pkg_path.name} from disk... (Skipping data prep and training)")
    # Load the entire package (data + fitted search object)
    forecast_pkg_wt10_with_lags_no_2020 = load(pkg_path)
    
    # Extract the model so downstream code works as expected
    best_forecast_model_wt10_with_lags_no_2020 = forecast_pkg_wt10_with_lags_no_2020.search.best_estimator_

else:
    print(f"File not found. Preparing data and training...")

    # --- PREPARE DATA ---
    forecast_pkg_wt10_with_lags_no_2020 = prepare_xgb_gridsearch(
        eda_hcpcs_df,
        approach="forecast",
        target_col="avg_mdcr_stdzd_amt",
        w_tail=10,
        include_lags=True,
        incl_2020_in_final_fit=False
    )

    # --- TRAIN ---
    if forecast_pkg_wt10_with_lags_no_2020.groups_train is None:
        forecast_pkg_wt10_with_lags_no_2020.search.fit(
            forecast_pkg_wt10_with_lags_no_2020.X_train,
            forecast_pkg_wt10_with_lags_no_2020.y_train,
            model__sample_weight=forecast_pkg_wt10_with_lags_no_2020.sample_w_train,
        )
    else:
        forecast_pkg_wt10_with_lags_no_2020.search.fit(
            forecast_pkg_wt10_with_lags_no_2020.X_train,
            forecast_pkg_wt10_with_lags_no_2020.y_train,
            groups=forecast_pkg_wt10_with_lags_no_2020.groups_train,
            model__sample_weight=forecast_pkg_wt10_with_lags_no_2020.sample_w_train,
        )
        
    best_forecast_model_wt10_with_lags_no_2020 = forecast_pkg_wt10_with_lags_no_2020.search.best_estimator_

    # --- SAVE ---
    # 3. Save the ENTIRE package
    dump(forecast_pkg_wt10_with_lags_no_2020, pkg_path)
    print(f"Training complete. Package and model saved to {pkg_path}")

#### Model diagnostics (weighted training including lag variables)

In [ ]:
# Run diagnostics

# get train and test predictions and clip test predictions
pred_test = best_forecast_model_wt10_with_lags_no_2020.predict(forecast_pkg_wt10_with_lags_no_2020.X_test)
pred_train = best_forecast_model_wt10_with_lags_no_2020.predict(forecast_pkg_wt10_with_lags_no_2020.X_train)

# get weighted-train unweighted/average-case metrics
r2_train = best_forecast_model_wt10_with_lags_no_2020.score(forecast_pkg_wt10_with_lags_no_2020.X_train,forecast_pkg_wt10_with_lags_no_2020.y_train)
r2_test = best_forecast_model_wt10_with_lags_no_2020.score(forecast_pkg_wt10_with_lags_no_2020.X_test, forecast_pkg_wt10_with_lags_no_2020.y_test)
mae = mean_absolute_error(forecast_pkg_wt10_with_lags_no_2020.y_test, pred_test)
rmse = root_mean_squared_error(forecast_pkg_wt10_with_lags_no_2020.y_test, pred_test)

# get weighted-train unweighted/average-case metrics in log space
log_pred_train = best_forecast_model_wt10_with_lags_no_2020.regressor_.predict(forecast_pkg_wt10_with_lags_no_2020.X_train)  # predictions in transformed target space
log_pred_test = best_forecast_model_wt10_with_lags_no_2020.regressor_.predict(forecast_pkg_wt10_with_lags_no_2020.X_test)  # predictions in transformed target space

r2_log_true_train = r2_score(np.log1p(forecast_pkg_wt10_with_lags_no_2020.y_train), log_pred_train)
r2_log_true_test = r2_score(np.log1p(forecast_pkg_wt10_with_lags_no_2020.y_test), log_pred_test)
mae_log_true = mean_absolute_error(np.log1p(forecast_pkg_wt10_with_lags_no_2020.y_test), log_pred_test)
rmse_log_true = root_mean_squared_error(np.log1p(forecast_pkg_wt10_with_lags_no_2020.y_test), log_pred_test)

# get weighted-train weighted/eval metrics

r2_train_w = r2_score(forecast_pkg_wt10_with_lags_no_2020.y_train, pred_train, sample_weight=forecast_pkg_wt10_with_lags_no_2020.sample_w_train)
r2_test_w = r2_score(forecast_pkg_wt10_with_lags_no_2020.y_test, pred_test, sample_weight=forecast_pkg_wt10_with_lags_no_2020.sample_w_test)
mae_w = mean_absolute_error(forecast_pkg_wt10_with_lags_no_2020.y_test, pred_test, sample_weight=forecast_pkg_wt10_with_lags_no_2020.sample_w_test)
rmse_w = root_mean_squared_error(forecast_pkg_wt10_with_lags_no_2020.y_test, pred_test, sample_weight=forecast_pkg_wt10_with_lags_no_2020.sample_w_test)

# Calculate tail evals
target_col = forecast_pkg_wt10_with_lags_no_2020.target_col
test_eval = forecast_pkg_wt10_with_lags_no_2020.test_df.copy()

test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

non_tail_pred_to_y_ratio = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() / test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()
tail_pred_to_y_ratio = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() / test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()

non_tail_sse = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"sq_err"].sum()
tail_sse = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = ((test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"]) < test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col]).mean()
tail_under_prediction_rate = ((test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"]) < test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col]).mean()

non_tail_bias = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() - test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()
tail_bias = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() - test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()

forecast_weight10_lag_train_performance_metrics_test_eval = pd.DataFrame({
    "train_sample_weights":[10],
    "r2_train": r2_train,
    "r2_test": r2_test,
    "mae": mae,
    "rmse": rmse,
    "r2_train_log": r2_log_true_train,
    "r2_test_log": r2_log_true_test,
    "mae_log": mae_log_true,
    "rmse_log": rmse_log_true,
    "r2_train_w": r2_train_w,
    "r2_test_w": r2_test_w,
    "mae_w": mae_w,
    "rmse_w": rmse_w,
    "non_tail_pred_to_y_ratio": non_tail_pred_to_y_ratio,
    "tail_pred_to_y_ratio": tail_pred_to_y_ratio,
    "non_tail_sse": non_tail_sse,
    "tail_sse": tail_sse,
    "non_tail_share_of_total_sse": non_tail_share_of_total_sse,
    "tail_share_of_total_sse": tail_share_of_total_sse,
    "non_tail_under_prediction_rate": non_tail_under_prediction_rate,
    "tail_under_prediction_rate": tail_under_prediction_rate,
    "non_tail_bias": non_tail_bias,
    "tail_bias": tail_bias
})

forecast_weight10_lag_train_performance_metrics_test_eval

## Finding baseline: Predict `2023` `avg_mdcr_stdzd_amt` using either "lag only" values or "lag + backoff mean"

#### **Baseline 1: Lag-only (no training)**

For each 2023 row:
- Prediction = `lag1_avg_amt`
    - This only works where `lag1_avg_amt` is not null.
    - Rows with `null` lag are “can’t predict” for this baseline (or we can drop them when scoring this baseline).

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

target_col = "avg_mdcr_stdzd_amt"
tail_flag = "is_top_1pct_avg_mdcr_stdzd_amt"

# Updated to use the correctly named package variable
test_df = forecast_pkg_wt10_with_lags_no_2020.test_df.copy()   # should already be 2023 only
y_true = test_df[target_col]

# Lag-only predictions
pred_lag = test_df["lag1_avg_amt"]

# Score only rows where lag exists
mask = pred_lag.notna()
y_true_lag = y_true[mask]
pred_lag_valid = pred_lag[mask]

r2_lag = r2_score(y_true_lag, pred_lag_valid)
mae_lag = mean_absolute_error(y_true_lag, pred_lag_valid)
rmse_lag = root_mean_squared_error(y_true_lag, pred_lag_valid)

print("Lag-only baseline (lag present rows only)")
print("rows:", mask.sum(), "of", len(test_df))
print("r2:", r2_lag, "mae:", mae_lag, "rmse:", rmse_lag)

#### **Baseline 2: Lag + backoff mean (uses training data only)**
- Here we fill missing lags with a group mean computed from train years only (2021–2022).
    - Mean over (`provider_type`, `HCPCS_Cd`, `Place_Of_Srvc`)

In [ ]:
# Added this line to ensure test_df is pulled from the correctly named package
test_df = forecast_pkg_wt10_with_lags_no_2020.test_df.copy()  
train_df = forecast_pkg_wt10_with_lags_no_2020.train_df.copy()  # should be 2021-2022 only

group_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"]
train_means = (
    train_df.groupby(group_cols)[target_col]
    .mean()
    .rename("mean_train_grp")
    .reset_index()
)

test_with_means = test_df.merge(train_means, on=group_cols, how="left")

# Optional fallback if group mean missing (new combo in 2023): mean by HCPCS only
hcpcs_means = (
    train_df.groupby(["HCPCS_Cd"])[target_col]
    .mean()
    .rename("mean_train_hcpcs")
    .reset_index()
)

test_with_means = test_with_means.merge(hcpcs_means, on="HCPCS_Cd", how="left")

global_mean = train_df[target_col].mean()

# Build final baseline pred:
# 1) use lag if present
# 2) else group mean (provider_type, HCPCS, POS)
# 3) else HCPCS mean
# 4) else global mean
pred_backoff = test_with_means["lag1_avg_amt"].copy()
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_grp"])
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_hcpcs"])
pred_backoff = pred_backoff.fillna(global_mean)

y_true = test_with_means[target_col]

r2_b = r2_score(y_true, pred_backoff)
mae_b = mean_absolute_error(y_true, pred_backoff)
rmse_b = root_mean_squared_error(y_true, pred_backoff)

print("Lag + backoff baseline (all rows)")
print("r2:", r2_b, "mae:", mae_b, "rmse:", rmse_b)

#### **Compare against the XGB model on the same metrics**

We already have `pred_test` from XGB. Now compute the same summary for:

- XGB predictions
- lag-only (lag-present rows)
- lag+backoff (all rows)

**1. Tail-only metrics**

In [ ]:
def tail_metrics(df, y_col, pred, tail_flag_col):
    is_tail = df[tail_flag_col].astype(bool)
    out = {}
    for name, m in [("non_tail", ~is_tail), ("tail", is_tail)]:
        yy = df.loc[m, y_col]
        pp = pred.loc[m] if isinstance(pred, pd.Series) else pd.Series(pred, index=df.index).loc[m]
        out[f"{name}_mae"] = mean_absolute_error(yy, pp)
        out[f"{name}_rmse"] = root_mean_squared_error(yy, pp)
        out[f"{name}_pred_to_y_ratio"] = pp.mean() / yy.mean()
        out[f"{name}_bias"] = pp.mean() - yy.mean()
    return out

# XGB
# Updated the index reference to use forecast_pkg_wt10_with_lags
pred_xgb = pd.Series(pred_test, index=forecast_pkg_wt10_with_lags_no_2020.X_test.index)
xgb_tail = tail_metrics(test_df, target_col, pred_xgb, tail_flag)

# Backoff baseline
pred_backoff = pd.Series(pred_backoff.values, index=test_df.index)
b_tail = tail_metrics(test_df, target_col, pred_backoff, tail_flag)

print("XGB tail metrics:", xgb_tail)
print("Baseline tail metrics:", b_tail)

**Results we got:**

- XGB
    - non-tail MAE: 8.31
    - non-tail RMSE: 53.79
    - tail MAE: 171.12
    - tail RMSE: 1219.92
    - tail bias: +51.74

- Baseline (lag + backoff)
    - non-tail MAE: 5.28
    - non-tail RMSE: 33.35
    - tail MAE: 78.95
    - tail RMSE: 271.61
    - tail bias: 11.76

**Interpretation**
- Our simple baseline is dramatically better than XGB, especially in the tail.
- That means, in our current setup, the model is not learning something useful beyond “use the past.”
- Instead it is sometimes producing wild predictions that explode errors.

**2. Worst-case rows**

In [ ]:
# Explicitly pull the test dataframe from your updated package
worst = forecast_pkg_wt10_with_lags_no_2020.test_df.copy()

worst["pred"] = pred_xgb  # using the pred_xgb calculated in the previous step
worst["sq_err"] = (worst[target_col] - worst["pred"])**2

cols = [
    "Rndrng_NPI", "Year", "HCPCS_Cd", "Place_Of_Srvc", "provider_type", "state",
    "services", "lag1_avg_amt", target_col, "pred", "sq_err", tail_flag
]

# Display the 20 rows with the largest squared errors
worst.sort_values("sq_err", ascending=False).head(20)[cols]

**What the worst rows show:**

- The top 5 are all:
    - `HCPCS_Cd` = `J3590`
    - `CA` (California)
    - `Medical Oncology` or `Hematology-Oncology`
    - True value around 3600
    - Lag value around 3633
    - XGB predicts around 28,000 to 31,000

*That is catastrophic overprediction.*

And it explains why our tail RMSE is enormous.

***What this strongly suggests (directly)***
- Our model is sometimes taking a code that is “normally $3,600” and predicting it as if it belongs to the “$30,000” regime.
- The baseline, using lag, is basically perfect here (lag is ~3633). So any sane model should stay near 3633 unless there is a strong reason not to. Yet XGB is jumping to 30k.

**That implies one of these is happening:**
1. The model is not actually using lag features effectively, or lag is getting lost in preprocessing.
2. Some other features are pushing it hard upward (like `services`, `stdzd_spend`, provider context) and the model is overreacting.
3. Train/test regime mismatch for what we are trying to predict, so it learns relationships that do not hold in 2023.
4. Feature leakage-like behavior inside the model inputs (not the classic kind from the future, but “proxy leakage” where a feature almost encodes the target differently in train vs test).
5. Objective mismatch with our evaluation goal, for example training in log space but evaluating in dollar space can create weird incentives unless handled carefully.

Given our baseline performance, the strongest immediate hypothesis is (1) or (2).

#### **Let's quantify “insane jumps” vs lag**

Thresholds like “more than 5x or more than $1000” can be useful. Let's do both.

**1. Run on rows where lag exists: percentage of cases insane jumps happen**

Let's also compute how often those insane jumps are actually helpful

In [ ]:
# Explicitly pull the test dataframe from your updated package
df = forecast_pkg_wt10_with_lags_no_2020.test_df.copy()

mask = df["lag1_avg_amt"].notna()

df = df.loc[mask].copy()
df["abs_jump"] = (pred_xgb.loc[df.index] - df["lag1_avg_amt"]).abs()
df["ratio_jump"] = df["abs_jump"] / (df["lag1_avg_amt"].abs() + 1e-9)

jump_1000 = (df["abs_jump"] > 1000).mean()
jump_5x = (df["ratio_jump"] > 5).mean()

print("Share with abs jump > 1000:", jump_1000)
print("Share with ratio jump > 5x:", jump_5x)

**2. Let's also quantify the MAE: XGB vs LAG (in general and where jumps are huge):**

In [ ]:
df["abs_err_xgb"] = (df[target_col] - pred_xgb.loc[df.index]).abs()
df["abs_err_lag"] = (df[target_col] - df["lag1_avg_amt"]).abs()

print("MAE XGB:", df["abs_err_xgb"].mean())
print("MAE LAG:", df["abs_err_lag"].mean())

# On insane-jump rows only
insane = (df["abs_jump"] > 1000) | (df["ratio_jump"] > 5)
print("MAE XGB insane:", df.loc[insane, "abs_err_xgb"].mean())
print("MAE LAG insane:", df.loc[insane, "abs_err_lag"].mean())

#### **Why J3590 specifically is showing up**

From our table, J3590 appears in California, oncology provider types, POS O, and is flagged as top 1 percent. So it is a tail-associated code in training.

Our model likely learned a branch like:
“If HCPCS is J3590 and provider_type is oncology and state is CA and POS is O, predict huge.”

That branch might be correct for some training cases, but it is clearly wrong for these particular NPIs because lag is telling you the truth: they were around $3600 and stayed around $3600.

So the fix is to force the model to respect lag, either by delta modeling or by explicit guardrails.

In [ ]:
forecast_pkg_wt10_with_lags_no_2020.cat_features

In [ ]:
forecast_pkg_wt10_with_lags_no_2020.num_features

In [ ]:
forecast_pkg_wt10_with_lags_no_2020.search.best_params_

## 1. Changing the modeling target to `log_delta_cost` (difference between `avg_mdcr_stdzd_amt` and `lag1_avg_amt`) and using `prepare_xgb_gridsearch` for modeling package

#### Adding `log_delta_cost` to the `eda_hcpcs_df` dataframe

In [ ]:
# Keep full data intact
eda_hcpcs_full_df = eda_hcpcs_df.copy()

# Lag-present mask
mask_lag = (
    eda_hcpcs_full_df["lag1_avg_amt"].notna()
    & (eda_hcpcs_full_df["lag1_avg_amt"] >= 0)
)

# Subset only for residual-model training
eda_hcpcs_lags_present_df = eda_hcpcs_full_df.loc[mask_lag].copy()

# Residual target (log space delta)
eda_hcpcs_lags_present_df["log_delta_cost"] = (
    np.log1p(eda_hcpcs_lags_present_df["avg_mdcr_stdzd_amt"])
    - np.log1p(eda_hcpcs_lags_present_df["lag1_avg_amt"])
)

In [ ]:
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

# We now point to a package file instead of just the model file
pkg_path = MODELS / "forecast_pkg_wt10_with_lags_no_2020_delta_trgt.joblib"

# 2. Check/Load or Train/Save
if pkg_path.exists():
    print(f"Loading {pkg_path.name} from disk... (Skipping data prep and training)")
    # Load the entire package (data + fitted search object)
    forecast_pkg_wt10_with_lags_no_2020_delta_trgt = load(pkg_path)
    
    # Extract the model so downstream code works as expected
    best_forecast_model_wt10_with_lags_no_2020_delta_trgt = forecast_pkg_wt10_with_lags_no_2020_delta_trgt.search.best_estimator_

else:
    print(f"File not found. Preparing data and training...")

    # --- PREPARE DATA ---
    forecast_pkg_wt10_with_lags_no_2020_delta_trgt = prepare_xgb_gridsearch(
        eda_hcpcs_lags_present_df,
        approach="forecast",
        target_col="log_delta_cost",
        w_tail=10,
        include_lags=True,
        incl_2020_in_final_fit=False,
        use_ttr=False
    )

    # --- TRAIN ---
    if forecast_pkg_wt10_with_lags_no_2020_delta_trgt.groups_train is None:
        forecast_pkg_wt10_with_lags_no_2020_delta_trgt.search.fit(
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt.X_train,
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt.y_train,
            model__sample_weight=forecast_pkg_wt10_with_lags_no_2020_delta_trgt.sample_w_train,
        )
    else:
        forecast_pkg_wt10_with_lags_no_2020_delta_trgt.search.fit(
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt.X_train,
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt.y_train,
            groups=forecast_pkg_wt10_with_lags_no_2020_delta_trgt.groups_train,
            model__sample_weight=forecast_pkg_wt10_with_lags_no_2020_delta_trgt.sample_w_train,
        )
        
    best_forecast_model_wt10_with_lags_no_2020_delta_trgt = forecast_pkg_wt10_with_lags_no_2020_delta_trgt.search.best_estimator_

    # --- SAVE ---
    # 3. Save the ENTIRE package
    dump(forecast_pkg_wt10_with_lags_no_2020_delta_trgt, pkg_path)
    print(f"Training complete. Package and model saved to {pkg_path}")

### Hybrid approach for predictions: use model for lag-present rows and use lag + backoff for lag-absent rows

In [ ]:
import numpy as np
import pandas as pd

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"
group_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"]

# 1) Full 2023 test set (all rows, lag and no-lag)
test_df_full_2023 = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"] == 2023].copy()

# 2) Choose which years define the backoff means
# If you want to match the package that trained on 2021-2022 only:
train_years_for_means = [2021, 2022]
# If you intentionally want more history for means, use:
# train_years_for_means = [2020, 2021, 2022]

train_df_for_means = eda_hcpcs_full_df.loc[
    eda_hcpcs_full_df["Year"].isin(train_years_for_means)
].copy()

# 3) Compute backoff mean at (provider_type, HCPCS, POS)
train_means_grp = (
    train_df_for_means.groupby(group_cols)[target_col]
    .mean()
    .rename("mean_train_grp")
    .reset_index()
)

# Optional fallback: HCPCS-only mean
train_means_hcpcs = (
    train_df_for_means.groupby(["HCPCS_Cd"])[target_col]
    .mean()
    .rename("mean_train_hcpcs")
    .reset_index()
)

global_mean = float(train_df_for_means[target_col].mean())

# 4) Attach means to the FULL 2023 test set (preserve original index)
test_with_means = test_df_full_2023.copy()
test_with_means["_orig_idx"] = test_with_means.index

test_with_means = test_with_means.merge(train_means_grp, on=group_cols, how="left")
test_with_means = test_with_means.merge(train_means_hcpcs, on="HCPCS_Cd", how="left")

# Restore original index
test_with_means = test_with_means.set_index("_orig_idx", drop=True)
test_with_means = test_with_means.loc[test_df.index]  # keep same order

# 5) Split lag vs no-lag for residual model
mask_2023_lag = test_with_means["lag1_avg_amt"].notna() & (test_with_means["lag1_avg_amt"] >= 0)
test_2023_lag = test_with_means.loc[mask_2023_lag].copy()
test_2023_nolag = test_with_means.loc[~mask_2023_lag].copy()

# 6) Residual model prediction for lag-present rows
feature_cols = forecast_pkg_wt10_with_lags_no_2020_delta_trgt.feature_cols
X_2023_lag = test_2023_lag[feature_cols]

log_delta_hat = best_forecast_model_wt10_with_lags_no_2020_delta_trgt.predict(X_2023_lag)

pred_level_lag = np.expm1(np.log1p(test_2023_lag["lag1_avg_amt"].values) + log_delta_hat)

# 7) Backoff prediction for no-lag rows (group mean -> HCPCS mean -> global)
pred_level_nolag = (
    test_2023_nolag["mean_train_grp"]
    .fillna(test_2023_nolag["mean_train_hcpcs"])
    .fillna(global_mean)
    .to_numpy()
)

# 8) Combine into a full prediction vector aligned to original 2023 test index
pred_full = pd.Series(index=test_df_full_2023.index, dtype=float)
pred_full.loc[test_2023_lag.index] = pred_level_lag
pred_full.loc[test_2023_nolag.index] = pred_level_nolag

# Rebuild pred_backoff

pred_backoff = test_with_means["lag1_avg_amt"].copy()
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_grp"])
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_hcpcs"])
pred_backoff = pred_backoff.fillna(global_mean)

### Calculate model metrics and tail performance

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Inputs you already have from your hybrid prediction cell:
# - test_df_full_2023  (full 2023 rows)
# - pred_full          (full 2023 predictions aligned to test_df_full_2023.index)
# ------------------------------------------------------------

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"

y_true = test_df_full_2023[target_col].astype(float)
pred_test = pred_full.astype(float)

# Basic sanity check
assert pred_test.index.equals(test_df_full_2023.index)

# ------------------------------------------------------------
# 1) Overall metrics (2023 full)
# ------------------------------------------------------------
r2_test = r2_score(y_true, pred_test)
mae = mean_absolute_error(y_true, pred_test)
rmse = root_mean_squared_error(y_true, pred_test)

# "log space" metrics on LEVELS (not on delta), if you still want them:
r2_log_true_test = r2_score(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))
mae_log_true = mean_absolute_error(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))
rmse_log_true = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))

# ------------------------------------------------------------
# 2) Weighted (tail-emphasized) metrics on 2023 full
#    Use the SAME w_tail you used for weighting (10 here)
# ------------------------------------------------------------
w_tail = 10
is_tail_test = test_df_full_2023[tail_flag_col].astype(int).to_numpy()
sample_w_test = (1 + (w_tail - 1) * is_tail_test).astype(float)

r2_test_w = r2_score(y_true, pred_test, sample_weight=sample_w_test)
mae_w = mean_absolute_error(y_true, pred_test, sample_weight=sample_w_test)
rmse_w = root_mean_squared_error(y_true, pred_test, sample_weight=sample_w_test)

# ------------------------------------------------------------
# 3) Tail analysis table (same style as before)
# ------------------------------------------------------------
test_eval = test_df_full_2023.copy()
test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"] = (test_eval[target_col] - test_eval["pred"]) ** 2

non_tail_pred_to_y_ratio = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_pred_to_y_ratio = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

non_tail_sse = test_eval.loc[~test_eval[tail_flag_col], "sq_err"].sum()
tail_sse = test_eval.loc[test_eval[tail_flag_col], "sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = (
    (test_eval.loc[~test_eval[tail_flag_col], "pred"] < test_eval.loc[~test_eval[tail_flag_col], target_col]).mean()
)
tail_under_prediction_rate = (
    (test_eval.loc[test_eval[tail_flag_col], "pred"] < test_eval.loc[test_eval[tail_flag_col], target_col]).mean()
)

non_tail_bias = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_bias = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

forecast_hybrid_metrics_2023 = pd.DataFrame({
    "w_tail":[w_tail],
    "r2_test": [r2_test],
    "mae": [mae],
    "rmse": [rmse],
    "r2_test_log": [r2_log_true_test],
    "mae_log": [mae_log_true],
    "rmse_log": [rmse_log_true],
    "r2_test_w": [r2_test_w],
    "mae_w": [mae_w],
    "rmse_w": [rmse_w],
    "non_tail_pred_to_y_ratio": [non_tail_pred_to_y_ratio],
    "tail_pred_to_y_ratio": [tail_pred_to_y_ratio],
    "non_tail_sse": [non_tail_sse],
    "tail_sse": [tail_sse],
    "non_tail_share_of_total_sse": [non_tail_share_of_total_sse],
    "tail_share_of_total_sse": [tail_share_of_total_sse],
    "non_tail_under_prediction_rate": [non_tail_under_prediction_rate],
    "tail_under_prediction_rate": [tail_under_prediction_rate],
    "non_tail_bias": [non_tail_bias],
    "tail_bias": [tail_bias],
})

forecast_hybrid_metrics_2023

## 2. Recalculate baseline 2: lag + backoff mean (using train data only) 

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"
target_col = "avg_mdcr_stdzd_amt"
w_tail = 10

eval_df = test_with_means.copy()
pred = pred_backoff.astype(float)          # already aligned now
y_true = eval_df[target_col].astype(float)

eval_df["pred"] = pred
is_tail = eval_df[tail_flag_col].astype(bool)
is_non_tail = ~is_tail

eval_df["abs_err"] = (eval_df[target_col] - eval_df["pred"]).abs()
eval_df["sq_err"] = (eval_df[target_col] - eval_df["pred"]) ** 2

y_true = test_with_means[target_col].astype(float)
pred = pred_backoff.astype(float)

# overall
r2 = r2_score(y_true, pred)
mae = mean_absolute_error(y_true, pred)
rmse = root_mean_squared_error(y_true, pred)

# log-level (optional)
pred_clip = pred.clip(lower=0)
r2_log = r2_score(np.log1p(y_true), np.log1p(pred_clip))
mae_log = mean_absolute_error(np.log1p(y_true), np.log1p(pred_clip))
rmse_log = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_clip))

# weighted
r2_w = r2_score(y_true, pred, sample_weight=sample_w)
mae_w = mean_absolute_error(y_true, pred, sample_weight=sample_w)
rmse_w = root_mean_squared_error(y_true, pred, sample_weight=sample_w)

non_tail_pred_to_y_ratio = eval_df.loc[is_non_tail, "pred"].mean() / eval_df.loc[is_non_tail, target_col].mean()
tail_pred_to_y_ratio = eval_df.loc[is_tail, "pred"].mean() / eval_df.loc[is_tail, target_col].mean()

non_tail_sse = eval_df.loc[is_non_tail, "sq_err"].sum()
tail_sse = eval_df.loc[is_tail, "sq_err"].sum()

non_tail_share = non_tail_sse / (non_tail_sse + tail_sse)
tail_share = tail_sse / (non_tail_sse + tail_sse)

non_tail_upr = (eval_df.loc[is_non_tail, "pred"] < eval_df.loc[is_non_tail, target_col]).mean()
tail_upr = (eval_df.loc[is_tail, "pred"] < eval_df.loc[is_tail, target_col]).mean()

non_tail_bias = eval_df.loc[is_non_tail, "pred"].mean() - eval_df.loc[is_non_tail, target_col].mean()
tail_bias = eval_df.loc[is_tail, "pred"].mean() - eval_df.loc[is_tail, target_col].mean()

baseline_metrics_2023 = pd.DataFrame({
    "w_tail":[w_tail],
    "r2_test":[r2],
    "mae":[mae],
    "rmse":[rmse],
    "r2_test_log":[r2_log],
    "mae_log":[mae_log],
    "rmse_log":[rmse_log],
    "r2_test_w":[r2_w],
    "mae_w":[mae_w],
    "rmse_w":[rmse_w],
    "non_tail_pred_to_y_ratio":[non_tail_pred_to_y_ratio],
    "tail_pred_to_y_ratio":[tail_pred_to_y_ratio],
    "non_tail_sse":[non_tail_sse],
    "tail_sse":[tail_sse],
    "non_tail_share_of_total_sse":[non_tail_share],
    "tail_share_of_total_sse":[tail_share],
    "non_tail_under_prediction_rate":[non_tail_upr],
    "tail_under_prediction_rate":[tail_upr],
    "non_tail_bias":[non_tail_bias],
    "tail_bias":[tail_bias],
})

baseline_metrics_2023

> **Baseline 2 (lag + backoff mean) beats our most sophisticated model!**

## 1. Changing the modeling target to `log_delta_cost` (difference between `avg_mdcr_stdzd_amt` and `lag1_avg_amt`) and using `prepare_xgb_randomizedsearch` for modeling package

In [ ]:
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

# We now point to a package file instead of just the model file
pkg_path = MODELS / "forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.joblib"

# 2. Check/Load or Train/Save
if pkg_path.exists():
    print(f"Loading {pkg_path.name} from disk... (Skipping data prep and training)")
    # Load the entire package (data + fitted search object)
    forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50 = load(pkg_path)
    
    # Extract the model so downstream code works as expected
    best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50 = forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.search.best_estimator_

else:
    print(f"File not found. Preparing data and training...")

    # --- PREPARE DATA ---
    forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50 = prepare_xgb_randomizedsearch(
        eda_hcpcs_lags_present_df,
        approach="forecast",
        target_col="log_delta_cost",
        w_tail=10,
        include_lags=True,
        incl_2020_in_final_fit=False,
        use_ttr=False
    )

    # --- TRAIN ---
    if forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.groups_train is None:
        forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.search.fit(
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.X_train,
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.y_train,
            model__sample_weight=forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.sample_w_train,
        )
    else:
        forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.search.fit(
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.X_train,
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.y_train,
            groups=forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.groups_train,
            model__sample_weight=forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.sample_w_train,
        )
        
    best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50 = forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.search.best_estimator_

    # --- SAVE ---
    # 3. Save the ENTIRE package
    dump(forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50, pkg_path)
    print(f"Training complete. Package and model saved to {pkg_path}")

### Hybrid approach for predictions: use model for lag-present rows and use lag + backoff for lag-absent rows

In [ ]:
import numpy as np
import pandas as pd

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"
group_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"]

# 1) Full 2023 test set (all rows, lag and no-lag)
test_df_full_2023 = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"] == 2023].copy()

# 2) Choose which years define the backoff means
# If you want to match the package that trained on 2021-2022 only:
train_years_for_means = [2021, 2022]
# If you intentionally want more history for means, use:
# train_years_for_means = [2020, 2021, 2022]

train_df_for_means = eda_hcpcs_full_df.loc[
    eda_hcpcs_full_df["Year"].isin(train_years_for_means)
].copy()

# 3) Compute backoff mean at (provider_type, HCPCS, POS)
train_means_grp = (
    train_df_for_means.groupby(group_cols)[target_col]
    .mean()
    .rename("mean_train_grp")
    .reset_index()
)

# Optional fallback: HCPCS-only mean
train_means_hcpcs = (
    train_df_for_means.groupby(["HCPCS_Cd"])[target_col]
    .mean()
    .rename("mean_train_hcpcs")
    .reset_index()
)

global_mean = float(train_df_for_means[target_col].mean())

# 4) Attach means to the FULL 2023 test set (preserve original index)
test_with_means = test_df_full_2023.copy()
test_with_means["_orig_idx"] = test_with_means.index

test_with_means = test_with_means.merge(train_means_grp, on=group_cols, how="left")
test_with_means = test_with_means.merge(train_means_hcpcs, on="HCPCS_Cd", how="left")

# Restore original index
test_with_means = test_with_means.set_index("_orig_idx", drop=True)
test_with_means = test_with_means.loc[test_df.index]  # keep same order

# 5) Split lag vs no-lag for residual model
mask_2023_lag = test_with_means["lag1_avg_amt"].notna() & (test_with_means["lag1_avg_amt"] >= 0)
test_2023_lag = test_with_means.loc[mask_2023_lag].copy()
test_2023_nolag = test_with_means.loc[~mask_2023_lag].copy()

# 6) Residual model prediction for lag-present rows
feature_cols = forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.feature_cols
X_2023_lag = test_2023_lag[feature_cols]

log_delta_hat = best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50.predict(X_2023_lag)

pred_level_lag = np.expm1(np.log1p(test_2023_lag["lag1_avg_amt"].values) + log_delta_hat)

# 7) Backoff prediction for no-lag rows (group mean -> HCPCS mean -> global)
pred_level_nolag = (
    test_2023_nolag["mean_train_grp"]
    .fillna(test_2023_nolag["mean_train_hcpcs"])
    .fillna(global_mean)
    .to_numpy()
)

# 8) Combine into a full prediction vector aligned to original 2023 test index
pred_full = pd.Series(index=test_df_full_2023.index, dtype=float)
pred_full.loc[test_2023_lag.index] = pred_level_lag
pred_full.loc[test_2023_nolag.index] = pred_level_nolag

# Rebuild pred_backoff

pred_backoff = test_with_means["lag1_avg_amt"].copy()
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_grp"])
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_hcpcs"])
pred_backoff = pred_backoff.fillna(global_mean)

### Calculate model metrics and tail performance

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Inputs you already have from your hybrid prediction cell:
# - test_df_full_2023  (full 2023 rows)
# - pred_full          (full 2023 predictions aligned to test_df_full_2023.index)
# ------------------------------------------------------------

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"

y_true = test_df_full_2023[target_col].astype(float)
pred_test = pred_full.astype(float)

# Basic sanity check
assert pred_test.index.equals(test_df_full_2023.index)

# ------------------------------------------------------------
# 1) Overall metrics (2023 full)
# ------------------------------------------------------------
r2_test = r2_score(y_true, pred_test)
mae = mean_absolute_error(y_true, pred_test)
rmse = root_mean_squared_error(y_true, pred_test)

# "log space" metrics on LEVELS (not on delta), if you still want them:
r2_log_true_test = r2_score(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))
mae_log_true = mean_absolute_error(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))
rmse_log_true = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))

# ------------------------------------------------------------
# 2) Weighted (tail-emphasized) metrics on 2023 full
#    Use the SAME w_tail you used for weighting (10 here)
# ------------------------------------------------------------
w_tail = 10
is_tail_test = test_df_full_2023[tail_flag_col].astype(int).to_numpy()
sample_w_test = (1 + (w_tail - 1) * is_tail_test).astype(float)

r2_test_w = r2_score(y_true, pred_test, sample_weight=sample_w_test)
mae_w = mean_absolute_error(y_true, pred_test, sample_weight=sample_w_test)
rmse_w = root_mean_squared_error(y_true, pred_test, sample_weight=sample_w_test)

# ------------------------------------------------------------
# 3) Tail analysis table (same style as before)
# ------------------------------------------------------------
test_eval = test_df_full_2023.copy()
test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"] = (test_eval[target_col] - test_eval["pred"]) ** 2

non_tail_pred_to_y_ratio = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_pred_to_y_ratio = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

non_tail_sse = test_eval.loc[~test_eval[tail_flag_col], "sq_err"].sum()
tail_sse = test_eval.loc[test_eval[tail_flag_col], "sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = (
    (test_eval.loc[~test_eval[tail_flag_col], "pred"] < test_eval.loc[~test_eval[tail_flag_col], target_col]).mean()
)
tail_under_prediction_rate = (
    (test_eval.loc[test_eval[tail_flag_col], "pred"] < test_eval.loc[test_eval[tail_flag_col], target_col]).mean()
)

non_tail_bias = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_bias = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

forecast_hybrid_metrics_2023_rs50 = pd.DataFrame({
    "w_tail":[w_tail],
    "r2_test": [r2_test],
    "mae": [mae],
    "rmse": [rmse],
    "r2_test_log": [r2_log_true_test],
    "mae_log": [mae_log_true],
    "rmse_log": [rmse_log_true],
    "r2_test_w": [r2_test_w],
    "mae_w": [mae_w],
    "rmse_w": [rmse_w],
    "non_tail_pred_to_y_ratio": [non_tail_pred_to_y_ratio],
    "tail_pred_to_y_ratio": [tail_pred_to_y_ratio],
    "non_tail_sse": [non_tail_sse],
    "tail_sse": [tail_sse],
    "non_tail_share_of_total_sse": [non_tail_share_of_total_sse],
    "tail_share_of_total_sse": [tail_share_of_total_sse],
    "non_tail_under_prediction_rate": [non_tail_under_prediction_rate],
    "tail_under_prediction_rate": [tail_under_prediction_rate],
    "non_tail_bias": [non_tail_bias],
    "tail_bias": [tail_bias],
})

forecast_hybrid_metrics_2023_rs50

In [ ]:
forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.search.best_params_

## Narrowing the `param_distributions` to generate a more strinctly tuned model: using `prepare_xgb_randomizedsearch()`

In [ ]:
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

# We now point to a package file instead of just the model file
pkg_path = MODELS / "forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.joblib"

# 2. Check/Load or Train/Save
if pkg_path.exists():
    print(f"Loading {pkg_path.name} from disk... (Skipping data prep and training)")
    # Load the entire package (data + fitted search object)
    forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1 = load(pkg_path)
    
    # Extract the model so downstream code works as expected
    best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1 = forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.search.best_estimator_

else:
    print(f"File not found. Preparing data and training...")

    # --- PREPARE DATA ---
    forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1 = prepare_xgb_randomizedsearch(
        eda_hcpcs_lags_present_df,
        approach="forecast",
        target_col="log_delta_cost",
        w_tail=10,
        include_lags=True,
        incl_2020_in_final_fit=False,
        use_ttr=False,
        param_distributions = {
            "model__n_estimators": randint(500, 1300),
            "model__learning_rate": loguniform(0.01, 0.05),

            "model__max_depth": randint(1, 4),              # cap depth
            "model__min_child_weight": randint(30, 151),    # even more conservative
            "model__gamma": uniform(0.0, 3.0),

            "model__subsample": uniform(0.6, 0.35),
            "model__colsample_bytree": uniform(0.6, 0.35),

            "model__reg_lambda": loguniform(10.0, 500.0),   # stronger L2
            "model__reg_alpha": loguniform(1e-6, 0.1),

            "model__max_delta_step": randint(0, 11),
            "model__max_bin": [256, 512],
        }
    )

    # --- TRAIN ---
    if forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.groups_train is None:
        forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.search.fit(
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.X_train,
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.y_train,
            model__sample_weight=forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.sample_w_train,
        )
    else:
        forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.search.fit(
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.X_train,
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.y_train,
            groups=forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.groups_train,
            model__sample_weight=forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.sample_w_train,
        )
        
    best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1 = forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.search.best_estimator_

    # --- SAVE ---
    # 3. Save the ENTIRE package
    dump(forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1, pkg_path)
    print(f"Training complete. Package and model saved to {pkg_path}")

### Hybrid approach for predictions: use model for lag-present rows and use lag + backoff for lag-absent rows

In [ ]:
import numpy as np
import pandas as pd

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"
group_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"]

# 1) Full 2023 test set (all rows, lag and no-lag)
test_df_full_2023 = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"] == 2023].copy()

# 2) Choose which years define the backoff means
# If you want to match the package that trained on 2021-2022 only:
train_years_for_means = [2021, 2022]
# If you intentionally want more history for means, use:
# train_years_for_means = [2020, 2021, 2022]

train_df_for_means = eda_hcpcs_full_df.loc[
    eda_hcpcs_full_df["Year"].isin(train_years_for_means)
].copy()

# 3) Compute backoff mean at (provider_type, HCPCS, POS)
train_means_grp = (
    train_df_for_means.groupby(group_cols)[target_col]
    .mean()
    .rename("mean_train_grp")
    .reset_index()
)

# Optional fallback: HCPCS-only mean
train_means_hcpcs = (
    train_df_for_means.groupby(["HCPCS_Cd"])[target_col]
    .mean()
    .rename("mean_train_hcpcs")
    .reset_index()
)

global_mean = float(train_df_for_means[target_col].mean())

# 4) Attach means to the FULL 2023 test set (preserve original index)
test_with_means = test_df_full_2023.copy()
test_with_means["_orig_idx"] = test_with_means.index

test_with_means = test_with_means.merge(train_means_grp, on=group_cols, how="left")
test_with_means = test_with_means.merge(train_means_hcpcs, on="HCPCS_Cd", how="left")

# Restore original index
test_with_means = test_with_means.set_index("_orig_idx", drop=True)
test_with_means = test_with_means.loc[test_df.index]  # keep same order

# 5) Split lag vs no-lag for residual model
mask_2023_lag = test_with_means["lag1_avg_amt"].notna() & (test_with_means["lag1_avg_amt"] >= 0)
test_2023_lag = test_with_means.loc[mask_2023_lag].copy()
test_2023_nolag = test_with_means.loc[~mask_2023_lag].copy()

# 6) Residual model prediction for lag-present rows
feature_cols = forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.feature_cols
X_2023_lag = test_2023_lag[feature_cols]

log_delta_hat = best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.predict(X_2023_lag)

pred_level_lag = np.expm1(np.log1p(test_2023_lag["lag1_avg_amt"].values) + log_delta_hat)

# 7) Backoff prediction for no-lag rows (group mean -> HCPCS mean -> global)
pred_level_nolag = (
    test_2023_nolag["mean_train_grp"]
    .fillna(test_2023_nolag["mean_train_hcpcs"])
    .fillna(global_mean)
    .to_numpy()
)

# 8) Combine into a full prediction vector aligned to original 2023 test index
pred_full = pd.Series(index=test_df_full_2023.index, dtype=float)
pred_full.loc[test_2023_lag.index] = pred_level_lag
pred_full.loc[test_2023_nolag.index] = pred_level_nolag

# Rebuild pred_backoff

pred_backoff = test_with_means["lag1_avg_amt"].copy()
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_grp"])
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_hcpcs"])
pred_backoff = pred_backoff.fillna(global_mean)

### Calculate model metrics and tail performance

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Inputs you already have from your hybrid prediction cell:
# - test_df_full_2023  (full 2023 rows)
# - pred_full          (full 2023 predictions aligned to test_df_full_2023.index)
# ------------------------------------------------------------

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"

y_true = test_df_full_2023[target_col].astype(float)
pred_test = pred_full.astype(float)

# Basic sanity check
assert pred_test.index.equals(test_df_full_2023.index)

# ------------------------------------------------------------
# 1) Overall metrics (2023 full)
# ------------------------------------------------------------
r2_test = r2_score(y_true, pred_test)
mae = mean_absolute_error(y_true, pred_test)
rmse = root_mean_squared_error(y_true, pred_test)

# "log space" metrics on LEVELS (not on delta), if you still want them:
r2_log_true_test = r2_score(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))
mae_log_true = mean_absolute_error(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))
rmse_log_true = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))

# ------------------------------------------------------------
# 2) Weighted (tail-emphasized) metrics on 2023 full
#    Use the SAME w_tail you used for weighting (10 here)
# ------------------------------------------------------------
w_tail = 10
is_tail_test = test_df_full_2023[tail_flag_col].astype(int).to_numpy()
sample_w_test = (1 + (w_tail - 1) * is_tail_test).astype(float)

r2_test_w = r2_score(y_true, pred_test, sample_weight=sample_w_test)
mae_w = mean_absolute_error(y_true, pred_test, sample_weight=sample_w_test)
rmse_w = root_mean_squared_error(y_true, pred_test, sample_weight=sample_w_test)

# ------------------------------------------------------------
# 3) Tail analysis table (same style as before)
# ------------------------------------------------------------
test_eval = test_df_full_2023.copy()
test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"] = (test_eval[target_col] - test_eval["pred"]) ** 2

non_tail_pred_to_y_ratio = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_pred_to_y_ratio = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

non_tail_sse = test_eval.loc[~test_eval[tail_flag_col], "sq_err"].sum()
tail_sse = test_eval.loc[test_eval[tail_flag_col], "sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = (
    (test_eval.loc[~test_eval[tail_flag_col], "pred"] < test_eval.loc[~test_eval[tail_flag_col], target_col]).mean()
)
tail_under_prediction_rate = (
    (test_eval.loc[test_eval[tail_flag_col], "pred"] < test_eval.loc[test_eval[tail_flag_col], target_col]).mean()
)

non_tail_bias = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_bias = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

forecast_hybrid_metrics_2023_rs50_retune_v1 = pd.DataFrame({
    "w_tail":[w_tail],
    "r2_test": [r2_test],
    "mae": [mae],
    "rmse": [rmse],
    "r2_test_log": [r2_log_true_test],
    "mae_log": [mae_log_true],
    "rmse_log": [rmse_log_true],
    "r2_test_w": [r2_test_w],
    "mae_w": [mae_w],
    "rmse_w": [rmse_w],
    "non_tail_pred_to_y_ratio": [non_tail_pred_to_y_ratio],
    "tail_pred_to_y_ratio": [tail_pred_to_y_ratio],
    "non_tail_sse": [non_tail_sse],
    "tail_sse": [tail_sse],
    "non_tail_share_of_total_sse": [non_tail_share_of_total_sse],
    "tail_share_of_total_sse": [tail_share_of_total_sse],
    "non_tail_under_prediction_rate": [non_tail_under_prediction_rate],
    "tail_under_prediction_rate": [tail_under_prediction_rate],
    "non_tail_bias": [non_tail_bias],
    "tail_bias": [tail_bias],
})

forecast_hybrid_metrics_2023_rs50_retune_v1

In [ ]:
forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.train_df["log_delta_cost"].describe()

## Systematic investigation of underperformance of `best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50`

### 1. Build a single “explain table” for 2023 with all components

In [ ]:
import numpy as np
import pandas as pd

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"

# Full 2023 rows (same as your evaluation set)
df23 = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"] == 2023].copy()

# Identify lag-present rows (eligible for residual model)
mask_lag = df23["lag1_avg_amt"].notna() & (df23["lag1_avg_amt"] >= 0)

# Residual truth for lag-present rows (in log space)
df23.loc[mask_lag, "log_delta_true"] = (
    np.log1p(df23.loc[mask_lag, target_col]) - np.log1p(df23.loc[mask_lag, "lag1_avg_amt"])
)

# Residual model prediction on lag-present rows
feature_cols = forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.feature_cols
X23_lag = df23.loc[mask_lag, feature_cols]
df23.loc[mask_lag, "log_delta_hat"] = best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50.predict(X23_lag)

# Convert to level prediction for lag-present rows
df23.loc[mask_lag, "pred_level_lag"] = np.expm1(
    np.log1p(df23.loc[mask_lag, "lag1_avg_amt"].to_numpy()) + df23.loc[mask_lag, "log_delta_hat"].to_numpy()
)

# If you already have your backoff prediction for no-lag rows:
# df23.loc[~mask_lag, "pred_level_nolag"] = <your backoff array or series aligned to df23>

# Final prediction column (use pred_full if you already computed it correctly)
df23["pred"] = pred_full.reindex(df23.index).astype(float)

# Error decomposition
df23["err"] = df23[target_col] - df23["pred"]
df23["abs_err"] = df23["err"].abs()
df23["sq_err"] = df23["err"] ** 2

# Helpful ratios
df23["pred_to_y"] = df23["pred"] / (df23[target_col] + 1e-9)
df23["lag_to_y"] = df23["lag1_avg_amt"] / (df23[target_col] + 1e-9)
df23["y_to_lag"] = (df23[target_col] + 1e-9) / (df23["lag1_avg_amt"] + 1e-9)

Now you can directly answer: is the failure coming from the residual piece (`log_delta_hat`), or from backoff?

### 2. Separate “catastrophic underprediction” into root causes

In [ ]:
# Example definition. Tweak thresholds to your taste.
catastrophic = (df23[target_col] > 1000) & (df23["pred"] < 200)
share_nolag_within_cat = ((~mask_lag) & catastrophic).sum() / max(1, catastrophic.sum())
df23.loc[catastrophic, ["lag1_avg_amt", target_col, "pred", "log_delta_true", "log_delta_hat"]].head()
print("Catastrophic count:", catastrophic.sum())
print("Share no-lag inside catastrophic:", share_nolag_within_cat)

- If **many catastrophic rows are no-lag**, the model is not “routing expensive to cheap”. Our backoff is. Fix is a better no-lag policy (richer grouping, per-HCPCS quantiles, or a classifier to detect high-cost risk).

- If **catastrophic rows have lag present**, then the residual model is predicting a too-negative delta, or lag itself is not informative in those regimes.

### 2.5 Split catastrophic by mechanism: no-lag vs lag-present

In [ ]:
cat_no_lag = catastrophic & (~mask_lag)
cat_lag = catastrophic & mask_lag

print("Catastrophic no-lag:", cat_no_lag.sum())
print("Catastrophic lag-present:", cat_lag.sum())

### 3. Quantify whether 2023 deltas are out-of-distribution vs training

In [ ]:
train_df = forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.train_df.copy()  # should be 2021-2022 lag-present
train_delta = train_df["log_delta_cost"].astype(float)

test_delta = df23.loc[mask_lag, "log_delta_true"].dropna().astype(float)

print("Train delta quantiles:", train_delta.quantile([0.001,0.01,0.1,0.5,0.9,0.99,0.999]))
print("Test  delta quantiles:", test_delta.quantile([0.001,0.01,0.1,0.5,0.9,0.99,0.999]))

# Where does catastrophic underprediction live in delta space?
print(df23.loc[catastrophic & mask_lag, "log_delta_true"].describe())
print(df23.loc[catastrophic & mask_lag, "log_delta_hat"].describe())

- If catastrophic rows have **true delta far outside training support**, the model will regress toward `0` and fail.

- If `log_delta_hat` has a narrow range while `log_delta_true` is wide, you are seeing “shrinkage to the mean” due to regularization, depth limits, `min_child_weight`, or simply lack of signal.

### 4. Check support. Are these combinations “rare or new”?

For catastrophic lag-present rows, check if their categorical combinations are sparse in training.

If we see lots of `n_train == 0` or tiny counts, then “routing” is just the model being forced to generalize across unseen combos. In that case, the lag-only baseline winning is expected.

In [ ]:
key_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc", "state"]
train_key = train_df[key_cols].copy()
train_key["n_train"] = 1
train_counts = train_key.groupby(key_cols)["n_train"].sum().reset_index()

bad = df23.loc[catastrophic & mask_lag, key_cols + ["lag1_avg_amt", target_col, "pred"]].copy()
bad = bad.merge(train_counts, on=key_cols, how="left")
bad["n_train"] = bad["n_train"].fillna(0).astype(int)

bad.sort_values("n_train").head(20)
print("Median n_train among catastrophic:", bad["n_train"].median())
print("Share with n_train==0:", (bad["n_train"] == 0).mean())

### 5. Compare residual model vs “do nothing” residual (delta = 0)

This isolates whether the residual model is adding value or harming.

If the residual model is worse than delta=0 a lot of the time in the tail, then it is “overcorrecting” the lag. That’s our mechanism.

In [ ]:
# For lag-present rows:
y = df23.loc[mask_lag, target_col].astype(float)
lag = df23.loc[mask_lag, "lag1_avg_amt"].astype(float)

# Delta=0 baseline prediction is just lag (in level space)
pred_delta0 = lag

# Residual model prediction (in level space)
pred_resid = df23.loc[mask_lag, "pred_level_lag"].astype(float)

# Where is residual model worse than delta=0?
worse = (y - pred_resid).abs() > (y - pred_delta0).abs()
print("Share residual worse than delta=0:", worse.mean())

# Specifically among catastrophic underpredictions
cat_lag = catastrophic & mask_lag
worse_cat = worse.reindex(df23.index, fill_value=False) & cat_lag
print("Share catastrophic where residual worse than lag:", worse_cat.mean())

### 6. Explain individual failures with local attribution

In [ ]:
import shap

best_pipe = best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50  # Pipeline(preprocess, model)

pre = best_pipe.named_steps["preprocess"]
xgb = best_pipe.named_steps["model"]

X_trans = pre.transform(X23_lag)
feat_names = pre.get_feature_names_out()

explainer = shap.TreeExplainer(xgb)
# Use a small sample first to keep it fast
lag_err = df23.loc[mask_lag].copy()
lag_err["abs_err_lagmodel"] = (lag_err[target_col] - lag_err["pred_level_lag"]).abs()

idx_bad = lag_err.sort_values("abs_err_lagmodel", ascending=False).head(50).index

X_bad = pre.transform(df23.loc[idx_bad, feature_cols])
shap_values = explainer.shap_values(X_bad)

# Show one example
i = 0
shap.plots.waterfall(shap.Explanation(values=shap_values[i], base_values=explainer.expected_value, data=X_bad[i].toarray()[0] if hasattr(X_bad[i], "toarray") else X_bad[i], feature_names=feat_names))

### 7. Explain the backoff model to explain the true catastrophic failures

In [ ]:
# add this right after you set the index back (after set_index)
df23["pred"] = pred_full.reindex(df23.index).astype(float)

# now catastrophic will work
catastrophic = (df23[target_col] > 1000) & (df23["pred"] < 200)
df23.loc[catastrophic, ["backoff_source", "mean_train_grp", "mean_train_hcpcs",
                        "lag1_avg_amt", target_col, "pred"]]

In [ ]:
print("pred_full covers df23 index:", pred_full.index.isin(df23.index).mean())
print("pred NaN rate in df23:", df23["pred"].isna().mean())

In [ ]:
bad_idx = df23.loc[catastrophic].index

cols = [
    "HCPCS_Cd", "provider_type", "Place_Of_Srvc", "state",
    "lag1_avg_amt", target_col, "pred",
    "mean_train_grp", "mean_train_hcpcs", "backoff_source"
]
df23.loc[bad_idx, cols]

In [ ]:
train_years_for_means = [2021, 2022]
train_df_for_means = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"].isin(train_years_for_means)].copy()

bad_hcpcs = df23.loc[bad_idx, "HCPCS_Cd"]

train_df_for_means.loc[train_df_for_means["HCPCS_Cd"].isin(bad_hcpcs), ["Year","HCPCS_Cd"]].value_counts().head(20)

In [ ]:
train_years_for_means = [2020, 2021, 2022]
train_df_for_means = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"].isin(train_years_for_means)].copy()

bad_hcpcs = df23.loc[bad_idx, "HCPCS_Cd"]

train_df_for_means.loc[train_df_for_means["HCPCS_Cd"].isin(bad_hcpcs), ["Year","HCPCS_Cd"]].value_counts().head(20)

In [ ]:
import numpy as np
import pandas as pd

target_col = "avg_mdcr_stdzd_amt"
group_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"]
pt_pos_cols = ["provider_type", "Place_Of_Srvc"]
pt_cols = ["provider_type"]
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"

# Full 2023 test
df23 = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"] == 2023].copy()

# Means training window (keep as 2021-2022 to match your baseline intent)
train_years_for_means = [2021, 2022]
train_df = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"].isin(train_years_for_means)].copy()

global_mean = float(train_df[target_col].mean())

# Compute multiple mean tables
means_grp = train_df.groupby(group_cols)[target_col].mean().rename("mean_grp").reset_index()
means_hcpcs = train_df.groupby(["HCPCS_Cd"])[target_col].mean().rename("mean_hcpcs").reset_index()
means_pt_pos = train_df.groupby(pt_pos_cols)[target_col].mean().rename("mean_pt_pos").reset_index()
means_pt = train_df.groupby(pt_cols)[target_col].mean().rename("mean_pt").reset_index()

# Merge while preserving original index
df23["_orig_idx"] = df23.index
df23 = df23.merge(means_grp, on=group_cols, how="left")
df23 = df23.merge(means_hcpcs, on="HCPCS_Cd", how="left")
df23 = df23.merge(means_pt_pos, on=pt_pos_cols, how="left")
df23 = df23.merge(means_pt, on=pt_cols, how="left")
df23 = df23.set_index("_orig_idx", drop=True)

# Identify lag-present
mask_lag = df23["lag1_avg_amt"].notna() & (df23["lag1_avg_amt"] >= 0)

# Build backoff prediction for NO-LAG rows only
pred_nolag = df23["mean_grp"]
source = pd.Series(index=df23.index, dtype="object")

source[pred_nolag.notna()] = "grp_mean"
pred_nolag = pred_nolag.fillna(df23["mean_hcpcs"])
source[(source.isna()) & (df23["mean_hcpcs"].notna())] = "hcpcs_mean"

pred_nolag = pred_nolag.fillna(df23["mean_pt_pos"])
source[(source.isna()) & (df23["mean_pt_pos"].notna())] = "pt_pos_mean"

pred_nolag = pred_nolag.fillna(df23["mean_pt"])
source[(source.isna()) & (df23["mean_pt"].notna())] = "pt_mean"

pred_nolag = pred_nolag.fillna(global_mean)
source[source.isna()] = "global_mean"

# Final pred_backoff: use lag when available, else the improved backoff
pred_backoff = df23["lag1_avg_amt"].copy()
pred_backoff.loc[~mask_lag] = pred_nolag.loc[~mask_lag]

df23["pred_backoff"] = pred_backoff.astype(float)
df23["backoff_source"] = "lag"
df23.loc[~mask_lag, "backoff_source"] = source.loc[~mask_lag]

# Now inspect the catastrophic rows again
catastrophic = (df23[target_col] > 1000) & (df23["pred_backoff"] < 200)
df23.loc[catastrophic, ["HCPCS_Cd","provider_type","Place_Of_Srvc","state",
                        "lag1_avg_amt", target_col, "pred_backoff", "backoff_source",
                        "mean_grp","mean_hcpcs","mean_pt_pos","mean_pt"]]

In [ ]:
import numpy as np
import pandas as pd

target_col = "avg_mdcr_stdzd_amt"
group_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"]
pt_pos_state_cols = ["provider_type", "Place_Of_Srvc", "state"]
pt_pos_cols = ["provider_type", "Place_Of_Srvc"]
pt_cols = ["provider_type"]
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"

# Full 2023 test
df23 = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"] == 2023].copy()

# Means training window (keep as 2021-2022 to match your baseline intent)
train_years_for_means = [2021, 2022]
train_df = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"].isin(train_years_for_means)].copy()

global_mean = float(train_df[target_col].mean())

# Compute multiple mean tables
means_grp = train_df.groupby(group_cols)[target_col].mean().rename("mean_grp").reset_index()
means_hcpcs = train_df.groupby(["HCPCS_Cd"])[target_col].mean().rename("mean_hcpcs").reset_index()
means_pt_pos_state = train_df.groupby(pt_pos_state_cols)[target_col].mean().rename("mean_pt_pos_state").reset_index()
medians_pt_pos_state = train_df.groupby(pt_pos_state_cols)[target_col].median().rename("median_pt_pos_state").reset_index()
means_pt_pos = train_df.groupby(pt_pos_cols)[target_col].mean().rename("mean_pt_pos").reset_index()
means_pt = train_df.groupby(pt_cols)[target_col].mean().rename("mean_pt").reset_index()

# Merge while preserving original index
df23["_orig_idx"] = df23.index
df23 = df23.merge(means_grp, on=group_cols, how="left")
df23 = df23.merge(means_hcpcs, on="HCPCS_Cd", how="left")
df23 = df23.merge(means_pt_pos_state, on=pt_pos_state_cols, how="left")
df23 = df23.merge(medians_pt_pos_state, on=pt_pos_state_cols, how="left")
df23 = df23.merge(means_pt_pos, on=pt_pos_cols, how="left")
df23 = df23.merge(means_pt, on=pt_cols, how="left")
df23 = df23.set_index("_orig_idx", drop=True)

# Identify lag-present
mask_lag = df23["lag1_avg_amt"].notna() & (df23["lag1_avg_amt"] >= 0)

# Build backoff prediction for NO-LAG rows only
pred_nolag = df23["mean_grp"]
source = pd.Series(index=df23.index, dtype="object")

source[pred_nolag.notna()] = "grp_mean"
pred_nolag = pred_nolag.fillna(df23["mean_hcpcs"])
source[(source.isna()) & (df23["mean_hcpcs"].notna())] = "hcpcs_mean"

pred_nolag = pred_nolag.fillna(df23["mean_pt_pos_state"])
source[(source.isna()) & (df23["mean_pt_pos_state"].notna())] = "pt_pos_state_mean"

pred_nolag = pred_nolag.fillna(df23["median_pt_pos_state"])
source[(source.isna()) & (df23["median_pt_pos_state"].notna())] = "pt_pos_state_median"

pred_nolag = pred_nolag.fillna(df23["mean_pt_pos"])
source[(source.isna()) & (df23["mean_pt_pos"].notna())] = "pt_pos_mean"

pred_nolag = pred_nolag.fillna(df23["mean_pt"])
source[(source.isna()) & (df23["mean_pt"].notna())] = "pt_mean"

pred_nolag = pred_nolag.fillna(global_mean)
source[source.isna()] = "global_mean"

# Final pred_backoff: use lag when available, else the improved backoff
pred_backoff = df23["lag1_avg_amt"].copy()
pred_backoff.loc[~mask_lag] = pred_nolag.loc[~mask_lag]

df23["pred_backoff"] = pred_backoff.astype(float)
df23["backoff_source"] = "lag"
df23.loc[~mask_lag, "backoff_source"] = source.loc[~mask_lag]

# Now inspect the catastrophic rows again
catastrophic = (df23[target_col] > 1000) & (df23["pred_backoff"] < 200)
df23.loc[catastrophic, [
    "HCPCS_Cd", "provider_type", "Place_Of_Srvc", "state",
    "lag1_avg_amt", target_col, "pred_backoff", "backoff_source",
    "mean_grp", "mean_hcpcs", "mean_pt_pos_state", "median_pt_pos_state",
    "mean_pt_pos", "mean_pt"
]]

So the next question is not really “is the code correct?”

It is “do we want a stronger no-lag unseen-HCPCS fallback policy?”

A very practical next step would be to quantify how many no-lag rows use each fallback source and how each source performs. For example:

In [ ]:
df23.value_counts(mask_lag)

In [ ]:
df23.loc[~mask_lag, "backoff_source"].value_counts(dropna=False)

and then error by source:

In [ ]:
df23.loc[~mask_lag].groupby("backoff_source").apply(
    lambda g: pd.Series({
        "n": len(g),
        "mae": (g[target_col] - g["pred_backoff"]).abs().mean(),
        "rmse": np.sqrt(((g[target_col] - g["pred_backoff"]) ** 2).mean()),
        "mean_true": g[target_col].mean(),
        "mean_pred": g["pred_backoff"].mean(),
    })
)

## Let's restrict evaluation to the subset where `lag1_avg_amt` exists, then compare:
- model-based lag-row predictions from `best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50`
- naive lag prediction using `lag1_avg_amt` directly

That gives us a clean answer to:

On the rows where the residual model is actually allowed to operate, did it improve on simply carrying lag forward?

What this table tells us

This is the cleanest apples-to-apples test for the residual model.
- If the residual model has better R² / lower MAE / lower RMSE than `lag1_avg_amt_baseline_on_lag_rows`, then it is adding value on lag-present rows.

- If it is worse, then the residual model is not beating the simpler “just use lag” rule on the rows it was specifically designed for.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"
w_tail = 10

# ------------------------------------------------------------
# 1) Build the lag-present evaluation subset
# ------------------------------------------------------------
df_lag = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"] == 2023].copy()
mask_lag = df_lag["lag1_avg_amt"].notna() & (df_lag["lag1_avg_amt"] >= 0)
df_lag = df_lag.loc[mask_lag].copy()

# ------------------------------------------------------------
# 2) Generate model predictions on lag-present rows only
# ------------------------------------------------------------
feature_cols = forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.feature_cols
X_lag = df_lag[feature_cols]

log_delta_hat = best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50.predict(X_lag)

pred_model = np.expm1(np.log1p(df_lag["lag1_avg_amt"].to_numpy()) + log_delta_hat)
pred_lag = df_lag["lag1_avg_amt"].to_numpy()

y_true = df_lag[target_col].astype(float).to_numpy()

# ------------------------------------------------------------
# 3) Tail weights on this lag-present subset
# ------------------------------------------------------------
is_tail = df_lag[tail_flag_col].astype(int).to_numpy()
sample_w = (1 + (w_tail - 1) * is_tail).astype(float)

# ------------------------------------------------------------
# 4) Helper to compute the same metrics table style
# ------------------------------------------------------------
def eval_metrics_table(df, y_true, pred, sample_w, model_label, target_col, tail_flag_col):
    df_eval = df.copy()
    df_eval["pred"] = pred
    df_eval["abs_err"] = np.abs(y_true - pred)
    df_eval["sq_err"] = (y_true - pred) ** 2

    is_tail = df_eval[tail_flag_col].astype(bool)
    is_non_tail = ~is_tail

    r2_test = r2_score(y_true, pred)
    mae = mean_absolute_error(y_true, pred)
    rmse = root_mean_squared_error(y_true, pred)

    pred_clip = np.clip(pred, a_min=0, a_max=None)
    r2_test_log = r2_score(np.log1p(y_true), np.log1p(pred_clip))
    mae_log = mean_absolute_error(np.log1p(y_true), np.log1p(pred_clip))
    rmse_log = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_clip))

    r2_test_w = r2_score(y_true, pred, sample_weight=sample_w)
    mae_w = mean_absolute_error(y_true, pred, sample_weight=sample_w)
    rmse_w = root_mean_squared_error(y_true, pred, sample_weight=sample_w)

    non_tail_pred_to_y_ratio = (
        df_eval.loc[is_non_tail, "pred"].mean() /
        df_eval.loc[is_non_tail, target_col].mean()
    )
    tail_pred_to_y_ratio = (
        df_eval.loc[is_tail, "pred"].mean() /
        df_eval.loc[is_tail, target_col].mean()
    )

    non_tail_sse = df_eval.loc[is_non_tail, "sq_err"].sum()
    tail_sse = df_eval.loc[is_tail, "sq_err"].sum()

    non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
    tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

    non_tail_under_prediction_rate = (
        (df_eval.loc[is_non_tail, "pred"] < df_eval.loc[is_non_tail, target_col]).mean()
    )
    tail_under_prediction_rate = (
        (df_eval.loc[is_tail, "pred"] < df_eval.loc[is_tail, target_col]).mean()
    )

    non_tail_bias = (
        df_eval.loc[is_non_tail, "pred"].mean() -
        df_eval.loc[is_non_tail, target_col].mean()
    )
    tail_bias = (
        df_eval.loc[is_tail, "pred"].mean() -
        df_eval.loc[is_tail, target_col].mean()
    )

    return pd.DataFrame({
        "model": [model_label],
        "rows_eval": [len(df_eval)],
        "w_tail": [w_tail],
        "r2_test": [r2_test],
        "mae": [mae],
        "rmse": [rmse],
        "r2_test_log": [r2_test_log],
        "mae_log": [mae_log],
        "rmse_log": [rmse_log],
        "r2_test_w": [r2_test_w],
        "mae_w": [mae_w],
        "rmse_w": [rmse_w],
        "non_tail_pred_to_y_ratio": [non_tail_pred_to_y_ratio],
        "tail_pred_to_y_ratio": [tail_pred_to_y_ratio],
        "non_tail_sse": [non_tail_sse],
        "tail_sse": [tail_sse],
        "non_tail_share_of_total_sse": [non_tail_share_of_total_sse],
        "tail_share_of_total_sse": [tail_share_of_total_sse],
        "non_tail_under_prediction_rate": [non_tail_under_prediction_rate],
        "tail_under_prediction_rate": [tail_under_prediction_rate],
        "non_tail_bias": [non_tail_bias],
        "tail_bias": [tail_bias],
    })

# ------------------------------------------------------------
# 5) Build side-by-side comparison table
# ------------------------------------------------------------
lag_subset_model_metrics = eval_metrics_table(
    df=df_lag,
    y_true=y_true,
    pred=pred_model,
    sample_w=sample_w,
    model_label="residual_model_on_lag_rows",
    target_col=target_col,
    tail_flag_col=tail_flag_col,
)

lag_subset_naive_metrics = eval_metrics_table(
    df=df_lag,
    y_true=y_true,
    pred=pred_lag,
    sample_w=sample_w,
    model_label="lag1_avg_amt_baseline_on_lag_rows",
    target_col=target_col,
    tail_flag_col=tail_flag_col,
)

lag_rows_comparison_metrics = pd.concat(
    [lag_subset_model_metrics, lag_subset_naive_metrics],
    ignore_index=True,
)

lag_rows_comparison_metrics

### Let's explicitly subtract baseline from model for the main metrics

In [ ]:
compare = lag_rows_comparison_metrics.set_index("model")

delta_summary = pd.DataFrame({
    "metric": ["r2_test", "mae", "rmse", "r2_test_w", "mae_w", "rmse_w"],
    "model_minus_baseline": [
        compare.loc["residual_model_on_lag_rows", "r2_test"] - compare.loc["lag1_avg_amt_baseline_on_lag_rows", "r2_test"],
        compare.loc["residual_model_on_lag_rows", "mae"] - compare.loc["lag1_avg_amt_baseline_on_lag_rows", "mae"],
        compare.loc["residual_model_on_lag_rows", "rmse"] - compare.loc["lag1_avg_amt_baseline_on_lag_rows", "rmse"],
        compare.loc["residual_model_on_lag_rows", "r2_test_w"] - compare.loc["lag1_avg_amt_baseline_on_lag_rows", "r2_test_w"],
        compare.loc["residual_model_on_lag_rows", "mae_w"] - compare.loc["lag1_avg_amt_baseline_on_lag_rows", "mae_w"],
        compare.loc["residual_model_on_lag_rows", "rmse_w"] - compare.loc["lag1_avg_amt_baseline_on_lag_rows", "rmse_w"],
    ]
})

delta_summary

# Summarize hybrid model (residual model (`best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50`) plus fallback ladder) vs lag baseline (lag + fallback ladder)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"
w_tail = 10

# -------------------------------------------------------------------
# Assumes you already have:
# 1) df23 with:
#    - lag1_avg_amt
#    - pred_backoff
#    - backoff_source
#    - target_col
#    - tail_flag_col
# 2) fitted residual model:
#    - best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50
#    - forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.feature_cols
# -------------------------------------------------------------------

# If not already present, define lag mask on df23
mask_lag = df23["lag1_avg_amt"].notna() & (df23["lag1_avg_amt"] >= 0)
mask_nolag = ~mask_lag

# -------------------------------------------------------------------
# Rebuild residual-model predictions on lag-present rows
# -------------------------------------------------------------------
feature_cols = forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.feature_cols
X_lag = df23.loc[mask_lag, feature_cols]

log_delta_hat = best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50.predict(X_lag)

pred_residual_lag = np.expm1(
    np.log1p(df23.loc[mask_lag, "lag1_avg_amt"].to_numpy()) + log_delta_hat
)

# Baseline on lag-present rows is simply lag1_avg_amt
pred_lag_baseline_lag = df23.loc[mask_lag, "lag1_avg_amt"].to_numpy()

# On no-lag rows, both methods use the same fallback ladder
pred_nolag_shared = df23.loc[mask_nolag, "pred_backoff"].to_numpy()

# -------------------------------------------------------------------
# Build full-row predictions for both strategies
# -------------------------------------------------------------------
pred_all_lag_baseline = df23["pred_backoff"].astype(float).to_numpy()

pred_all_hybrid = df23["pred_backoff"].astype(float).copy()
pred_all_hybrid[mask_lag.to_numpy()] = pred_residual_lag

# -------------------------------------------------------------------
# Helper to compute a one-row metrics summary
# -------------------------------------------------------------------
def summarize_segment(
    df_segment: pd.DataFrame,
    pred: np.ndarray,
    segment_name: str,
    strategy_name: str,
    target_col: str,
    tail_flag_col: str,
    w_tail: int,
    fallback_note: str = ""
) -> pd.DataFrame:
    y_true = df_segment[target_col].astype(float).to_numpy()
    pred = np.asarray(pred, dtype=float)

    sample_w = (
        1 + (w_tail - 1) * df_segment[tail_flag_col].astype(int).to_numpy()
    ).astype(float)

    df_eval = df_segment.copy()
    df_eval["pred"] = pred
    df_eval["abs_err"] = np.abs(y_true - pred)
    df_eval["sq_err"] = (y_true - pred) ** 2

    is_tail = df_eval[tail_flag_col].astype(bool)
    is_non_tail = ~is_tail

    r2_test = r2_score(y_true, pred)
    mae = mean_absolute_error(y_true, pred)
    rmse = root_mean_squared_error(y_true, pred)

    pred_clip = np.clip(pred, a_min=0, a_max=None)
    r2_test_log = r2_score(np.log1p(y_true), np.log1p(pred_clip))
    mae_log = mean_absolute_error(np.log1p(y_true), np.log1p(pred_clip))
    rmse_log = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_clip))

    r2_test_w = r2_score(y_true, pred, sample_weight=sample_w)
    mae_w = mean_absolute_error(y_true, pred, sample_weight=sample_w)
    rmse_w = root_mean_squared_error(y_true, pred, sample_weight=sample_w)

    non_tail_pred_to_y_ratio = (
        df_eval.loc[is_non_tail, "pred"].mean()
        / df_eval.loc[is_non_tail, target_col].mean()
        if is_non_tail.sum() > 0 else np.nan
    )
    tail_pred_to_y_ratio = (
        df_eval.loc[is_tail, "pred"].mean()
        / df_eval.loc[is_tail, target_col].mean()
        if is_tail.sum() > 0 else np.nan
    )

    non_tail_sse = df_eval.loc[is_non_tail, "sq_err"].sum()
    tail_sse = df_eval.loc[is_tail, "sq_err"].sum()
    total_sse = non_tail_sse + tail_sse

    non_tail_share_of_total_sse = non_tail_sse / total_sse if total_sse > 0 else np.nan
    tail_share_of_total_sse = tail_sse / total_sse if total_sse > 0 else np.nan

    non_tail_under_prediction_rate = (
        (df_eval.loc[is_non_tail, "pred"] < df_eval.loc[is_non_tail, target_col]).mean()
        if is_non_tail.sum() > 0 else np.nan
    )
    tail_under_prediction_rate = (
        (df_eval.loc[is_tail, "pred"] < df_eval.loc[is_tail, target_col]).mean()
        if is_tail.sum() > 0 else np.nan
    )

    non_tail_bias = (
        df_eval.loc[is_non_tail, "pred"].mean()
        - df_eval.loc[is_non_tail, target_col].mean()
        if is_non_tail.sum() > 0 else np.nan
    )
    tail_bias = (
        df_eval.loc[is_tail, "pred"].mean()
        - df_eval.loc[is_tail, target_col].mean()
        if is_tail.sum() > 0 else np.nan
    )

    return pd.DataFrame({
        "segment": [segment_name],
        "prediction_strategy": [strategy_name],
        "rows_eval": [len(df_segment)],
        "tail_rows": [int(is_tail.sum())],
        "non_tail_rows": [int(is_non_tail.sum())],
        "w_tail": [w_tail],
        "fallback_note": [fallback_note],
        "r2_test": [r2_test],
        "mae": [mae],
        "rmse": [rmse],
        "r2_test_log": [r2_test_log],
        "mae_log": [mae_log],
        "rmse_log": [rmse_log],
        "r2_test_w": [r2_test_w],
        "mae_w": [mae_w],
        "rmse_w": [rmse_w],
        "non_tail_pred_to_y_ratio": [non_tail_pred_to_y_ratio],
        "tail_pred_to_y_ratio": [tail_pred_to_y_ratio],
        "non_tail_sse": [non_tail_sse],
        "tail_sse": [tail_sse],
        "non_tail_share_of_total_sse": [non_tail_share_of_total_sse],
        "tail_share_of_total_sse": [tail_share_of_total_sse],
        "non_tail_under_prediction_rate": [non_tail_under_prediction_rate],
        "tail_under_prediction_rate": [tail_under_prediction_rate],
        "non_tail_bias": [non_tail_bias],
        "tail_bias": [tail_bias],
    })

# -------------------------------------------------------------------
# Build the four segment rows
# -------------------------------------------------------------------

# 1) Lag-present, lag baseline
tbl_lag_baseline_lag = summarize_segment(
    df_segment=df23.loc[mask_lag].copy(),
    pred=pred_lag_baseline_lag,
    segment_name="lag_present",
    strategy_name="lag_baseline",
    target_col=target_col,
    tail_flag_col=tail_flag_col,
    w_tail=w_tail,
    fallback_note="uses lag1_avg_amt directly"
)

# 2) Lag-present, residual model
tbl_residual_lag = summarize_segment(
    df_segment=df23.loc[mask_lag].copy(),
    pred=pred_residual_lag,
    segment_name="lag_present",
    strategy_name="residual_model",
    target_col=target_col,
    tail_flag_col=tail_flag_col,
    w_tail=w_tail,
    fallback_note="uses residual model on top of lag1_avg_amt"
)

# 3) No-lag, lag baseline path (really fallback ladder)
tbl_lag_baseline_nolag = summarize_segment(
    df_segment=df23.loc[mask_nolag].copy(),
    pred=pred_nolag_shared,
    segment_name="no_lag",
    strategy_name="lag_baseline",
    target_col=target_col,
    tail_flag_col=tail_flag_col,
    w_tail=w_tail,
    fallback_note="fallback ladder: grp -> hcpcs -> pt_pos_state_mean -> pt_pos_state_median -> pt_pos -> pt -> global"
)

# 4) No-lag, residual model path (same fallback ladder)
tbl_residual_nolag = summarize_segment(
    df_segment=df23.loc[mask_nolag].copy(),
    pred=pred_nolag_shared,
    segment_name="no_lag",
    strategy_name="residual_model",
    target_col=target_col,
    tail_flag_col=tail_flag_col,
    w_tail=w_tail,
    fallback_note="same fallback ladder as lag baseline"
)

# -------------------------------------------------------------------
# Build two all-rows rows
# -------------------------------------------------------------------

# 5) All rows, lag baseline + fallback
tbl_lag_baseline_all = summarize_segment(
    df_segment=df23.copy(),
    pred=pred_all_lag_baseline,
    segment_name="all_rows",
    strategy_name="lag_baseline",
    target_col=target_col,
    tail_flag_col=tail_flag_col,
    w_tail=w_tail,
    fallback_note="lag1_avg_amt when present, else fallback ladder"
)

# 6) All rows, hybrid = residual on lag rows + fallback on no-lag rows
tbl_hybrid_all = summarize_segment(
    df_segment=df23.copy(),
    pred=pred_all_hybrid,
    segment_name="all_rows",
    strategy_name="hybrid",
    target_col=target_col,
    tail_flag_col=tail_flag_col,
    w_tail=w_tail,
    fallback_note="residual model on lag-present rows, same fallback ladder on no-lag rows"
)

# -------------------------------------------------------------------
# Final comprehensive summary table
# -------------------------------------------------------------------
lag_vs_residual_segment_summary = pd.concat([
    tbl_lag_baseline_lag,
    tbl_residual_lag,
    tbl_lag_baseline_nolag,
    tbl_residual_nolag,
    tbl_lag_baseline_all,
    tbl_hybrid_all,
], ignore_index=True)

lag_vs_residual_segment_summary

## Forecast Model Comparison Summary

The segment-level comparison tells a clear and internally consistent story about how the residual model behaves relative to the naive lag baseline.

---

### Overall Conclusion

The **residual model is not uniformly better** than the lag baseline. Its benefit depends on which performance criterion matters most:

* It **improves R² and RMSE** on lag-present rows and on the full 2023 evaluation set.
* It **worsens MAE, weighted metrics, and tail calibration** relative to the simpler lag-based baseline.
* On **no-lag rows**, both approaches are identical because both use the same fallback ladder.

> **Key Takeaway:** The residual model is helping mostly on **average squared-error fit**, but it is not improving the aspects of performance that matter most for **robustness, tail behavior, or practical reliability**.

---

### 1. Performance on lag-present rows

Among the **250,090 rows** where `lag1_avg_amt` exists, the naive lag baseline is already extremely strong. This confirms that year-over-year lag is a very powerful predictor in this dataset. For most providers and services, cost per service is highly persistent from one year to the next.

| Metric | Lag Baseline | Residual Model | Difference |
| :--- | :--- | :--- | :--- |
| **R²** | 0.972676 | 0.976289 | Improved by ~0.0036 |
| **MAE** | 4.413878 | 4.613294 | Worsened by ~0.20 |
| **RMSE** | 29.008265 | 27.021981 | Improved by ~2.0 |

This indicates that the residual model is improving fit in a **squared-error sense**, but not in an **absolute-error sense**. In other words, it helps enough moderate-to-large errors to lower RMSE, but it also introduces enough additional deviation from lag to make typical absolute errors slightly worse.

#### Tail-specific behavior on lag-present rows

The lag-present row comparison is especially important because this is the only segment where the residual model is actually making learned adjustments. Compared with the lag baseline, the residual model showed:

* Worse weighted R²
* Worse weighted MAE
* Worse weighted RMSE
* Stronger underprediction in the tail
* More negative tail bias

**For example:**
* **Tail prediction-to-actual ratio:** Fell from about 1.0002 under the lag baseline to 0.9883 under the residual model.
* **Tail bias:** Moved from about +0.27 to about -15.60.

This means that although the residual model improves average squared-error fit, it is also **less well calibrated in high-cost regimes**. The model is pushing predictions away from lag in ways that help some rows overall, but can hurt the most important expensive rows.

---

### 2. Performance on no-lag rows

Among the **41,919 rows** where lag is missing, both approaches are identical because both use the same fallback ladder. 

**Metrics:**
* **R²:** 0.748200
* **MAE:** 14.540462
* **RMSE:** 83.642306

This is materially worse than lag-present performance, which is expected. These rows are inherently harder because the strongest signal, prior-year lag, is absent.

Still, the fallback ladder performs reasonably given the difficulty of the problem. It provides a workable systematic strategy for no-lag rows, but clearly these rows remain the hardest segment of the forecasting problem. Because the lag baseline and residual-model system use the **same no-lag fallback strategy**, there is nothing to compare between them on this segment unless the fallback ladder itself is changed.

---

### 3. Full 2023 performance across all rows

When evaluated on all **292,009 rows** in 2023, the tradeoff seen on lag-present rows carries through to the full end-to-end system:

| Metric | Lag Baseline + Fallback | Hybrid Model |
| :--- | :--- | :--- |
| **R²** | 0.943189 | 0.946328 |
| **MAE** | 5.867587 | 6.038377 |
| **RMSE** | 41.532990 | 40.369217 |

* The **hybrid model** improves R² and RMSE.
* The **lag baseline** retains better MAE.

So the hybrid system is better if the goal is to optimize **variance explained** or **squared-error loss**, but the simpler lag baseline remains better if the goal is **absolute accuracy** and **predictive stability**.

---

### 4. Tail behavior across all rows

The tail metrics are especially important here because the most expensive rows often matter most analytically and operationally.

| Metric | Lag Baseline + Fallback | Hybrid Model |
| :--- | :--- | :--- |
| **Tail prediction-to-actual ratio** | 0.997044 | 0.986999 |
| **Tail bias** | -3.901280 | -17.156784 |
| **Tail underprediction rate** | 0.311879 | 0.601849 |

**This is one of the most important findings in the notebook.**

The lag baseline is **much better calibrated in the tail**. Its tail ratio is very close to 1, its bias is only mildly negative, and it underpredicts tail rows much less often.

The hybrid model, by contrast, has a much stronger tendency to **underpredict tail observations**. Even though it improves global R² and RMSE, it does so at the cost of **more tail underestimation and worse tail-weighted behavior**. This is a strong sign that the residual model is not conservative enough relative to the already-powerful lag anchor.

---

### 5. Interpretation of model behavior

These results suggest the following behavioral pattern:

* **Lag is already an exceptionally strong predictor** for rows where it exists.
* The residual model learns corrections that sometimes improve fit enough to reduce RMSE.
* However, because lag is already so strong, the model has limited room to add value.
* As a result, some learned corrections improve average squared-error performance while still making predictions **less reliable in the tail**.
* On no-lag rows, the system depends entirely on the fallback ladder, so performance there is driven by the quality of the backoff policy rather than the residual model.

In practical terms, the residual model behaves more like a **challenger model** that can improve average fit, but not yet like a clearly superior replacement for the simpler lag-based system.

---

### 6. Practical takeaway

If the goal is to build the **strongest practical forecasting system**, the evidence currently favors the **lag baseline + fallback ladder**.

**Why:**
* It is simpler.
* It is easier to explain.
* It is extremely strong on lag-present rows.
* It is better calibrated in the tail.
* It performs better on MAE and weighted metrics.
* It appears more robust and production-safe.

If the goal is instead to optimize **R² or RMSE**, then the **hybrid residual approach** is a legitimate challenger because it does improve those metrics. But based on the current evidence, the hybrid model is best viewed as interesting and technically competitive, but **not yet superior in practical forecasting terms**.

---

### Final Summary

* **On lag-present rows:** The residual model improved R² and RMSE relative to the naive lag baseline, but worsened MAE and degraded tail-weighted performance.
* **On no-lag rows:** Both approaches were identical because both used the same fallback ladder.
* **Across all 2023 rows:** The hybrid model improved R² (0.946 vs 0.943) and RMSE (40.37 vs 41.53), but the lag baseline retained better MAE (5.87 vs 6.04), better weighted metrics, and much better tail calibration.
* **Overall:** The simpler **lag + fallback** approach appears to be the stronger practical forecasting strategy, while the **residual model + fallback** remains a useful challenger for improving average squared-error fit.

## Rebuild `forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50` but without lag features: build `forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50`

In [ ]:
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

# We now point to a package file instead of just the model file
pkg_path = MODELS / "forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.joblib"

# 2. Check/Load or Train/Save
if pkg_path.exists():
    print(f"Loading {pkg_path.name} from disk... (Skipping data prep and training)")
    # Load the entire package (data + fitted search object)
    forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50 = load(pkg_path)
    
    # Extract the model so downstream code works as expected
    best_forecast_model_wt10_without_lags_no_2020_delta_trgt_rs50 = forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.search.best_estimator_

else:
    print(f"File not found. Preparing data and training...")

    # --- PREPARE DATA ---
    forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50 = prepare_xgb_randomizedsearch(
        eda_hcpcs_lags_present_df,
        approach="forecast",
        target_col="log_delta_cost",
        w_tail=10,
        include_lags=False,
        incl_2020_in_final_fit=False,
        use_ttr=False
    )

    # --- TRAIN ---
    if forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.groups_train is None:
        forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.search.fit(
            forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.X_train,
            forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.y_train,
            model__sample_weight=forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.sample_w_train,
        )
    else:
        forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.search.fit(
            forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.X_train,
            forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.y_train,
            groups=forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.groups_train,
            model__sample_weight=forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.sample_w_train,
        )
        
    best_forecast_model_wt10_without_lags_no_2020_delta_trgt_rs50 = forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.search.best_estimator_

    # --- SAVE ---
    # 3. Save the ENTIRE package
    dump(forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50, pkg_path)
    print(f"Training complete. Package and model saved to {pkg_path}")

In [ ]:
import numpy as np
import pandas as pd

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"
group_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"]

# 1) Full 2023 test set (all rows, lag and no-lag)
test_df_full_2023 = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"] == 2023].copy()

# 2) Choose which years define the backoff means
# If you want to match the package that trained on 2021-2022 only:
train_years_for_means = [2021, 2022]
# If you intentionally want more history for means, use:
# train_years_for_means = [2020, 2021, 2022]

train_df_for_means = eda_hcpcs_full_df.loc[
    eda_hcpcs_full_df["Year"].isin(train_years_for_means)
].copy()

# 3) Compute backoff mean at (provider_type, HCPCS, POS)
train_means_grp = (
    train_df_for_means.groupby(group_cols)[target_col]
    .mean()
    .rename("mean_train_grp")
    .reset_index()
)

# Optional fallback: HCPCS-only mean
train_means_hcpcs = (
    train_df_for_means.groupby(["HCPCS_Cd"])[target_col]
    .mean()
    .rename("mean_train_hcpcs")
    .reset_index()
)

global_mean = float(train_df_for_means[target_col].mean())

# 4) Attach means to the FULL 2023 test set (preserve original index)
test_with_means = test_df_full_2023.copy()
test_with_means["_orig_idx"] = test_with_means.index

test_with_means = test_with_means.merge(train_means_grp, on=group_cols, how="left")
test_with_means = test_with_means.merge(train_means_hcpcs, on="HCPCS_Cd", how="left")

# Restore original index
test_with_means = test_with_means.set_index("_orig_idx", drop=True)
test_with_means = test_with_means.loc[test_df.index]  # keep same order

# 5) Split lag vs no-lag for residual model
mask_2023_lag = test_with_means["lag1_avg_amt"].notna() & (test_with_means["lag1_avg_amt"] >= 0)
test_2023_lag = test_with_means.loc[mask_2023_lag].copy()
test_2023_nolag = test_with_means.loc[~mask_2023_lag].copy()

# 6) Residual model prediction for lag-present rows
feature_cols = forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.feature_cols
X_2023_lag = test_2023_lag[feature_cols]

log_delta_hat = best_forecast_model_wt10_without_lags_no_2020_delta_trgt_rs50.predict(X_2023_lag)

pred_level_lag = np.expm1(np.log1p(test_2023_lag["lag1_avg_amt"].values) + log_delta_hat)

# 7) Backoff prediction for no-lag rows (group mean -> HCPCS mean -> global)
pred_level_nolag = (
    test_2023_nolag["mean_train_grp"]
    .fillna(test_2023_nolag["mean_train_hcpcs"])
    .fillna(global_mean)
    .to_numpy()
)

# 8) Combine into a full prediction vector aligned to original 2023 test index
pred_full = pd.Series(index=test_df_full_2023.index, dtype=float)
pred_full.loc[test_2023_lag.index] = pred_level_lag
pred_full.loc[test_2023_nolag.index] = pred_level_nolag

# Rebuild pred_backoff

pred_backoff = test_with_means["lag1_avg_amt"].copy()
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_grp"])
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_hcpcs"])
pred_backoff = pred_backoff.fillna(global_mean)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Inputs you already have from your hybrid prediction cell:
# - test_df_full_2023  (full 2023 rows)
# - pred_full          (full 2023 predictions aligned to test_df_full_2023.index)
# ------------------------------------------------------------

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"

y_true = test_df_full_2023[target_col].astype(float)
pred_test = pred_full.astype(float)

# Basic sanity check
assert pred_test.index.equals(test_df_full_2023.index)

# ------------------------------------------------------------
# 1) Overall metrics (2023 full)
# ------------------------------------------------------------
r2_test = r2_score(y_true, pred_test)
mae = mean_absolute_error(y_true, pred_test)
rmse = root_mean_squared_error(y_true, pred_test)

# "log space" metrics on LEVELS (not on delta), if you still want them:
r2_log_true_test = r2_score(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))
mae_log_true = mean_absolute_error(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))
rmse_log_true = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))

# ------------------------------------------------------------
# 2) Weighted (tail-emphasized) metrics on 2023 full
#    Use the SAME w_tail you used for weighting (10 here)
# ------------------------------------------------------------
w_tail = 10
is_tail_test = test_df_full_2023[tail_flag_col].astype(int).to_numpy()
sample_w_test = (1 + (w_tail - 1) * is_tail_test).astype(float)

r2_test_w = r2_score(y_true, pred_test, sample_weight=sample_w_test)
mae_w = mean_absolute_error(y_true, pred_test, sample_weight=sample_w_test)
rmse_w = root_mean_squared_error(y_true, pred_test, sample_weight=sample_w_test)

# ------------------------------------------------------------
# 3) Tail analysis table (same style as before)
# ------------------------------------------------------------
test_eval = test_df_full_2023.copy()
test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"] = (test_eval[target_col] - test_eval["pred"]) ** 2

non_tail_pred_to_y_ratio = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_pred_to_y_ratio = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

non_tail_sse = test_eval.loc[~test_eval[tail_flag_col], "sq_err"].sum()
tail_sse = test_eval.loc[test_eval[tail_flag_col], "sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = (
    (test_eval.loc[~test_eval[tail_flag_col], "pred"] < test_eval.loc[~test_eval[tail_flag_col], target_col]).mean()
)
tail_under_prediction_rate = (
    (test_eval.loc[test_eval[tail_flag_col], "pred"] < test_eval.loc[test_eval[tail_flag_col], target_col]).mean()
)

non_tail_bias = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_bias = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

forecast_hybrid_metrics_2023_without_lags_rs50 = pd.DataFrame({
    "w_tail":[w_tail],
    "r2_test": [r2_test],
    "mae": [mae],
    "rmse": [rmse],
    "r2_test_log": [r2_log_true_test],
    "mae_log": [mae_log_true],
    "rmse_log": [rmse_log_true],
    "r2_test_w": [r2_test_w],
    "mae_w": [mae_w],
    "rmse_w": [rmse_w],
    "non_tail_pred_to_y_ratio": [non_tail_pred_to_y_ratio],
    "tail_pred_to_y_ratio": [tail_pred_to_y_ratio],
    "non_tail_sse": [non_tail_sse],
    "tail_sse": [tail_sse],
    "non_tail_share_of_total_sse": [non_tail_share_of_total_sse],
    "tail_share_of_total_sse": [tail_share_of_total_sse],
    "non_tail_under_prediction_rate": [non_tail_under_prediction_rate],
    "tail_under_prediction_rate": [tail_under_prediction_rate],
    "non_tail_bias": [non_tail_bias],
    "tail_bias": [tail_bias],
})

forecast_hybrid_metrics_2023_without_lags_rs50

## 2. New provider approach: Training a model that predicts the average cost per service for new providers 

In [ ]:
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

# We now point to a package file instead of just the model file
pkg_path = MODELS / "newprov_pkg_wt10_with_lags.joblib"

# 2. Check/Load or Train/Save
if pkg_path.exists():
    print(f"Loading {pkg_path.name} from disk... (Skipping data prep and training)")
    # Load the entire package (data + fitted search object)
    newprov_pkg_wt10_with_lags = load(pkg_path)
    
    # Extract the model so downstream code works as expected
    best_newprov_model_wt10_with_lags = newprov_pkg_wt10_with_lags.search.best_estimator_

else:
    print(f"File not found. Preparing data and training...")

    # --- PREPARE DATA ---
    # 2) New providers: group holdout + GroupKFold (by NPI)
    newprov_pkg_wt10_with_lags = prepare_xgb_gridsearch(
        eda_hcpcs_df,
        approach="new_providers",
        target_col="avg_mdcr_stdzd_amt",
        w_tail=10,
        include_lags=True,
        test_size=0.2,
        random_state=0,
        n_splits=3,
    )

    # --- TRAIN ---
    newprov_pkg_wt10_with_lags.search.fit(
        newprov_pkg_wt10_with_lags.X_train,
        newprov_pkg_wt10_with_lags.y_train,
        groups=newprov_pkg_wt10_with_lags.groups_train,
        model__sample_weight=newprov_pkg_wt10_with_lags.sample_w_train,
    )
    
    best_newprov_model_wt10_with_lags = newprov_pkg_wt10_with_lags.search.best_estimator_

    # --- SAVE ---
    # 3. Save the ENTIRE package
    dump(newprov_pkg_wt10_with_lags, pkg_path)
    print(f"Training complete. Package and model saved to {pkg_path}")

#### Model diagnostics (weighted training including lag variables)

In [ ]:
# Run diagnostics

# get train and test predictions and clip test predictions 
pred_test = best_newprov_model_wt10_with_lags.predict(newprov_pkg_wt10_with_lags.X_test)
pred_train = best_newprov_model_wt10_with_lags.predict(newprov_pkg_wt10_with_lags.X_train)

# get weighted-train unweighted/average-case metrics
r2_train = best_newprov_model_wt10_with_lags.score(newprov_pkg_wt10_with_lags.X_train, newprov_pkg_wt10_with_lags.y_train)
r2_test = best_newprov_model_wt10_with_lags.score(newprov_pkg_wt10_with_lags.X_test, newprov_pkg_wt10_with_lags.y_test)
mae = mean_absolute_error(newprov_pkg_wt10_with_lags.y_test, pred_test)
rmse = root_mean_squared_error(newprov_pkg_wt10_with_lags.y_test, pred_test)

# get weighted-train unweighted/average-case metrics in log space
log_pred_train = best_newprov_model_wt10_with_lags.regressor_.predict(newprov_pkg_wt10_with_lags.X_train)  # predictions in transformed target space
log_pred_test = best_newprov_model_wt10_with_lags.regressor_.predict(newprov_pkg_wt10_with_lags.X_test)  # predictions in transformed target space

r2_log_true_train = r2_score(np.log1p(newprov_pkg_wt10_with_lags.y_train), log_pred_train)
r2_log_true_test = r2_score(np.log1p(newprov_pkg_wt10_with_lags.y_test), log_pred_test)
mae_log_true = mean_absolute_error(np.log1p(newprov_pkg_wt10_with_lags.y_test), log_pred_test)
rmse_log_true = root_mean_squared_error(np.log1p(newprov_pkg_wt10_with_lags.y_test), log_pred_test)

# get weighted-train weighted/eval metrics

r2_train_w = r2_score(newprov_pkg_wt10_with_lags.y_train, pred_train, sample_weight=newprov_pkg_wt10_with_lags.sample_w_train)
r2_test_w = r2_score(newprov_pkg_wt10_with_lags.y_test, pred_test, sample_weight=newprov_pkg_wt10_with_lags.sample_w_test)
mae_w = mean_absolute_error(newprov_pkg_wt10_with_lags.y_test, pred_test, sample_weight=newprov_pkg_wt10_with_lags.sample_w_test)
rmse_w = root_mean_squared_error(newprov_pkg_wt10_with_lags.y_test, pred_test, sample_weight=newprov_pkg_wt10_with_lags.sample_w_test)

# Calculate tail evals
target_col = "avg_mdcr_stdzd_amt"
test_eval = newprov_pkg_wt10_with_lags.test_df.copy()

test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

non_tail_pred_to_y_ratio = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() / test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()
tail_pred_to_y_ratio = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() / test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()

non_tail_sse = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"sq_err"].sum()
tail_sse = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = ((test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"]) < test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col]).mean()
tail_under_prediction_rate = ((test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"]) < test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col]).mean()

non_tail_bias = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() - test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()
tail_bias = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() - test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()

newprov_weight10_lag_train_performance_metrics_test_eval = pd.DataFrame({
    "train_sample_weights":[10],
    "r2_train": r2_train,
    "r2_test": r2_test,
    "mae": mae,
    "rmse": rmse,
    "r2_train_log": r2_log_true_train,
    "r2_test_log": r2_log_true_test,
    "mae_log": mae_log_true,
    "rmse_log": rmse_log_true,
    "r2_train_w": r2_train_w,
    "r2_test_w": r2_test_w,
    "mae_w": mae_w,
    "rmse_w": rmse_w,
    "non_tail_pred_to_y_ratio": non_tail_pred_to_y_ratio,
    "tail_pred_to_y_ratio": tail_pred_to_y_ratio,
    "non_tail_sse": non_tail_sse,
    "tail_sse": tail_sse,
    "non_tail_share_of_total_sse": non_tail_share_of_total_sse,
    "tail_share_of_total_sse": tail_share_of_total_sse,
    "non_tail_under_prediction_rate": non_tail_under_prediction_rate,
    "tail_under_prediction_rate": tail_under_prediction_rate,
    "non_tail_bias": non_tail_bias,
    "tail_bias": tail_bias
})

newprov_weight10_lag_train_performance_metrics_test_eval